# Pairwise cosine similarity: precomputed norms and a coalescing layout

Retrieval scores a batch of queries against a set of documents with the cosine-similarity matrix

$$S_{ij} = \frac{a_i \cdot b_j}{\max\!\left(\lVert a_i \rVert_2 \, \lVert b_j \rVert_2,\ \varepsilon\right)}, \qquad \varepsilon = 10^{-8}$$

for rows $a_i$ of $A$ ($m \times d$) and $b_j$ of $B$ ($n \times d$). The clamp makes the similarity 0 whenever either vector is zero.

## Two decisions this problem is about
1. **Do not recompute norms.** A naive kernel where thread $(i, j)$ also computes $\lVert a_i \rVert$ and $\lVert b_j \rVert$ repeats each norm $n$ (or $m$) times. Compute all $m + n$ norms first with a small kernel (`row_norms`, one thread per row, blocks of 64), then run the main kernel.
2. **Pick the layout that coalesces.** In the main kernel, block `(16, 16)` with `j, i = cuda.grid(2)` (x → $j$). Threads of a warp then share $i$ and have consecutive $j$. Reading `B[j, k]` would make them touch addresses $d$ floats apart; reading from the transposed copy `Bt = B.T` (shape $(d, n)$, made C-contiguous on the host with `np.ascontiguousarray(B.T)`) as `Bt[k, j]` makes them read consecutive floats. `A[i, k]` is the same address for the whole half-warp: a broadcast.

Write `cosine_similarity_matrix(A, B)`.

## Inputs and output
- `A`: `float32` `(m, d)`, `B`: `float32` `(n, d)`, with $m, n, d \ge 1$. Rows may be all zero.
- Return `S`: `float32` `(m, n)`. Accumulate dot products and norms in `float32`. Compared with relative tolerance $10^{-5}$, absolute $10^{-6}$.

## Runtime
GPU in Colab; checked here on Numba's CUDA simulator.

**Runtime → Change runtime type → GPU** before running.

Write your solution in the next cell, then run the test cell.

In [ ]:
!pip -q install "numba-cuda[cu12]"

In [ ]:
import math
import numpy as np
from numba import cuda, float32

EPS = 1e-8

def cosine_similarity_matrix(A, B):
    """A: float32 (m, d), B: float32 (n, d). Returns float32 (m, n)."""
    m, n = A.shape[0], B.shape[0]
    d_A, d_B = cuda.to_device(np.ascontiguousarray(A)), cuda.to_device(np.ascontiguousarray(B))
    d_S = cuda.device_array((m, n), dtype=np.float32)
    grid = ((n + 15) // 16, (m + 15) // 16)
    naive_kernel[grid, (16, 16)](d_A, d_B, d_S)
    return d_S.copy_to_host()

@cuda.jit
def naive_kernel(A, B, S):
    j, i = cuda.grid(2)
    if i < S.shape[0] and j < S.shape[1]:
        dot = float32(0.0)
        for k in range(A.shape[1]):
            dot += A[i, k] * B[j, k]     # TODO: uncoalesced reads of B; use a transposed copy
        S[i, j] = dot                    # TODO: divide by the clamped norm product (precomputed)


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[1.0, 0.0], [1.0, 1.0]], "dtype": "float32"}, {"__nd__": [[0.0, 2.0], [3.0, 0.0], [-1.0, -1.0]], "dtype": "float32"}], "e": {"__nd__": [[0.0, 1.0, -0.7071067690849304], [0.7071067690849304, 0.7071068286895752, -1.0000001192092896]], "dtype": "float32"}}, {"a": [{"__nd__": [[1.0, 2.0, 3.0]], "dtype": "float32"}, {"__nd__": [[2.0, 4.0, 6.0], [-1.0, -2.0, -3.0]], "dtype": "float32"}], "e": {"__nd__": [[0.9999999403953552, -0.9999999403953552]], "dtype": "float32"}}, {"a": [{"__nd__": [[0.8422613143920898, -2.9761111736297607, -0.3050238788127899, 1.4498878717422485], [0.0, 0.0, 0.0, 0.0], [0.8106216192245483, 1.8988978862762451, 0.44696423411369324, 1.6291476488113403]], "dtype": "float32"}, {"__nd__": [[-0.1381521373987198, -0.4209144711494446, 0.4661547839641571, -1.43979012966156], [1.1097849607467651, 0.044554539024829865, -1.2931625843048096, 1.132064700126648], [0.6771097779273987, -0.3272091746330261, -0.6277669072151184, 0.37554043531417847], [-1.5776101350784302, -1.1557908058166504, 1.289809226989746, 0.36251455545425415], [1.2679108381271362, 0.2435932606458664, 1.7445708513259888, -0.9939135909080505]], "dtype": "float32"}], "e": {"__nd__": [[-0.20218093693256378, 0.40438112616539, 0.6337038278579712, 0.27591121196746826, -0.19913238286972046], [0.0, 0.0, 0.0, 0.0, 0.0], [-0.7246862649917603, 0.41225486993789673, 0.09245575219392776, -0.3647118806838989, 0.10220838338136673]], "dtype": "float32"}}, {"a": [{"__nd__": [[1.2836039066314697]], "dtype": "float32"}, {"__nd__": [[2.0], [-3.0], [0.0]], "dtype": "float32"}], "e": {"__nd__": [[1.0, -1.0, 0.0]], "dtype": "float32"}}, {"a": [{"__nd__": [[1.0434935092926025, 1.2708619832992554, 1.726274013519287, -0.23176805675029755, 0.16713619232177734, 2.1622564792633057, 1.4350650310516357, 1.1506468057632446], [-1.2681869268417358, 2.0862479209899902, 0.6557680368423462, 1.0790976285934448, -1.5902751684188843, 0.46994858980178833, -2.1974236965179443, 0.9308745861053467], [0.15693768858909607, 0.3761836886405945, -0.8557656407356262, 0.4484199583530426, -0.018605245277285576, -1.596941351890564, 0.40510764718055725, -0.10243937373161316], [-1.3762798309326172, 1.126343846321106, -1.2802605628967285, -0.666135847568512, -1.8222962617874146, -1.639007329940796, 0.5363144278526306, 2.5194907188415527], [-0.5171361565589905, -0.16490621864795685, -0.2074250876903534, 1.164212942123413, 2.55859637260437, 0.5033157467842102, -0.03721887618303299, 1.1593677997589111], [0.8534607291221619, -0.11203297227621078, 0.224992036819458, -0.7621589303016663, 0.23827531933784485, -1.3012847900390625, 1.6100558042526245, 1.6031911373138428], [-2.0664851665496826, -2.0461924076080322, -0.8315076231956482, -0.09256856888532639, 1.1616754531860352, -0.35562190413475037, 1.1127029657363892, -1.6296650171279907], [-0.2896048426628113, -0.17411744594573975, -1.116098165512085, -1.0049880743026733, 0.6478930115699768, 0.3095487058162689, 0.9758644104003906, -2.1449429988861084], [1.609957218170166, 1.9154813289642334, 2.23423433303833, -0.059682101011276245, -0.7041205167770386, 0.6082262396812439, 0.16037094593048096, 1.0027074813842773], [-2.4716062545776367, -0.3741470277309418, 0.2344651222229004, -1.0925546884536743, -1.113527536392212, 1.8850672245025635, 0.5628777742385864, -1.0534812211990356], [1.8000341653823853, -0.42317476868629456, -0.9744610786437988, -0.8823820948600769, -0.6733700633049011, 0.5566169619560242, -0.06864205747842789, 0.2400577962398529], [-0.5438532829284668, 0.6162131428718567, -0.8310179710388184, 0.3450532555580139, -1.1736043691635132, 1.0992215871810913, 0.7265241742134094, -0.26834067702293396], [0.19825811684131622, -0.6115925908088684, -0.43509605526924133, 0.23821021616458893, 1.0981332063674927, -1.7770975828170776, -0.35216888785362244, 0.77967369556427], [-1.2274513244628906, -1.3638908863067627, 1.0571256875991821, -0.47250983119010925, 1.9929910898208618, 0.9131319522857666, 0.13831208646297455, -0.08181025087833405], [-0.0978340208530426, -0.21839043498039246, -0.43605107069015503, -1.431913137435913, -0.5041676759719849, 0.3578622341156006, -1.0903072357177734, -1.1921929121017456], [-0.7704358696937561, 1.7511605024337769, 1.4633630514144897, 0.650047779083252, -0.5644097328186035, 1.0392900705337524, -1.85066556930542, 0.7539731860160828], [0.1477680206298828, 2.4066004753112793, 0.6785816550254822, -0.4481279253959656, 0.26228079199790955, -0.7607478499412537, 0.46180105209350586, -0.4559703767299652]], "dtype": "float32"}, {"__nd__": [[0.7897467017173767, 1.0237902402877808, -1.183301329612732, 1.5021508932113647, -2.855642080307007, -1.3456134796142578, 0.8249421715736389, 0.8292105793952942], [2.0242130756378174, -0.23116637766361237, -0.08522958308458328, 0.8714592456817627, -0.6112756729125977, 0.5674033164978027, -0.6180122494697571, -0.4419918358325958], [0.9102054834365845, 0.5998796224594116, -0.9202430248260498, 1.4433082342147827, -1.4923760890960693, 0.44125667214393616, 0.8041813969612122, -0.4751535952091217], [-1.2869292497634888, -1.0486310720443726, 0.2102082222700119, 1.8724521398544312, -0.248392716050148, -1.9640531539916992, -0.5641230344772339, -1.7967183589935303], [1.3628499507904053, 0.5773535370826721, 0.5141634941101074, -2.1968376636505127, -1.712467908859253, -1.3018878698349, -0.27891895174980164, -0.27238228917121887], [-0.6314191818237305, 0.9996672868728638, -1.1800777912139893, 1.16194748878479, 0.5886594653129578, -0.9965808391571045, -0.44969287514686584, 1.9805947542190552], [0.4018079340457916, -1.2991302013397217, -0.15018770098686218, -0.3949248492717743, 0.6992083191871643, -0.7273745536804199, 0.8397661447525024, -0.35221317410469055], [-0.6501776576042175, 1.0787196159362793, 0.3480468988418579, -1.8389720916748047, 0.15442213416099548, -0.02530323527753353, 0.33653998374938965, 1.2132234573364258], [-1.5593914985656738, 0.6420815587043762, -0.39768022298812866, 0.8476988077163696, -0.9481561183929443, 0.21079711616039276, 1.5868655443191528, -0.0285387821495533], [0.813677191734314, -2.686556339263916, -1.2052265405654907, -1.1089333295822144, -0.31252390146255493, -0.2675335109233856, 0.6985428333282471, -1.1908150911331177], [2.0048773288726807, 2.0998849868774414, -0.6865043044090271, 0.7829101085662842, 1.984002709388733, -0.32099199295043945, 0.2844715714454651, 1.1794955730438232], [1.2865817546844482, 0.3150086998939514, -0.049694377928972244, 0.6154710054397583, -0.041710589081048965, 1.2237966060638428, -0.1389826536178589, -0.3284696638584137], [-0.8078445196151733, 1.8869985342025757, -0.4094121754169464, -1.2696527242660522, 0.6203843951225281, -0.5334112644195557, 0.6768353581428528, -0.08900249004364014], [0.08590500801801682, 0.047818414866924286, 0.35229387879371643, -0.15088056027889252, -0.8848891854286194, 0.2839365303516388, 0.5161774158477783, -1.692436933517456], [-0.4186732769012451, 0.3084165155887604, 0.7970148921012878, 1.7789041996002197, 0.9842368960380554, 0.7689024209976196, 0.11146941035985947, 0.3898022472858429], [-0.04733194783329964, 0.6496368050575256, -2.0774993896484375, -0.12426423281431198, 1.7399688959121704, -0.5864866375923157, -1.4205400943756104, -0.5927595496177673], [-0.1287037581205368, 1.051672101020813, -2.0097477436065674, -0.26804912090301514, 1.2823063135147095, 0.4229530096054077, -0.8507725596427917, 1.3219441175460815], [0.1074245423078537, -0.0995112732052803, 0.07556893676519394, -0.5046716332435608, 0.9986974596977234, 1.1738783121109009, 0.2544439435005188, 0.7760627865791321], [0.0794021412730217, -0.2998627722263336, 0.16368982195854187, -0.3176538944244385, -0.029086876660585403, -0.10046158730983734, 1.5268174409866333, -1.6482986211776733], [-0.4050882160663605, 0.7718493938446045, -0.6950333118438721, -0.1827775090932846, 0.021916957572102547, -0.11637220531702042, 0.1479203999042511, -0.7919517755508423], [-0.4877476394176483, -1.0972317457199097, -0.42491185665130615, -0.8510266542434692, -1.6685231924057007, -0.1790560781955719, 0.022739358246326447, 0.16086529195308685], [0.03358246758580208, 0.7181849479675293, 0.27160149812698364, 0.10911103338003159, 1.0619086027145386, 0.08368479460477829, -0.8608959913253784, 0.7548807859420776], [0.07543372362852097, -1.648603916168213, -0.46088579297065735, -1.0952624082565308, 0.3315308392047882, -0.3132280111312866, -2.0113019943237305, -1.1861563920974731], [1.1668757200241089, -0.0897522047162056, 0.04427559673786163, -0.40413960814476013, -0.028168000280857086, 1.0433876514434814, 0.26150810718536377, -0.47401782870292664], [-0.04707527160644531, 1.427589774131775, 0.460602343082428, -0.24630655348300934, 1.4572159051895142, -1.215663194656372, -2.3944432735443115, -0.37042778730392456], [0.10706352442502975, 0.24426165223121643, -1.5404900312423706, -0.11114222556352615, -0.5184229016304016, 1.013488531112671, 0.940750241279602, 0.07224347442388535], [-0.9957920908927917, 0.38733941316604614, 0.020639922469854355, -0.06133495271205902, 0.1527269184589386, 1.0066159963607788, -0.9443917870521545, -1.371096134185791], [0.19547739624977112, 0.5126492977142334, 0.04366849362850189, -1.7727267742156982, -0.2713491916656494, -0.6488364338874817, -0.06997012346982956, 0.06714920699596405], [-1.1568384170532227, -0.6206830143928528, -0.9909635782241821, -0.378280907869339, 0.8836607933044434, -1.2058519124984741, 0.30323195457458496, 0.17236191034317017], [-1.0248218774795532, 0.7733777165412903, -2.5909156799316406, 1.7219552993774414, 1.172070026397705, -0.3696863651275635, -1.1352510452270508, -0.9115808606147766], [-1.268282175064087, 1.6796972751617432, -1.3136402368545532, 0.27638912200927734, -1.5920650959014893, -0.051151663064956665, 0.40958043932914734, -0.48032906651496887], [1.0840568542480469, -0.10346566885709763, 0.5975690484046936, -0.19388537108898163, -1.1039185523986816, -0.24731145799160004, 0.3799663186073303, 0.7841452956199646], [-0.5888591408729553, -1.0133253335952759, -0.32572630047798157, -0.7312119007110596, -0.20162485539913177, -0.5146240592002869, 0.23548775911331177, 0.5314506888389587]], "dtype": "float32"}], "e": {"__nd__": [[-0.09979099035263062, 0.12956620752811432, 0.10876694321632385, -0.7183569073677063, -0.02031438983976841, -0.18377666175365448, -0.28084567189216614, 0.3709232211112976, 0.08505242317914963, -0.40757960081100464, 0.3328559100627899, 0.4958706200122833, 0.09689183533191681, 0.006987809203565121, 0.3747659921646118, -0.5378751754760742, -0.06761326640844345, 0.6190105676651001, 0.015253366902470589, -0.29861053824424744, -0.34097784757614136, 0.2130342721939087, -0.6385824680328369, 0.5282634496688843, -0.2843996286392212, 0.16558007895946503, -0.14836324751377106, -0.01755526475608349, -0.6418986916542053, -0.573030412197113, -0.16540977358818054, 0.39646342396736145, -0.4111343324184418], [0.29687848687171936, 0.0020588792394846678, 0.130609393119812, 0.04324078932404518, -0.006223395001143217, 0.3866773843765259, -0.9361790418624878, 0.14017191529273987, 0.19256524741649628, -0.7662592530250549, -0.038706161081790924, 0.03843586891889572, 0.04824872687458992, -0.13620592653751373, 0.26282861828804016, -0.042750902473926544, 0.16276739537715912, -0.23475998640060425, -0.6365195512771606, 0.057298894971609116, -0.021783238276839256, 0.3861300051212311, -0.1899518370628357, -0.37394386529922485, 0.3715742826461792, -0.16888268291950226, 0.3422755300998688, -0.09055206924676895, -0.3685811161994934, 0.17328542470932007, 0.4714674651622772, 0.016597280278801918, -0.35483014583587646], [0.5820571184158325, -0.08331260830163956, 0.2818588316440582, 0.43539494276046753, 0.15191464126110077, 0.4955027997493744, 0.2729063630104065, -0.1519482135772705, 0.19918756186962128, 0.07036600261926651, 0.3315369188785553, -0.35252323746681213, 0.2849150002002716, -0.09841987490653992, -0.22835564613342285, 0.3706611692905426, 0.13181531429290771, -0.6117046475410461, 0.12580057978630066, 0.3914334774017334, -0.04435072839260101, -0.14278656244277954, -0.14773909747600555, -0.4732779562473297, 0.1500210464000702, 0.033701930195093155, -0.4383672773838043, 0.10219930112361908, 0.5104953050613403, 0.43915778398513794, 0.3389981985092163, 0.005903798621147871, 0.1078251525759697], [0.6094366908073425, -0.454118013381958, 0.07556651532649994, -0.1327759176492691, 0.2722150385379791, 0.6134645938873291, -0.24982304871082306, 0.5377382636070251, 0.4389142096042633, -0.23157277703285217, -0.00010600388486636803, -0.5600029230117798, 0.3975861072540283, -0.37773773074150085, -0.3235161006450653, -0.0675356462597847, 0.30314338207244873, -0.21691060066223145, -0.3663486838340759, 0.11430694907903671, 0.4460899233818054, -0.03924116492271423, -0.3625330924987793, -0.5823987722396851, -0.11249370872974396, 0.23528514802455902, -0.43211811780929565, 0.3715428113937378, 0.34993794560432434, 0.0009788438910618424, 0.5567423701286316, 0.29230165481567383, 0.4712575674057007], [-0.4401983916759491, -0.22350898385047913, -0.3367207646369934, -0.06111351400613785, -0.8084522485733032, 0.5222354531288147, 0.09087803214788437, -0.034291841089725494, -0.07637815177440643, -0.3013073801994324, 0.48876142501831055, 0.023529261350631714, -0.01148425042629242, -0.6890811920166016, 0.7184314131736755, 0.36665189266204834, 0.5222097635269165, 0.5934792757034302, -0.345131516456604, -0.2089189887046814, -0.6498246788978577, 0.6576083898544312, -0.15853941440582275, -0.2192307561635971, 0.2027219980955124, -0.09959244728088379, -0.040341924875974655, -0.5131797790527344, 0.3225615918636322, 0.3909810185432434, -0.3894011080265045, -0.49387678503990173, -0.09379365295171738], [0.24481406807899475, -0.21189644932746887, -0.1268690824508667, -0.34196844696998596, 0.3290077745914459, 0.23387427628040314, 0.46879345178604126, 0.451110303401947, -0.016775056719779968, 0.10346642881631851, 0.3552965819835663, -0.31708207726478577, 0.2294968217611313, -0.37520289421081543, -0.22081977128982544, -0.2729697823524475, 0.003280917415395379, 0.16553588211536407, 0.04834073781967163, -0.33032286167144775, 0.05942125990986824, -0.021012071520090103, -0.40773123502731323, -0.07717692852020264, -0.24578554928302765, -0.0015744572738185525, -0.9106122255325317, 0.39245837926864624, 0.2576260268688202, -0.4837997257709503, -0.2524569034576416, 0.5746129155158997, 0.41508108377456665], [-0.39556676149368286, -0.48497116565704346, -0.2600169777870178, 0.5067216157913208, -0.4241800606250763, -0.21645034849643707, 0.6089350581169128, -0.2429971843957901, 0.2737736403942108, 0.5439826846122742, -0.4824877083301544, -0.46285873651504517, 0.007344069425016642, 0.21237684786319733, -0.025568081066012383, 0.17190021276474, -0.16589854657649994, -0.03934076055884361, 0.539902925491333, 0.2589810788631439, 0.18125519156455994, -0.41463443636894226, 0.3263976573944092, -0.24075950682163239, -0.23841695487499237, 0.06533325463533401, 0.2660188376903534, -0.21082326769828796, 0.6936119794845581, 0.28803256154060364, -0.029646843671798706, -0.6284261345863342, 0.47855883836746216], [-0.32352203130722046, -0.16632072627544403, 0.02303255721926689, 0.09028847515583038, -0.00611727312207222, -0.49494820833206177, 0.4332037568092346, -0.0959617868065834, 0.12747453153133392, 0.5463888645172119, -0.1805640459060669, -0.012388479895889759, 0.35154709219932556, 0.5700322985649109, -0.352000892162323, 0.33736875653266907, -0.03762677684426308, 0.0005997606785967946, 0.7700191140174866, 0.6831231117248535, 0.030603190883994102, -0.4521026015281677, 0.2837284207344055, 0.32470250129699707, -0.14798620343208313, 0.399158775806427, 0.4159470796585083, 0.1728711873292923, 0.3100288212299347, 0.2506310045719147, 0.1871248334646225, -0.5194704532623291, 0.05180077627301216], [0.1793508529663086, 0.3110616207122803, 0.1462383270263672, -0.4929344654083252, 0.3712562024593353, -0.08135305345058441, -0.463186651468277, 0.3252668082714081, -0.12325553596019745, -0.5807428956031799, 0.37741222977638245, 0.41920849680900574, 0.07569989562034607, 0.028086960315704346, 0.20327769219875336, -0.49958086013793945, -0.18969061970710754, 0.1528998464345932, -0.18119105696678162, -0.311294287443161, -0.3092237114906311, 0.3082906901836395, -0.523000955581665, 0.331061989068985, 0.09490863233804703, -0.20348778367042542, -0.22707141935825348, 0.1876610815525055, -0.7706920504570007, -0.5850765705108643, -0.10035185515880585, 0.6330233812332153, -0.536748468875885], [-0.26587507128715515, -0.436309278011322, -0.08131316304206848, -0.0199241004884243, -0.11514459550380707, -0.45591336488723755, -0.19432690739631653, 0.20715771615505219, 0.5007838010787964, 0.10742918401956558, -0.7421186566352844, -0.18940162658691406, 0.13755588233470917, 0.4785773456096649, -0.09328991919755936, -0.30798253417015076, -0.24105395376682281, 0.11361716687679291, 0.32189568877220154, 0.2758789658546448, 0.4515838623046875, -0.41936564445495605, 0.05838490277528763, 0.03892338648438454, -0.3863655924797058, 0.29389721155166626, 0.6099713444709778, 0.03274483606219292, 0.0005245071370154619, -0.13445596396923065, 0.3842632472515106, -0.30339759588241577, 0.22924242913722992], [0.21374043822288513, 0.6116562485694885, 0.31394749879837036, -0.5566255450248718, 0.474639892578125, -0.2481657862663269, 0.15588819980621338, -0.03054835833609104, -0.42733773589134216, 0.5159057378768921, 0.15278396010398865, 0.49105969071388245, -0.23231276869773865, 0.047480713576078415, -0.5744189023971558, 0.02918691188097, 0.1678076982498169, 0.1698610782623291, -0.026029961183667183, -0.14603571593761444, 0.32552579045295715, -0.24199402332305908, 0.22286026179790497, 0.6994588375091553, -0.2983132004737854, 0.48510172963142395, -0.3178844749927521, 0.30864477157592773, -0.31670501828193665, -0.24361523985862732, -0.14289255440235138, 0.5057820081710815, 0.07333735376596451], [0.4480021297931671, 0.0258327666670084, 0.6943615674972534, -0.15835171937942505, -0.14943648874759674, -0.04531143233180046, -0.45743831992149353, -0.034959763288497925, 0.754764199256897, -0.07540790736675262, -0.20852771401405334, 0.26942047476768494, 0.12226268649101257, 0.4128204584121704, 0.006863324902951717, -0.20710282027721405, 0.04240093007683754, -0.06643935292959213, 0.20165349543094635, 0.4574506878852844, 0.2809889614582062, -0.46275267004966736, -0.43290188908576965, 0.17093877494335175, -0.5727316737174988, 0.7990012168884277, 0.2808506190776825, -0.21001268923282623, -0.2532821297645569, 0.16978852450847626, 0.7871062755584717, -0.031000973656773567, -0.16642992198467255], [0.0014821637887507677, -0.17074911296367645, -0.3856430947780609, 0.30957889556884766, -0.06133511662483215, 0.5613442659378052, 0.49456679821014404, -0.05414417386054993, -0.3614172041416168, 0.11988577246665955, 0.3405710756778717, -0.4721550941467285, -0.018283866345882416, -0.6504177451133728, -0.0373317152261734, 0.44499579071998596, 0.2874586880207062, -0.12456847727298737, -0.29451990127563477, -0.2191556841135025, -0.15373443067073822, 0.3305206894874573, 0.22202153503894806, -0.5310596227645874, 0.38533610105514526, -0.41200393438339233, -0.5272669196128845, 0.045306235551834106, 0.6636150479316711, 0.26885807514190674, -0.38787901401519775, -0.07080560177564621, 0.38227641582489014], [-0.8883863687515259, -0.4451361894607544, -0.7148825526237488, 0.011150709353387356, -0.5124638676643372, -0.24507971107959747, 0.35869595408439636, 0.09952735900878906, -0.15196837484836578, 0.11313925683498383, -0.23527851700782776, -0.2186979502439499, -0.06848055124282837, -0.16231749951839447, 0.36887338757514954, -0.01810630038380623, -0.05855803191661835, 0.5959993600845337, 0.11705578863620758, -0.27461880445480347, -0.2062007635831833, 0.22750113904476166, 0.24618473649024963, -0.01905893161892891, 0.019549937918782234, -0.30414873361587524, 0.28364086151123047, -0.19611887633800507, 0.29596590995788574, -0.12001519650220871, -0.5749983787536621, -0.49415749311447144, 0.24296514689922333], [-0.3052307963371277, 0.061817534267902374, -0.20528045296669006, -0.027937626466155052, 0.46176087856292725, -0.5373096466064453, -0.051489055156707764, 0.06875022500753403, -0.3521469533443451, 0.408548504114151, -0.4832068681716919, -0.011697216890752316, 0.08597686141729355, 0.43666619062423706, -0.6734315752983093, 0.28382551670074463, -0.014149348251521587, -0.13953550159931183, 0.13625171780586243, 0.3640376031398773, 0.42978435754776, -0.22450974583625793, 0.7570468187332153, 0.2882496416568756, 0.2187560647726059, 0.05635451525449753, 0.5936573147773743, 0.5007978677749634, -0.03632989525794983, 0.02212257869541645, 0.12129945307970047, -0.21652397513389587, 0.1320444792509079], [-0.018615111708641052, 0.026252789422869682, -0.06517848372459412, -0.12110315263271332, -0.06676728278398514, 0.21045462787151337, -0.8989753723144531, 0.21337789297103882, 0.009945310652256012, -0.834293782711029, 0.04135806858539581, 0.18041783571243286, 0.030567221343517303, -0.14426670968532562, 0.4286254346370697, -0.1228376105427742, 0.10575567930936813, 0.0476439893245697, -0.6011968851089478, -0.100359246134758, -0.24289724230766296, 0.5630029439926147, -0.18901069462299347, -0.1709471344947815, 0.43727561831474304, -0.3161748945713043, 0.40992873907089233, -0.10074500739574432, -0.5240384340286255, -0.01216187234967947, 0.18422645330429077, -0.0021417082753032446, -0.4911414086818695], [0.12326864153146744, -0.20214968919754028, 0.022445574402809143, -0.13956493139266968, 0.362837553024292, 0.09313556551933289, -0.3157113790512085, 0.45327574014663696, 0.13610044121742249, -0.5816468000411987, 0.4594409763813019, -0.040857259184122086, 0.7683936357498169, 0.17806659638881683, -0.004389998968690634, 0.0808192640542984, 0.03387464955449104, -0.16247479617595673, 0.16933231055736542, 0.5050129294395447, -0.4769272804260254, 0.27982744574546814, -0.4468260407447815, -0.06323645263910294, 0.4477582275867462, -0.15163303911685944, 0.047413334250450134, 0.44815361499786377, -0.15273161232471466, -0.03200788050889969, 0.3450165092945099, 0.022983910515904427, -0.49446287751197815]], "dtype": "float32"}}, {"a": [{"__nd__": [[-55.27865219116211, 6.361706733703613, -32.22842025756836, 42.739112854003906, -43.3539924621582, -50.82692337036133, -70.15026092529297, -21.50908851623535, -5.973923206329346, -49.03822708129883, -5.709827899932861, -131.32904052734375, 0.19760139286518097, -7.836935043334961, -112.11797332763672, 62.90492248535156, 17.218952178955078, 23.92289924621582, 49.56658935546875, -35.5444221496582, -67.5605239868164, 89.29725646972656, -36.99771499633789, -100.42154693603516, -103.85594177246094, 102.05131530761719, 1.4829068183898926, 13.173851013183594, 6.769881248474121, 20.029499053955078, 96.13240051269531, -26.326597213745117, 49.36547088623047, 5.954183101654053, -27.84431266784668, -44.06008529663086, -7.3176703453063965, -33.190364837646484, 47.84423828125, 87.03181457519531, 24.649438858032227, -28.272327423095703, -68.56201934814453, -51.417755126953125, 44.52470779418945, 35.63017654418945, 65.06329345703125, -49.674461364746094, -3.353685140609741, 61.937034606933594, 15.428159713745117, -54.34284210205078, -45.154850006103516, -79.02114868164062, -67.83773040771484, 69.0528793334961, -58.63425064086914, 70.58623504638672, 53.7335319519043, 19.677555084228516, -40.75007247924805, 107.70553588867188, 43.95795822143555, 55.218631744384766, 1.704328179359436, -75.51873779296875, 27.2042236328125, 35.655853271484375, -70.58833312988281, -1.545344352722168, 10.412327766418457, -65.76119995117188, 6.296148300170898, -0.9898605942726135, 20.031299591064453, -4.9716796875, -20.507753372192383, -12.263326644897461, -27.148441314697266, 72.42545318603516, 18.651412963867188, -37.3147087097168, -9.149807929992676, 49.18865966796875, 62.87387466430664, -40.2899284362793, 39.36378479003906, 32.38587951660156, 29.039169311523438, -42.79674530029297, 64.22782897949219, -21.05970001220703, -54.85553741455078, 116.49735260009766, 23.483184814453125, 27.620891571044922, 119.09561157226562, -86.44345092773438, -36.55126190185547, -99.35569763183594, -21.211929321289062, 86.88758087158203, 25.78187370300293, 39.01523208618164, -33.958770751953125, -40.72062301635742, -43.13367462158203, 104.48358917236328, 5.896756649017334, 37.4744758605957, -93.91505432128906, 28.501972198486328, -76.67115020751953, -9.950990676879883, 32.583316802978516, 22.79256820678711, 36.445865631103516, -38.263214111328125, -43.334415435791016, 23.2474308013916, 64.61618041992188, -2.7145438194274902, 3.8012397289276123, 6.3589653968811035, -16.007322311401367, 9.45603084564209, -29.484498977661133, -11.460506439208984, 32.654335021972656, -60.15925598144531, -8.588711738586426, -4.748947620391846, 0.2642205059528351, -32.25616455078125, 4.270393371582031, 28.921106338500977, 49.34451675415039, -97.15884399414062, 16.01983070373535, -22.80033302307129, 68.04683685302734, -54.90454864501953, -22.56557846069336, 11.266613006591797, -104.19004821777344, 89.32238006591797, 42.49005889892578, -21.113292694091797, -73.22917938232422, 9.241631507873535, 28.481966018676758, 29.18878173828125, 29.353639602661133, 57.29166793823242, 45.77540588378906, -152.0366668701172, 86.37559509277344, 55.87398910522461, -29.760234832763672, 1.977268934249878, 44.337158203125, -66.44026947021484, 94.67729187011719, -19.899234771728516, 1.1986650228500366, -48.8481330871582, 11.133617401123047, 83.64954376220703, 65.359619140625, -9.193828582763672, -106.68592834472656, -17.084163665771484, -40.456092834472656, -3.639740228652954, -62.789833068847656, 72.68209838867188, 24.525447845458984, -15.107193946838379, -15.001045227050781, 5.757923603057861, 41.9190673828125, -58.0768928527832, -27.814537048339844, -1.8594251871109009, -14.898863792419434, 38.697086334228516, -4.320512771606445, 54.6874885559082, -34.07644271850586, -59.90391540527344, -14.041413307189941, -120.26910400390625, -32.99709701538086, 37.53437042236328, 71.95552825927734, -65.2986068725586, -8.252320289611816, 21.586679458618164, -91.98812866210938, 42.85247802734375, 32.4571533203125, 53.08489227294922, 48.464256286621094, -35.6483268737793, -65.4063491821289, 11.910202026367188, 18.479084014892578, 170.7456512451172, -65.79147338867188, -57.3740348815918, 1.1048651933670044, -28.268518447875977, -73.20050811767578, -43.9702262878418, 51.590232849121094, -4.059542179107666, 35.79551696777344, 2.8061938285827637, 120.41136932373047, -25.767908096313477, 31.732097625732422, -27.491374969482422, -44.2857551574707, -53.19888687133789, -62.123382568359375, -30.81465721130371, -32.89834976196289, -59.30693054199219, -43.73026657104492, 19.921701431274414, -59.281307220458984, 55.17823791503906, 39.72430419921875, 3.063765048980713, -13.006542205810547, 66.63011932373047, 67.60418701171875, 100.96257781982422, 77.25322723388672, 9.565323829650879, -55.75817108154297, -24.468555450439453, -94.0627212524414, -17.75690460205078, -15.446050643920898, 29.710834503173828, -32.59418869018555, 48.282806396484375, 33.59085464477539, -18.720481872558594, -0.5307791829109192, -17.859952926635742, -8.813726425170898, -27.201858520507812, -50.66640853881836, -32.77437973022461, 23.606319427490234, -51.27107620239258, -48.77341079711914, -10.30157470703125, 96.17472076416016, 27.410978317260742, -4.39580774307251, 42.18465805053711, 18.712812423706055, -133.0824432373047, 4.836338043212891, 33.62348937988281, 51.459693908691406, 38.639251708984375, 39.7596435546875, -84.92598724365234, -25.517070770263672, 4.185247421264648, 87.861083984375, -36.185665130615234, 44.35755157470703, 54.154048919677734, 85.00969696044922, 39.938140869140625, 75.0262451171875, 18.210859298706055, -48.745758056640625, -67.30516052246094, 33.25941467285156, -60.17802429199219, 90.13179779052734, 51.18391418457031, 21.564472198486328, -13.936720848083496, 1.3645092248916626, 48.730892181396484, 20.31556510925293, -51.73009490966797, 80.60092163085938, -3.817624568939209, -23.172170639038086, 6.118557929992676, -1.8780142068862915, 76.98881530761719], [11.184853553771973, -16.162418365478516, 3.1688623428344727, 14.594483375549316, -51.247318267822266, -81.03338623046875, -29.678836822509766, 1.7869114875793457, -84.31731414794922, -78.37950897216797, -25.449647903442383, 5.656518459320068, 23.00185203552246, -51.49540710449219, 38.1884765625, -86.02074432373047, -19.73980140686035, -10.604093551635742, 99.02645874023438, 41.968692779541016, -63.61588668823242, 32.23701477050781, -16.07944107055664, -0.12046506255865097, 52.97812271118164, -14.45285701751709, -54.437644958496094, -8.816291809082031, 13.20020866394043, -3.7718005180358887, -44.395172119140625, -21.210693359375, -14.859994888305664, -7.380430221557617, 34.95997619628906, 54.297019958496094, -17.80867576599121, -62.171138763427734, 19.36109733581543, -39.958648681640625, 65.0022964477539, 14.449557304382324, 28.953950881958008, -117.14689636230469, -13.372641563415527, 27.087942123413086, -76.47280883789062, -12.334075927734375, 53.35215759277344, 25.63677406311035, -58.945560455322266, 37.13071823120117, 29.581636428833008, 29.185527801513672, 9.528639793395996, 7.461284637451172, -20.389833450317383, -38.49577331542969, -92.98082733154297, 16.53548240661621, 8.515625, -10.445700645446777, 15.409845352172852, 22.227487564086914, -47.05234909057617, -8.518012046813965, 19.829679489135742, 71.79771423339844, -20.22273063659668, 5.828644752502441, -34.393531799316406, -34.691158294677734, 61.20245361328125, -122.13164520263672, 46.628604888916016, -38.24774932861328, -25.993135452270508, -1.7350150346755981, 3.571129322052002, 31.654115676879883, 46.94011306762695, 51.28491973876953, -81.0165023803711, -2.806635856628418, 14.435677528381348, 17.548465728759766, -75.10397338867188, 71.49392700195312, 44.119056701660156, -6.484922885894775, 144.47943115234375, -55.90411376953125, -16.688535690307617, -12.201188087463379, 26.6727352142334, -35.96209716796875, -34.69303512573242, 54.959041595458984, -12.327142715454102, 46.38054275512695, 68.47623443603516, -14.362496376037598, -46.96696090698242, -24.266164779663086, 25.060199737548828, 56.21352767944336, 15.7016019821167, -60.711830139160156, 13.512405395507812, 87.04256439208984, 79.6865463256836, -32.951698303222656, 13.865010261535645, 20.56806182861328, -77.43975830078125, 21.047800064086914, -86.4815444946289, -80.05018615722656, -44.29632568359375, -75.85702514648438, 3.486407995223999, 15.844282150268555, -33.5607795715332, -18.81279182434082, 34.54185485839844, 12.214518547058105, 101.02717590332031, 65.60417938232422, -91.75505065917969, -16.813215255737305, -44.46579360961914, -69.948486328125, 51.37565612792969, -1.6356709003448486, -61.031402587890625, 75.68409729003906, 74.26519775390625, -13.327620506286621, 29.15620231628418, -27.847694396972656, -1.666489839553833, -78.9195556640625, -19.366724014282227, 9.984063148498535, -40.884178161621094, 71.9535140991211, -52.44977569580078, 13.214900970458984, 10.789886474609375, -81.15355682373047, 58.33550262451172, 8.371376037597656, 51.527099609375, 8.60017204284668, 16.889554977416992, 88.1312255859375, 0.07131306827068329, 16.505237579345703, -28.35073471069336, 12.992341995239258, -8.142821311950684, -1.7090052366256714, 63.6921272277832, -26.334774017333984, 1.3146936893463135, -20.089021682739258, 121.33848571777344, -2.597083806991577, -18.944812774658203, 5.830914497375488, -77.13652038574219, 75.68252563476562, -49.323387145996094, -45.334228515625, 29.596508026123047, -60.36146926879883, 69.73206329345703, 10.967740058898926, 69.71229553222656, -91.51304626464844, -11.402556419372559, 24.404417037963867, -38.344635009765625, 84.48444366455078, 10.390722274780273, -12.661130905151367, 12.257230758666992, 45.574462890625, 0.314860999584198, 2.551753520965576, -14.115086555480957, 27.260879516601562, 3.26153826713562, -2.465134382247925, 60.67422103881836, -13.5361967086792, -51.948486328125, -11.172677040100098, -27.951881408691406, -42.54978561401367, 33.40712356567383, 0.3857375681400299, 5.270858287811279, 71.0410385131836, 33.250911712646484, 12.745031356811523, -19.182985305786133, 50.06877136230469, -19.64586639404297, 70.58060455322266, 55.70282745361328, -11.562765121459961, 32.41018295288086, 37.96424865722656, 1.792279601097107, 22.743371963500977, 67.46237182617188, -83.94544219970703, 14.966638565063477, 24.807491302490234, 21.832260131835938, -34.07063293457031, -14.726397514343262, -7.321382999420166, 93.59178924560547, -34.93472671508789, 41.77513885498047, 57.2256965637207, 0.31562963128089905, 100.9643783569336, -8.135879516601562, -38.27091598510742, -20.507143020629883, 13.855321884155273, 48.53917694091797, -53.120582580566406, -47.197959899902344, 57.05546188354492, 32.554317474365234, -30.730571746826172, -61.194313049316406, -87.48480987548828, -48.69108963012695, 40.62007141113281, -29.471635818481445, 62.86056137084961, 7.2356953620910645, -62.66317367553711, 51.93121337890625, 7.828527450561523, 2.837024688720703, 41.75070571899414, -24.751691818237305, 159.3580322265625, -26.56375503540039, -2.2419273853302, 4.022619724273682, 99.28451538085938, 23.272912979125977, -43.70513153076172, 56.834476470947266, -5.77484130859375, -0.6094741821289062, -31.343427658081055, 69.06448364257812, 28.28468132019043, -24.18531036376953, -11.379057884216309, 53.12773132324219, -65.18038177490234, 55.95378112792969, -92.52450561523438, 27.065292358398438, -14.981630325317383, 27.00096893310547, -3.198897123336792, 104.62788391113281, -11.604433059692383, -26.551475524902344, 4.4304890632629395, -47.741729736328125, -80.53978729248047, -25.49073028564453, -9.278076171875, -57.623016357421875, 19.125871658325195, -25.303081512451172, 54.864418029785156, 27.791542053222656, -122.89797973632812, 2.9853475093841553, 3.1681690216064453, -79.36487579345703, 32.833839416503906, -75.38661193847656, -22.407865524291992, -23.529003143310547, 76.30657196044922, -51.4634895324707, 29.162675857543945], [-7.918344974517822, -6.565468788146973, -76.68508911132812, 153.48341369628906, -90.58560943603516, -27.712783813476562, 58.24332809448242, -17.20235824584961, 12.623177528381348, -38.860382080078125, 30.594196319580078, 26.863155364990234, 27.43419075012207, -31.94806671142578, -30.340164184570312, -25.979482650756836, -137.6523895263672, -27.61005973815918, 75.1756820678711, -66.31733703613281, 41.1214714050293, -6.815061092376709, -12.583440780639648, -80.40707397460938, 57.81174850463867, -43.37824249267578, 2.217071533203125, 28.591876983642578, 53.97908020019531, 11.314362525939941, 3.699211359024048, -12.898524284362793, 62.842559814453125, -72.26801300048828, -16.911483764648438, -20.846466064453125, 14.400598526000977, 7.96519136428833, 23.41002655029297, -41.040283203125, -28.823116302490234, 39.1546516418457, 92.38468170166016, -10.520343780517578, -35.00566101074219, 35.10504913330078, 0.2925974726676941, -84.9592056274414, 61.37753677368164, -10.011008262634277, 37.918582916259766, -38.13167953491211, -83.65579986572266, 4.95176887512207, 9.817424774169922, 26.072967529296875, 19.975940704345703, 90.24234771728516, 54.856021881103516, 25.23867416381836, -18.716747283935547, -58.77773666381836, 30.97014045715332, -1.0235775709152222, 11.357465744018555, -26.84577178955078, 9.328691482543945, 14.71230697631836, 6.713888645172119, -4.249846458435059, -26.895824432373047, 34.788909912109375, 28.82876968383789, -57.70780944824219, -37.624977111816406, -36.591468811035156, -30.92732048034668, -32.756160736083984, -51.428253173828125, -88.42118072509766, 60.09973907470703, -36.117515563964844, -10.672327041625977, 62.0979118347168, 27.581912994384766, 65.9083480834961, -37.75782775878906, 5.639074802398682, 54.19135284423828, -4.852917194366455, -35.27463150024414, 22.258647918701172, 66.23265075683594, -79.0605239868164, 46.61933898925781, 30.05681037902832, -53.503299713134766, -25.647125244140625, 23.4364070892334, -100.87205505371094, -9.954975128173828, 8.839949607849121, -19.247982025146484, -15.702383041381836, -111.31536865234375, -79.28897857666016, -21.749685287475586, 124.7955093383789, 2.1357855796813965, 20.596099853515625, 4.012814998626709, 34.51142883300781, -57.3333625793457, 114.04444885253906, -28.618070602416992, -78.67138671875, -18.18036651611328, 10.999723434448242, 10.092423439025879, -67.86708068847656, 20.88077163696289, -17.27910614013672, 83.70437622070312, -10.45960521697998, 84.32147216796875, -74.02694702148438, 11.623151779174805, 92.44171142578125, 60.33850860595703, 43.417091369628906, 11.661676406860352, 34.597015380859375, 54.11181640625, -67.21534729003906, 41.88051986694336, 42.826839447021484, -33.773555755615234, 40.239952087402344, -41.39849090576172, -24.405155181884766, 36.8199462890625, -124.09205627441406, -35.91049575805664, -68.07426452636719, -31.587617874145508, -63.64142990112305, -30.69534683227539, -2.0871329307556152, 23.57483673095703, 23.737380981445312, 67.30845642089844, 55.763023376464844, 52.446441650390625, -21.121593475341797, 1.3587212562561035, 35.898441314697266, -80.74627685546875, -27.208911895751953, -9.61207103729248, -20.453210830688477, 1.309462308883667, -1.3392987251281738, 83.65254211425781, -21.602142333984375, -83.26842498779297, -69.46003723144531, -55.50386428833008, 3.0561039447784424, 43.60578155517578, 21.913480758666992, -46.128089904785156, 25.71677589416504, 89.61419677734375, -18.04426383972168, -10.933016777038574, 40.8857421875, 24.541641235351562, -55.57086944580078, -136.75929260253906, -75.67927551269531, 62.297882080078125, -73.65657806396484, 1.6973117589950562, -6.85421085357666, -35.36116027832031, -68.45780944824219, -27.40703773498535, -32.74060821533203, 61.05397415161133, 4.852334499359131, 129.01763916015625, 58.574520111083984, 17.372865676879883, -79.58325958251953, -5.819077968597412, -35.572479248046875, 4.991952419281006, -36.11579895019531, -32.921573638916016, 90.41934967041016, -6.638133525848389, 7.215219497680664, -24.70146942138672, 7.768763542175293, 89.99288940429688, -60.44453430175781, 0.9778238534927368, -21.410076141357422, 35.28080368041992, -71.91703033447266, 73.14917755126953, 29.627790451049805, 13.494232177734375, -51.23946762084961, 5.688991069793701, 88.1536636352539, 34.87264633178711, -41.21731948852539, -47.54243087768555, -65.86711120605469, 62.90774917602539, -42.44377136230469, -29.807830810546875, 12.35894775390625, -95.3232192993164, -8.097229957580566, 46.38501739501953, 74.023193359375, 28.760826110839844, -35.86094665527344, 27.246015548706055, -3.7793681621551514, -0.09673464298248291, 34.639957427978516, -77.23564147949219, -64.566650390625, 97.97648620605469, -54.091896057128906, -0.18729737401008606, -85.22208404541016, 5.027402400970459, 20.550546646118164, -76.66365051269531, 67.22415161132812, -12.658451080322266, 103.27822875976562, -48.804195404052734, 19.06354522705078, 78.09700012207031, 64.93380737304688, 74.27391815185547, 47.66543960571289, 7.40065860748291, 44.37042236328125, 24.492534637451172, -26.92820167541504, 32.50722122192383, 55.70244216918945, 42.2623291015625, -133.46371459960938, 0.6793130040168762, 1.4163678884506226, 15.143330574035645, 39.26445007324219, 55.46044158935547, 55.40617752075195, 19.095300674438477, 68.68798065185547, -36.721466064453125, 20.267295837402344, 91.86479187011719, 28.486888885498047, -39.725807189941406, 8.183185577392578, -2.731827974319458, -51.3608283996582, 4.380756378173828, 32.6008186340332, -51.20697021484375, 32.96490478515625, -120.92081451416016, -32.99834060668945, 18.142438888549805, 19.887157440185547, 45.74092483520508, -11.923657417297363, -28.627941131591797, -109.18362426757812, -33.3468132019043, -51.1274528503418, 51.87106704711914, 35.115806579589844, 11.561372756958008, 56.03633499145508, 80.37593841552734, 44.75606155395508, -18.67052459716797, -35.80827713012695, 47.590301513671875, 115.61772918701172], [33.86617660522461, -25.194644927978516, 31.485076904296875, -38.54289245605469, -81.86615753173828, 18.37312889099121, -26.494115829467773, 74.72579956054688, -142.84996032714844, -25.66607666015625, 9.172144889831543, -17.849529266357422, -1.7084723711013794, 65.9188003540039, -69.8560562133789, -60.49390411376953, -61.23346710205078, 53.137691497802734, -21.719280242919922, 4.9829912185668945, -76.02935791015625, -59.92148971557617, -11.606492042541504, 148.1142578125, -35.49409866333008, -52.93425750732422, 42.68478775024414, 22.207557678222656, -133.58489990234375, 17.26795768737793, -39.07306671142578, 15.235194206237793, -5.626117706298828, 71.75812530517578, 3.034240484237671, -40.09709548950195, 28.678882598876953, -71.60358428955078, -88.18245697021484, 10.045268058776855, -63.334381103515625, 30.323537826538086, 14.898941040039062, 2.181868553161621, 52.18471145629883, 41.7098388671875, 22.663330078125, 32.46316146850586, 27.531936645507812, 18.372743606567383, -59.863548278808594, -141.33074951171875, -90.42887878417969, -46.78632354736328, 28.7437801361084, -84.52867889404297, -16.658411026000977, -9.93624210357666, 11.758048057556152, 33.416378021240234, -2.735478401184082, 4.372424125671387, -91.01570892333984, -24.36500358581543, -24.46902847290039, 85.57820892333984, 73.2369155883789, 66.7040023803711, 6.986096382141113, -8.088020324707031, -41.9818000793457, -17.689523696899414, -85.8399658203125, -36.536128997802734, -44.966217041015625, -12.973139762878418, 184.54248046875, 77.60330963134766, -48.78534698486328, 72.1403579711914, -36.893375396728516, -61.179115295410156, -0.5396177768707275, -31.15152931213379, -60.29668426513672, -27.35437774658203, -9.802906036376953, 105.36583709716797, -52.58292007446289, -23.835975646972656, 64.22464752197266, -38.69511032104492, -24.177703857421875, -54.98904037475586, -14.87299919128418, 8.22298812866211, -46.248573303222656, 70.43050384521484, -74.87813568115234, -27.546632766723633, -31.519493103027344, -16.285390853881836, -83.36248779296875, -51.2166748046875, -18.270280838012695, 35.47661209106445, 18.158597946166992, -13.103778839111328, -16.37611961364746, 79.68563079833984, -12.138652801513672, 3.947462797164917, 1.2359082698822021, 10.830580711364746, 16.599037170410156, -9.45092487335205, 49.161956787109375, 37.19829177856445, 11.603072166442871, 92.21885681152344, -60.65925979614258, 5.630683422088623, 36.1725959777832, -40.49398422241211, 24.969085693359375, -12.674064636230469, -58.19743728637695, 33.04280471801758, -49.30544662475586, -32.67409896850586, -14.5428466796875, -32.837158203125, -35.619224548339844, -19.216480255126953, -28.31487274169922, 52.99858474731445, -7.740768909454346, 71.24430847167969, -19.158893585205078, 50.91288757324219, -36.904232025146484, -78.10694885253906, -8.470683097839355, -98.34584045410156, 54.024410247802734, 24.266799926757812, 12.976835250854492, -21.066402435302734, -0.20373782515525818, -19.071117401123047, 81.85035705566406, 19.4863224029541, -30.937257766723633, -41.140892028808594, 43.34627151489258, -1.435236930847168, 52.00155258178711, 7.180403232574463, 41.87035369873047, -115.51282501220703, 22.539226531982422, -39.22572326660156, 41.950836181640625, 45.68885803222656, 26.82707977294922, 0.38096877932548523, 2.238568067550659, 5.559712886810303, 6.053606986999512, -67.36388397216797, 24.18170928955078, 7.054959297180176, -24.449880599975586, 61.00627136230469, -18.05777931213379, 29.78742790222168, 54.68458557128906, 101.7353515625, 2.1811676025390625, 35.72256088256836, -80.39017486572266, -3.548786163330078, -24.92559242248535, 15.236513137817383, -82.7436294555664, 36.24571228027344, 24.898630142211914, 59.81957244873047, 3.466956615447998, -2.256023645401001, 3.0565683841705322, 31.344825744628906, -71.67008209228516, 59.67241668701172, -17.797208786010742, -90.70591735839844, 18.6115779876709, 98.95890045166016, -30.475299835205078, -52.074790954589844, -133.93905639648438, -87.84738159179688, 16.466012954711914, -36.421783447265625, 26.06116485595703, -32.5228157043457, 31.02659797668457, -38.47911834716797, 64.2357406616211, -29.65755844116211, 109.40648651123047, -24.42833137512207, -22.917221069335938, 65.85035705566406, 86.02525329589844, -52.17151641845703, -82.516845703125, 63.06636428833008, -23.889060974121094, -61.48061752319336, 63.693016052246094, -77.55911254882812, 0.7942777872085571, 26.825946807861328, 44.18547439575195, 53.6914176940918, -46.42892074584961, 22.40479850769043, 21.400665283203125, 79.05591583251953, 28.42903709411621, -55.022705078125, -64.16168975830078, -25.44034194946289, 118.03047943115234, -6.005625247955322, -50.43685531616211, 9.890748977661133, 51.63117218017578, 19.66533660888672, -11.916607856750488, 87.17613983154297, 56.30057907104492, 61.182594299316406, -49.182010650634766, -36.707183837890625, -33.59662628173828, -3.8360085487365723, 6.0288190841674805, -73.57868194580078, -54.09274673461914, 8.600150108337402, 19.621967315673828, 91.09443664550781, 3.3284764289855957, 51.237220764160156, -97.70963287353516, 21.07538604736328, -76.7266616821289, -44.3838996887207, 6.360319137573242, 6.049960613250732, -33.338321685791016, 35.192283630371094, -87.79671478271484, -26.190793991088867, 9.657981872558594, -25.338537216186523, 10.511719703674316, 99.7510757446289, -27.425518035888672, -6.210452079772949, -47.36259460449219, -63.97445297241211, -10.583308219909668, 47.65489959716797, -41.772945404052734, 54.76220703125, 14.481261253356934, -59.99318313598633, -71.4686279296875, -63.76990509033203, 16.871858596801758, -14.003056526184082, 58.25152587890625, -9.471938133239746, 12.314278602600098, 41.7921257019043, 99.20470428466797, -73.1039810180664, 55.676658630371094, 1.432621717453003, 41.15184783935547, -18.86670684814453, 68.27616119384766, -40.52333068847656, 22.231767654418945, 24.09744644165039, 52.73591232299805, -59.63533020019531], [-82.86051940917969, 89.42697143554688, 63.176544189453125, -12.148113250732422, 0.39686137437820435, -60.5611572265625, 87.2157211303711, -75.2041015625, 25.40667724609375, -0.27512475848197937, 53.67778015136719, 59.4390869140625, -53.33240509033203, -46.815711975097656, 15.219209671020508, -79.84662628173828, 51.50532150268555, 29.908510208129883, -97.00540924072266, 89.45768737792969, -39.950321197509766, 46.125858306884766, -48.77235412597656, 75.60828399658203, -11.679411888122559, -39.7830696105957, 55.50126266479492, -18.756494522094727, -47.81606674194336, -44.86975860595703, 12.283860206604004, 20.426651000976562, -21.76052474975586, -24.341508865356445, -2.7576229572296143, -35.90294647216797, 61.08741760253906, 61.12565994262695, -44.4250602722168, -30.416500091552734, 130.2431182861328, 31.698148727416992, -45.97846603393555, 11.689614295959473, -26.32975196838379, -13.865937232971191, 57.883487701416016, -31.19778060913086, -28.99608612060547, -10.841180801391602, 19.25748062133789, -14.014296531677246, 98.5967788696289, -35.55165100097656, 51.00054168701172, -51.4200439453125, -65.40851593017578, -9.875405311584473, -12.135632514953613, -100.02672576904297, -57.139320373535156, 41.69796371459961, 68.21336364746094, 91.03262329101562, 70.39678955078125, -34.66947555541992, 13.765727043151855, 8.637162208557129, -25.86112403869629, 42.09701919555664, 0.015497580170631409, 54.822784423828125, 46.361812591552734, 56.040775299072266, 29.819717407226562, -33.85822296142578, -42.79806137084961, 16.2658748626709, 0.12788496911525726, 55.89426040649414, -64.26976013183594, 47.03274154663086, 8.185901641845703, -73.75248718261719, 44.97397994995117, 16.450910568237305, -23.39849853515625, -35.406768798828125, 54.98390197753906, -38.69758605957031, 82.25112915039062, -12.312812805175781, 23.082136154174805, 10.389424324035645, 6.994561672210693, 19.3795166015625, -49.48525619506836, 72.42193603515625, 39.84247970581055, 28.984073638916016, 3.713723659515381, 0.45837894082069397, 66.70289611816406, -3.184007406234741, -20.797691345214844, -44.355323791503906, -69.328125, 42.73977279663086, 17.59012222290039, -69.39462280273438, 2.0882537364959717, 11.34774112701416, 76.42594909667969, 18.64836883544922, 46.91804504394531, 138.5069122314453, -74.18800354003906, 87.83565521240234, 28.28727149963379, 14.535272598266602, -40.036888122558594, -11.84953498840332, 46.8163948059082, -50.715553283691406, -58.16870880126953, 61.22032928466797, -7.85890007019043, -12.764769554138184, -67.0615463256836, -8.028373718261719, -7.4831767082214355, -9.04395866394043, -11.375794410705566, 6.767040252685547, 39.4599723815918, -58.47080993652344, -1.3733774423599243, 10.605931282043457, -26.407474517822266, -18.369916915893555, -73.7528305053711, -79.0333480834961, -31.749652862548828, -4.585497856140137, 27.80232048034668, 90.14398956298828, 23.959884643554688, 58.182090759277344, -31.056699752807617, -11.875767707824707, -25.651371002197266, -43.92087173461914, 69.83104705810547, -28.576269149780273, -9.641934394836426, 56.51511001586914, -18.83013916015625, 18.20030975341797, 42.02022933959961, 58.93460464477539, 20.10731315612793, -40.175201416015625, -18.341354370117188, 4.824739456176758, -30.536035537719727, -52.5071907043457, 44.005332946777344, 87.4259262084961, 3.1175246238708496, -94.45909881591797, 122.13113403320312, -13.844339370727539, -0.4467300474643707, 16.327560424804688, -27.073875427246094, 61.682884216308594, 34.43107986450195, 46.99637985229492, 15.917818069458008, -7.9440131187438965, 52.1672248840332, -53.50619125366211, -47.52642059326172, 10.311229705810547, 3.339097023010254, -64.9861831665039, -54.437660217285156, -51.96548080444336, 53.726261138916016, -44.973697662353516, -66.58061981201172, -79.54331970214844, 1.077609658241272, -55.93413162231445, -21.94269371032715, 74.47045135498047, 7.012712001800537, -105.94122314453125, 29.285966873168945, -83.4375228881836, -37.681182861328125, 23.534067153930664, -55.25578689575195, -88.26097106933594, 18.464641571044922, -160.33041381835938, -58.05675506591797, -6.35513162612915, -20.13709831237793, -120.23762512207031, -18.35247039794922, 28.174358367919922, -122.09233093261719, -17.418813705444336, -8.943082809448242, 49.785831451416016, 28.12875747680664, -47.86272048950195, -51.99922180175781, 4.905314922332764, -107.28795623779297, 39.20268249511719, 49.452030181884766, 35.417640686035156, -25.82188606262207, -41.91411590576172, -66.19300079345703, -21.771215438842773, 37.688026428222656, -18.39954948425293, -40.758426666259766, 31.008594512939453, -18.078876495361328, 5.5755109786987305, -40.459739685058594, 42.45034408569336, 17.732616424560547, -63.7138671875, 49.97660827636719, -16.911266326904297, 28.128612518310547, 8.971428871154785, -99.46589660644531, -38.13474655151367, 105.78401184082031, -8.425004959106445, -28.428054809570312, -28.353538513183594, -42.99664306640625, -2.3780572414398193, 76.91425323486328, -32.74461364746094, -68.45448303222656, 2.1679747104644775, -67.39227294921875, -11.238138198852539, 89.8116683959961, 44.94282913208008, -34.23027801513672, 125.93445587158203, -14.298674583435059, 69.84385681152344, -10.879294395446777, -44.74342346191406, -27.933544158935547, -14.245100021362305, 43.20119857788086, 5.611575126647949, 47.348751068115234, -24.320100784301758, 103.56874084472656, 90.49139404296875, -11.740127563476562, -7.553053855895996, -14.131538391113281, -87.23758697509766, -138.66392517089844, -13.524018287658691, -29.3699893951416, 67.98143005371094, -4.057889938354492, 69.70030975341797, 28.72001838684082, -49.019813537597656, 102.50959777832031, 50.82252502441406, 11.435810089111328, 111.05825805664062, -53.24552536010742, 56.85070037841797, 77.98482513427734, 38.160789489746094, 73.86611938476562, -41.98923873901367, 6.330737590789795, 21.47994041442871, -6.197412490844727, 97.25273132324219, -41.54443359375, 57.96074676513672], [0.6381697654724121, 30.192487716674805, 8.967300415039062, -25.73352813720703, 56.61469268798828, -73.20384979248047, 2.4259016513824463, 48.40155029296875, -35.74394607543945, 19.119712829589844, -5.821680068969727, -87.98271942138672, 6.338393688201904, 44.918907165527344, -9.20964527130127, 70.44480895996094, -71.56028747558594, -1.53098726272583, 40.52579879760742, -15.702231407165527, 2.334486961364746, -30.52142333984375, -47.80256652832031, 33.48978042602539, 147.226318359375, -13.625261306762695, 14.269121170043945, 43.20695877075195, -22.684755325317383, 66.49295043945312, 3.321662425994873, 87.2808837890625, 10.591754913330078, -49.64906692504883, 43.07319259643555, -77.1215591430664, 54.433719635009766, -54.300235748291016, -52.08308792114258, -43.6844482421875, 50.87425231933594, -19.01210594177246, 74.520263671875, 41.59012222290039, 12.789681434631348, 23.011554718017578, -17.38204002380371, 0.017672667279839516, -44.9390869140625, 1.0194112062454224, -43.41297912597656, 34.06755447387695, 10.206402778625488, -0.32348042726516724, 47.388668060302734, -43.22182083129883, 48.62504577636719, 24.81563949584961, -107.63490295410156, -2.17651104927063, 15.90075969696045, -35.60133743286133, 48.69768524169922, -84.19331359863281, -37.886173248291016, -45.94441223144531, 75.62269592285156, 24.451826095581055, -67.00250244140625, 85.24874114990234, 96.43698120117188, 71.50450134277344, -81.65918731689453, -30.36818504333496, -1.9238224029541016, 13.876058578491211, 33.436767578125, 1.9186969995498657, 44.892906188964844, 6.32953405380249, -60.94503402709961, -10.197361946105957, -57.52476501464844, -27.49277687072754, 42.458213806152344, 62.70643997192383, 39.975406646728516, -66.16116333007812, -44.286006927490234, 29.878087997436523, 50.81952667236328, -83.37931060791016, -76.94686889648438, -15.518048286437988, 8.590239524841309, 18.3203067779541, -37.189537048339844, 70.50631713867188, -91.2989501953125, 5.673412799835205, 21.08342742919922, 36.65882873535156, 0.515123724937439, 52.825260162353516, 3.532618999481201, -83.8579330444336, -18.174955368041992, 7.0189208984375, -70.73963928222656, 94.5708236694336, -92.5152816772461, -0.42683902382850647, 53.48249816894531, -41.748992919921875, 57.20900344848633, -82.23583221435547, -90.45259094238281, -3.72166109085083, 18.821256637573242, 50.86753845214844, 46.2097282409668, 54.70280456542969, 61.63029098510742, 27.319744110107422, 21.126863479614258, -40.11198425292969, -84.42754364013672, 54.984413146972656, 87.05305480957031, -24.77621841430664, -14.500422477722168, 12.107738494873047, -66.6615219116211, -29.901487350463867, -19.275115966796875, -91.36647033691406, 49.11695861816406, -33.90810012817383, 0.439438134431839, -85.1422348022461, -111.67784118652344, -21.41642951965332, -17.942636489868164, -27.998470306396484, -17.959041595458984, -11.633905410766602, -47.949947357177734, -11.304608345031738, 45.03376007080078, 75.42920684814453, 82.1240463256836, -54.55436706542969, -4.834043025970459, -22.83991050720215, -24.18640899658203, 76.64458465576172, -22.381134033203125, -30.971515655517578, -49.10770797729492, -18.46344566345215, -39.50356674194336, 2.571363687515259, -17.227479934692383, 83.53294372558594, 33.008296966552734, -71.17131805419922, -9.6824951171875, 15.645644187927246, 37.75861358642578, -64.9394302368164, -65.03221130371094, -24.747514724731445, 38.53474426269531, 22.720457077026367, 19.941492080688477, -114.53543853759766, -145.9104766845703, -38.397701263427734, -11.041023254394531, -16.23139762878418, 2.903808832168579, -69.74279022216797, 103.52934265136719, -74.6148681640625, 35.85641098022461, 58.08466720581055, -81.08499908447266, -27.26425552368164, -2.520695209503174, -31.731281280517578, -77.88533782958984, -1.8055932521820068, -24.45227813720703, -38.65733337402344, 5.52701473236084, 11.94217586517334, -21.708589553833008, -36.08273696899414, -2.2469711303710938, -87.3388442993164, -59.040313720703125, 12.100017547607422, 5.832627296447754, 15.505049705505371, -61.46400451660156, -6.125676155090332, -63.090396881103516, -19.95060920715332, 4.99796199798584, -92.69232177734375, -23.007766723632812, -46.501914978027344, -10.745710372924805, 71.24441528320312, -29.957468032836914, 8.991125106811523, 41.983802795410156, -54.761417388916016, -30.44637680053711, 6.117568016052246, -16.74367332458496, 1.1809823513031006, 5.256597995758057, -27.536630630493164, -80.3016586303711, -84.96785736083984, -45.07159423828125, 13.847028732299805, 43.03468704223633, -27.309810638427734, -62.83820343017578, -65.9213638305664, -18.003490447998047, -55.60775375366211, -30.075069427490234, -11.530435562133789, 55.798954010009766, 43.82606887817383, -94.53594970703125, -14.192891120910645, -76.4582748413086, 36.33649444580078, 28.13383674621582, -92.82830810546875, 24.149438858032227, 2.959172010421753, 24.092479705810547, 89.71340942382812, 34.02059555053711, 15.461078643798828, -46.51145935058594, -25.399986267089844, 1.1158174276351929, 48.517791748046875, -52.367156982421875, -92.31768798828125, 59.34437942504883, 58.046714782714844, -54.2534065246582, 91.91726684570312, 15.11961555480957, 29.03425407409668, -50.39215087890625, 52.870792388916016, -34.460384368896484, -69.55646514892578, -19.807069778442383, 52.12081527709961, 49.90985870361328, 42.54610061645508, -29.748994827270508, 27.8909854888916, -30.029529571533203, 29.750019073486328, -29.481420516967773, -46.98793411254883, -30.24894905090332, -16.202207565307617, -37.616424560546875, -0.009406591765582561, -20.537883758544922, -51.4414176940918, 26.422420501708984, -16.635040283203125, -23.917190551757812, -51.827125549316406, 52.04903030395508, 4.9747138023376465, -66.94589233398438, 17.6984920501709, 33.31569290161133, -40.687835693359375, -22.226280212402344, -19.275291442871094, 49.975074768066406, 9.522490501403809, 52.14387893676758, 37.69940185546875, -17.205738067626953, -27.672269821166992], [6.635660648345947, -58.056976318359375, 20.839035034179688, 0.5667470097541809, 56.81581497192383, 9.417216300964355, -24.660003662109375, 84.9887466430664, 50.28690719604492, 51.511043548583984, 95.29267883300781, 30.405763626098633, -30.893516540527344, 82.75011444091797, -19.650150299072266, -12.441208839416504, -42.3515739440918, 14.20643138885498, 30.007081985473633, 59.46572494506836, -14.145307540893555, -6.415538311004639, -53.53510284423828, -53.416072845458984, 12.934967994689941, -58.68800354003906, 5.143474578857422, 22.55475425720215, -7.624443054199219, 60.375946044921875, 7.301297664642334, 3.9107885360717773, 1.8766303062438965, -53.180484771728516, 33.07322692871094, -7.436024188995361, -9.211694717407227, 0.2489478588104248, 8.549569129943848, 34.42137145996094, -56.95226287841797, 36.5654182434082, -63.30112075805664, 7.781644344329834, 12.856419563293457, 1.6095929145812988, 75.02584838867188, -58.929420471191406, 33.81837463378906, -49.733551025390625, -106.50990295410156, -96.27621459960938, -5.609247207641602, -24.59339714050293, 78.80572509765625, 53.629905700683594, -59.01939392089844, 22.11062240600586, -26.963483810424805, 16.178375244140625, 104.86055755615234, 7.974368095397949, -27.079448699951172, -11.948090553283691, 8.015273094177246, -21.89497947692871, 32.83866882324219, 16.000930786132812, 108.17353820800781, -76.48011779785156, -27.94698715209961, -18.24668312072754, 12.236940383911133, 1.8882046937942505, -4.725222110748291, 28.785541534423828, 75.92015075683594, 5.287177562713623, 10.949278831481934, -88.11851501464844, -78.24691772460938, -79.04731750488281, 3.3111772537231445, -79.33448028564453, 15.953119277954102, 10.26070499420166, 58.785491943359375, -62.38581085205078, 13.002043724060059, 23.345176696777344, -109.4610824584961, -21.71753692626953, 18.33123207092285, -42.51049041748047, 35.35346984863281, 7.6351494789123535, 129.2599639892578, -60.6711311340332, 9.521677017211914, 4.982258319854736, -73.30233764648438, -7.298597812652588, 75.7193603515625, -62.09884262084961, 75.36788940429688, -31.944927215576172, -57.991493225097656, -49.144630432128906, -40.643924713134766, -12.244621276855469, -33.18486404418945, 19.032291412353516, 6.906864166259766, 30.598011016845703, 21.60433006286621, 43.9433708190918, -9.078550338745117, -78.16827392578125, -21.62015151977539, 34.51691436767578, 100.4449234008789, -94.4270248413086, 29.299592971801758, 78.67466735839844, 47.16903305053711, -28.289308547973633, 16.042240142822266, -36.187339782714844, -21.03957176208496, 38.78703689575195, 32.30309295654297, 10.673569679260254, -13.974203109741211, -9.148843765258789, -39.978023529052734, -52.2990837097168, -21.42411231994629, 49.555870056152344, 69.27095794677734, 20.510486602783203, -6.465596675872803, 42.157867431640625, 12.50108528137207, 25.460233688354492, -58.22138214111328, 42.323036193847656, 62.99663162231445, -49.97882080078125, -38.57483673095703, 90.74304962158203, 58.515037536621094, 10.32618522644043, 42.02322006225586, -33.757225036621094, 22.981752395629883, -54.12671661376953, 89.47943878173828, -48.98931884765625, -39.72933578491211, -39.297340393066406, -61.90290451049805, 15.276052474975586, -57.098514556884766, 136.33926391601562, -18.42208480834961, -19.814706802368164, 73.72654724121094, 112.1973648071289, 28.410852432250977, -66.11743927001953, -12.805521011352539, 51.733577728271484, -17.066753387451172, -9.413196563720703, 45.77873992919922, 24.65298080444336, 102.63381958007812, 33.747005462646484, -15.834027290344238, 46.23912811279297, -20.034833908081055, -8.992663383483887, 43.814937591552734, 69.21443176269531, -6.676501274108887, 36.92193603515625, -20.807018280029297, 5.795179843902588, 28.793415069580078, -28.718557357788086, 58.486724853515625, -38.20146560668945, -65.31172943115234, 19.640823364257812, 21.54534339904785, 24.31739044189453, -57.294212341308594, 80.41519165039062, -9.955141067504883, 41.56621551513672, -78.01575469970703, 16.95575523376465, 2.7034151554107666, 42.46577835083008, 17.25806427001953, -50.21275329589844, -32.197471618652344, 22.203365325927734, 30.62737274169922, 138.81240844726562, -22.553726196289062, -14.910324096679688, -69.00721740722656, -58.99757766723633, -14.28079605102539, -34.3154411315918, 54.89385223388672, 31.003385543823242, -17.3941650390625, -60.66907501220703, 3.6113593578338623, -36.995487213134766, 36.64282989501953, 14.423908233642578, -61.82884979248047, 9.42116641998291, 89.10025024414062, -90.9253921508789, -10.5046968460083, 2.8081698417663574, 31.226774215698242, -20.540813446044922, -30.232927322387695, 24.79977798461914, 19.421667098999023, 50.081478118896484, -4.073041915893555, 73.46295928955078, -16.222946166992188, -12.63125991821289, -3.629425048828125, -12.329023361206055, -30.401580810546875, -68.45953369140625, 85.26239013671875, -38.15297317504883, 1.3165452480316162, 29.56037139892578, 65.0052490234375, 54.445518493652344, -5.295350551605225, -15.977285385131836, 18.3115234375, 74.48798370361328, -66.81692504882812, -24.86767578125, -22.98321533203125, -19.766916275024414, -85.28292846679688, -37.21064758300781, 47.889373779296875, 67.05486297607422, 120.13223266601562, 3.722322702407837, -83.76921081542969, 34.45044708251953, -10.528036117553711, 18.050697326660156, -88.29570770263672, 14.735677719116211, 7.579845428466797, -75.41254425048828, 0.9552078247070312, 82.00044250488281, -16.01198387145996, -4.445424556732178, -11.613781929016113, 4.540546894073486, 48.11321258544922, 20.730684280395508, 15.815123558044434, -76.99449157714844, 55.65761947631836, 29.099756240844727, -26.031457901000977, -25.568723678588867, 1.8107235431671143, 43.1041374206543, -76.83401489257812, 8.363709449768066, -2.4059293270111084, 77.73194122314453, 9.2160005569458, 11.113359451293945, -44.031982421875, -88.22419738769531, -5.23703670501709, 25.698101043701172, 83.96624755859375, 63.902198791503906], [36.74443435668945, -10.056034088134766, 9.859396934509277, 29.777578353881836, -1.4117457866668701, 54.30638885498047, 34.19684982299805, -39.73756408691406, -16.521623611450195, 84.92027282714844, -20.932193756103516, -27.82112693786621, -42.563838958740234, 2.2010855674743652, 36.9150390625, -43.886051177978516, 31.93814468383789, -76.49776458740234, -12.319368362426758, 9.896306991577148, -6.757208824157715, 89.99992370605469, -6.973716735839844, 26.401809692382812, -61.33770751953125, 54.80311584472656, 14.064657211303711, 9.13494873046875, -3.563509464263916, 29.388410568237305, -16.884979248046875, -8.725103378295898, 41.13990783691406, -32.892520904541016, -13.04802417755127, -25.80062484741211, 111.64031982421875, 38.28266906738281, 20.798011779785156, 94.95198059082031, -32.34272766113281, -25.29431915283203, 38.384220123291016, 48.80694580078125, 15.615913391113281, 23.896615982055664, 63.81709671020508, 32.06541442871094, 6.04464054107666, -12.177605628967285, 10.078765869140625, 40.147823333740234, -47.606529235839844, -92.2437515258789, -17.489831924438477, -3.378499984741211, -26.960674285888672, 39.63296127319336, -5.335571765899658, 13.638420104980469, -26.763832092285156, 12.645360946655273, -11.336688041687012, 30.236831665039062, -26.991430282592773, 43.251617431640625, 35.006187438964844, -18.16935920715332, 71.31697082519531, 11.976560592651367, -15.001508712768555, 25.68817138671875, -3.7968013286590576, 39.32712173461914, -42.86031723022461, 52.841800689697266, 31.395238876342773, -3.1396992206573486, 37.10500717163086, 49.99581527709961, 62.590675354003906, -21.177791595458984, -85.83070373535156, -23.27000617980957, -16.858108520507812, -51.512664794921875, -30.10643768310547, -11.658584594726562, -81.12750244140625, 81.73866271972656, 116.86868286132812, -17.76251792907715, 50.39701461791992, 9.130521774291992, -43.344276428222656, -0.3657244145870209, -60.05218505859375, 132.95240783691406, -39.33915328979492, -50.39934539794922, -10.08376693725586, -9.20505142211914, 25.281051635742188, 55.8062858581543, 9.576775550842285, -8.069029808044434, -55.89482879638672, 40.917938232421875, -76.60128784179688, -62.31242370605469, -27.166397094726562, 66.98889923095703, -42.400997161865234, 28.049671173095703, -7.15033483505249, 16.91577911376953, 114.8855209350586, -63.732933044433594, -58.664817810058594, 41.101951599121094, -5.513000011444092, 35.744781494140625, 69.35259246826172, -65.21533203125, 0.8775209188461304, -16.694398880004883, -40.84415054321289, -94.63697814941406, 42.90306091308594, 26.096271514892578, 7.958330154418945, 40.80244064331055, 5.476954936981201, 26.827987670898438, 52.72599411010742, 8.387359619140625, 26.408437728881836, -23.148651123046875, -2.535485029220581, -57.85499954223633, -71.36241912841797, -6.120057582855225, 63.74748992919922, -92.79012298583984, -57.95652770996094, -93.82091522216797, -40.852630615234375, 67.97346496582031, 46.857749938964844, 43.25578689575195, -1.3044389486312866, 25.065513610839844, 25.42801856994629, 3.76879620552063, 72.4640884399414, -58.45262908935547, -6.122570514678955, 12.724128723144531, -74.62474822998047, -6.34901762008667, -38.8933219909668, 58.94427490234375, -24.873464584350586, -54.317901611328125, -9.123313903808594, 40.357948303222656, 3.2002382278442383, 76.73692321777344, -27.446762084960938, -98.92939758300781, 9.002674102783203, -17.99653434753418, 15.686565399169922, 52.45521545410156, 10.648468971252441, -7.822574138641357, -31.019893646240234, -43.50856399536133, -12.384780883789062, 41.508113861083984, 91.13353729248047, 28.62318229675293, 65.32117462158203, 40.4678840637207, 30.95758819580078, -101.85086822509766, -12.686431884765625, -13.010540962219238, -22.936697006225586, -31.861738204956055, -48.29259490966797, 9.474836349487305, 18.18001365661621, -9.245521545410156, 36.25210952758789, 109.75324249267578, 18.614192962646484, -1.2693767547607422, -42.7911262512207, -39.486961364746094, 58.316219329833984, 17.69643211364746, -73.64387512207031, 15.936359405517578, -71.38442993164062, -55.430335998535156, 53.13019561767578, 28.73735809326172, 28.40433692932129, -99.88813781738281, 31.339067459106445, 61.43164825439453, -36.518070220947266, 33.82607650756836, -45.097354888916016, -16.650863647460938, -5.499502182006836, -64.4203872680664, -57.211124420166016, 39.523216247558594, -63.64631652832031, -0.9831814169883728, 5.011607646942139, 29.43648910522461, 7.295795440673828, -70.69491577148438, 18.396516799926758, -68.23252868652344, 41.47854232788086, -87.60580444335938, -16.308290481567383, -40.067222595214844, -66.62841033935547, -51.45027542114258, -77.32931518554688, -64.27962493896484, -78.74166870117188, 6.8472981452941895, -73.74470520019531, -54.17141342163086, 93.83521270751953, 61.385440826416016, -15.864904403686523, -17.391786575317383, 73.28101348876953, -30.083486557006836, -15.47230339050293, 53.237491607666016, -86.23622131347656, 54.34781265258789, -56.100337982177734, -53.27433395385742, -2.2404537200927734, 32.79126739501953, 42.296409606933594, -125.52888488769531, -8.31857967376709, 25.878583908081055, 18.053298950195312, 17.217708587646484, -5.163176536560059, 69.56845092773438, -5.34847354888916, -17.783966064453125, 80.5064468383789, 13.948710441589355, 64.25310516357422, 20.68159294128418, 24.285518646240234, 41.386653900146484, -153.92947387695312, 29.716819763183594, -13.928596496582031, 71.71896362304688, -106.83941650390625, -17.22917938232422, -39.83736801147461, 97.97474670410156, 98.48503112792969, -113.26789855957031, 21.159666061401367, 66.84013366699219, 57.853389739990234, -37.120140075683594, 58.5662956237793, -58.28594970703125, 80.42056274414062, 77.6325912475586, -62.22838592529297, -48.98884201049805, 94.5706787109375, 64.51717376708984, 7.725922107696533, 52.85401153564453, -87.43275451660156, 30.028995513916016, -33.981876373291016, 12.128426551818848, 50.57687759399414, -16.01580238342285], [42.72672653198242, -45.84077453613281, 1.5442135334014893, 36.91885757446289, 9.070098876953125, -23.59417724609375, 12.608920097351074, -33.948509216308594, 24.17242431640625, -10.579784393310547, 76.38959503173828, 27.616374969482422, 49.161441802978516, -24.354848861694336, 72.13436889648438, 45.587982177734375, 87.14478302001953, -2.9937472343444824, -57.9099006652832, 12.645581245422363, -35.91188049316406, 7.118991851806641, 55.30461502075195, 1.6664667129516602, 32.63374328613281, 5.5961480140686035, -5.328983783721924, 43.1795539855957, -12.798481941223145, 84.9975814819336, -6.991944313049316, -1.9834707975387573, -109.17436218261719, -81.53633117675781, 10.160183906555176, -8.270339965820312, 9.412586212158203, 59.561771392822266, -102.74244689941406, 25.723478317260742, -102.28860473632812, -44.923126220703125, -16.843040466308594, 49.37925720214844, -56.62495422363281, 56.28976058959961, -2.5011329650878906, 35.36824417114258, 49.857452392578125, -1.9286725521087646, -148.35496520996094, 53.044761657714844, -9.693624496459961, 61.12185287475586, -28.133663177490234, -33.525718688964844, 10.59462833404541, 34.57887268066406, 21.977155685424805, 2.4758799076080322, 18.422313690185547, -54.63789749145508, -82.48052978515625, -87.6793212890625, -108.20291137695312, 21.29804039001465, 2.3432767391204834, 30.24021339416504, 9.969453811645508, 101.5130844116211, -52.704750061035156, -40.49274826049805, -91.62323760986328, 111.17365264892578, 55.74566650390625, -49.919925689697266, -88.33734130859375, -112.8586654663086, -31.263229370117188, 95.09780883789062, 67.4770736694336, -7.399803161621094, -54.20823669433594, 69.32235717773438, 3.0871098041534424, 65.59020233154297, 20.397216796875, 17.117504119873047, 20.31551170349121, -36.72875213623047, 41.37250900268555, -1.2776581048965454, -66.82343292236328, 89.40278625488281, 29.774141311645508, -48.51167678833008, 66.204833984375, -18.040924072265625, 6.743138313293457, 22.67503547668457, 28.29351043701172, 0.3510623574256897, -74.81795501708984, -15.93139934539795, 3.21399188041687, 45.79851150512695, -44.196754455566406, -19.368038177490234, 82.46728515625, -58.64254379272461, 3.7301924228668213, -54.49125289916992, -60.009185791015625, -47.52528381347656, 37.638206481933594, 21.382417678833008, -5.90113639831543, 19.879314422607422, 40.236183166503906, -14.696249008178711, 18.736835479736328, 12.642473220825195, 52.53453063964844, 16.223731994628906, -4.403297424316406, -39.12617492675781, 152.57200622558594, -32.568172454833984, -22.39130210876465, -21.462509155273438, -76.86026000976562, 9.776411056518555, -9.897598266601562, -56.557273864746094, 19.727691650390625, 43.336788177490234, -41.024147033691406, -1.8129106760025024, -96.33818054199219, -64.3619613647461, 39.8841667175293, -51.30216979980469, -13.580296516418457, -19.512399673461914, -28.169757843017578, -11.958067893981934, -13.620957374572754, -0.3505670130252838, -6.417037010192871, -4.194746971130371, -63.87260055541992, -69.41361236572266, 44.042903900146484, 88.91683959960938, -77.46766662597656, -14.384799003601074, -19.203723907470703, -52.80835723876953, 9.428210258483887, 15.404529571533203, 67.11601257324219, -66.35185241699219, 16.521791458129883, -26.27997398376465, -12.023893356323242, 1.5701824426651, -5.796148777008057, 58.057498931884766, -11.17110824584961, 33.435455322265625, -118.10978698730469, -6.956139087677002, 50.596595764160156, -46.990684509277344, -39.03158950805664, 31.707414627075195, 28.117534637451172, 71.25748443603516, 6.373891830444336, -11.919135093688965, -72.6873779296875, 82.0665054321289, -61.34830093383789, -77.00855255126953, -19.657920837402344, 46.79076385498047, -38.87641143798828, -5.671899795532227, -107.13615417480469, 16.09394645690918, 24.843368530273438, 64.98847961425781, 22.70400619506836, -40.115264892578125, 68.14591979980469, -114.43008422851562, 16.1929988861084, 27.309158325195312, 64.24368286132812, 23.55855941772461, 13.64101791381836, -62.72273635864258, -21.5789737701416, 39.3455696105957, -40.31692886352539, -4.861533164978027, -29.730606079101562, -3.4049439430236816, 35.49651336669922, 18.514339447021484, 85.97503662109375, -34.116119384765625, -19.29986000061035, 0.6255363821983337, 31.878599166870117, -5.540097713470459, -42.33523178100586, -31.612449645996094, -26.158693313598633, -31.961015701293945, 2.167083501815796, -37.53496551513672, -73.68226623535156, -56.868350982666016, 25.86724853515625, -11.727090835571289, 66.94244384765625, -16.612110137939453, 42.533103942871094, 13.758646011352539, -18.805923461914062, -28.239551544189453, -42.54755783081055, 44.43675994873047, 19.617778778076172, 9.920910835266113, 30.374568939208984, -41.880123138427734, 60.12156677246094, 66.11611938476562, -0.30318400263786316, 39.98322677612305, -75.56314086914062, -22.09486198425293, -40.300689697265625, -32.60525894165039, 67.95084381103516, 14.958434104919434, 47.730159759521484, -9.04132080078125, -12.05602741241455, 3.1090211868286133, 13.118317604064941, -105.27339172363281, -45.32746505737305, -97.26730346679688, -29.842992782592773, 94.57405090332031, -3.9148120880126953, -90.6863784790039, 31.506494522094727, -36.66225051879883, -61.44577407836914, 28.48261070251465, -27.66336441040039, -54.42002487182617, 45.9308967590332, 83.68028259277344, -5.797834873199463, 41.112022399902344, -50.300933837890625, -10.87381362915039, -56.85198974609375, -8.874746322631836, 20.377574920654297, -55.81959915161133, -46.0000114440918, -39.13425827026367, -66.49864959716797, 67.75988006591797, 76.13489532470703, -25.245662689208984, 7.883172988891602, 60.80081558227539, 1.9006215333938599, 47.107364654541016, -139.74404907226562, -86.14031219482422, -8.003617286682129, 17.9542236328125, 27.172718048095703, 35.05265426635742, -12.66750717163086, 22.573684692382812, 5.833392143249512, 26.707204818725586, -50.434120178222656, -5.528013706207275, -13.030116081237793, 8.017001152038574], [-59.041404724121094, 60.30848693847656, 6.881510257720947, -81.57540893554688, -10.893688201904297, -24.275056838989258, -34.766273498535156, -0.42490071058273315, -41.6353759765625, 71.10498809814453, 64.88932037353516, 23.490968704223633, -30.60934829711914, 44.18327331542969, -32.65628433227539, 18.40333366394043, 11.658373832702637, 33.010013580322266, -18.956050872802734, 104.21318817138672, 45.70464324951172, 10.761441230773926, 2.73608136177063, -48.14594268798828, -11.481698989868164, 20.488964080810547, -69.10343170166016, -13.640364646911621, -28.012704849243164, -12.048385620117188, -40.64668655395508, -8.246848106384277, -20.992507934570312, -105.80889892578125, -21.780529022216797, 73.51396179199219, -67.4572525024414, -41.98427963256836, -4.327690124511719, -23.739797592163086, -4.156299114227295, -85.7166748046875, 15.613966941833496, -90.48345947265625, -7.430960178375244, -64.18578338623047, -44.090736389160156, -16.82784652709961, -71.86895751953125, 62.19953155517578, -81.0314712524414, 66.69519805908203, 120.08226013183594, 20.94818115234375, -5.581022262573242, -11.893173217773438, 93.89590454101562, 21.160451889038086, -17.420745849609375, 1.1145262718200684, 97.56855010986328, 29.804906845092773, 3.319857358932495, 49.81666946411133, -7.855505466461182, -9.219435691833496, 69.75717163085938, 38.75972366333008, -39.05569839477539, -21.78376007080078, 27.186176300048828, 47.21820831298828, 25.930700302124023, -128.09999084472656, -57.055419921875, -11.639546394348145, 8.686543464660645, 5.929182529449463, 50.156097412109375, 56.925575256347656, 7.796957969665527, 15.554435729980469, 97.4045639038086, 99.00718688964844, 72.03620910644531, -18.44219970703125, -62.58369827270508, 89.13741302490234, -51.825077056884766, -23.88232421875, -5.751883506774902, -36.64394760131836, -40.935874938964844, 14.046667098999023, 100.09028625488281, 50.90802764892578, 41.466285705566406, -28.65790367126465, -79.54498291015625, -48.802886962890625, 39.87705993652344, 54.22974395751953, 101.26107025146484, -6.879883289337158, 58.78130340576172, -12.728750228881836, 51.01320266723633, -39.59243392944336, -11.950355529785156, 6.9518656730651855, 36.468990325927734, -83.39962005615234, -32.819847106933594, 42.3515739440918, -23.224498748779297, 12.50464153289795, 76.06120300292969, -21.44913101196289, -20.20471954345703, 12.218391418457031, -97.5225601196289, 76.34736633300781, -38.8012580871582, -2.613562822341919, -13.677820205688477, 20.865949630737305, -13.621878623962402, -3.9595530033111572, 30.03884506225586, 41.356422424316406, 20.154644012451172, -17.07459831237793, -26.15334129333496, 14.220155715942383, 72.10829162597656, -82.9990005493164, 11.502782821655273, 3.8616366386413574, -159.3269500732422, 1.2176884412765503, 47.99154281616211, 18.30934715270996, -2.121051788330078, -24.95134735107422, 16.22435188293457, 13.815410614013672, -65.6213607788086, -18.839153289794922, 2.5287652015686035, 30.540576934814453, 26.990520477294922, -13.593952178955078, 20.108627319335938, 9.804630279541016, 33.76640319824219, 64.43214416503906, -105.27296447753906, -132.768798828125, -26.821550369262695, 59.089927673339844, 55.59063720703125, 21.310789108276367, 46.88691329956055, -49.966392517089844, -15.227953910827637, -8.011380195617676, -17.348722457885742, -29.913297653198242, 84.75572204589844, 47.775299072265625, -54.53153991699219, 66.4417953491211, 2.4215502738952637, 57.68210220336914, -26.19522476196289, 0.8039613962173462, -48.37778091430664, 24.674936294555664, -79.60624694824219, -78.74093627929688, -49.730648040771484, -18.622222900390625, 80.83070373535156, 79.66325378417969, 39.399574279785156, 1.8622113466262817, -12.304705619812012, 33.93927764892578, 29.400358200073242, 25.99962615966797, 15.627524375915527, -18.960527420043945, -30.657297134399414, -4.833800792694092, 96.54510498046875, 0.16756215691566467, 26.226865768432617, 10.601151466369629, 63.9923210144043, 59.531150817871094, 9.142349243164062, -27.74557113647461, -116.00362396240234, 25.190439224243164, -36.56489181518555, -91.46993255615234, 41.86810302734375, -64.02820587158203, 90.75071716308594, -20.394866943359375, -19.18740463256836, -21.257568359375, 44.772342681884766, 0.19966135919094086, 56.51813888549805, -18.942468643188477, -5.844790935516357, 23.205909729003906, 6.748477935791016, -36.84297180175781, -22.86870574951172, 28.60740089416504, -20.47726821899414, -40.93302917480469, 5.847274303436279, -104.57244873046875, 27.2641658782959, 40.791465759277344, -35.26708221435547, 32.043357849121094, 28.686180114746094, 31.61461067199707, -21.643978118896484, 33.83404541015625, -32.03913116455078, 42.259315490722656, 54.63452911376953, -42.29962921142578, -72.579833984375, 31.473020553588867, -64.98136901855469, -79.12612915039062, 3.2348473072052, -56.7001953125, -44.442264556884766, 79.07328796386719, -69.93025207519531, -77.41073608398438, 28.480091094970703, -0.7001513242721558, 48.69520568847656, -85.66202545166016, -12.26011848449707, 31.289365768432617, 42.89098358154297, 102.67728424072266, 63.39579772949219, -154.2745361328125, -28.325929641723633, 45.296730041503906, -45.573787689208984, 27.195390701293945, 26.4191951751709, -93.81131744384766, -144.20413208007812, -7.677944660186768, -48.55745315551758, 27.524951934814453, 38.899864196777344, -16.350318908691406, 45.801025390625, -13.448819160461426, -34.09645080566406, -60.0268440246582, 31.117704391479492, -60.96500778198242, -77.19499969482422, -26.074934005737305, 41.75209045410156, 43.51605224609375, -23.949329376220703, 36.30079650878906, -14.83729362487793, 56.37787628173828, 56.08917999267578, 31.38376235961914, -1.4409620761871338, -47.59292984008789, -59.87831497192383, 24.304603576660156, 65.48621368408203, 12.943580627441406, 64.35481262207031, 75.60874938964844, 108.80298614501953, -0.47875648736953735, -27.372699737548828, 75.8790054321289, 79.5008773803711, 89.49872589111328]], "dtype": "float32"}, {"__nd__": [[1.3819345235824585, 1.2578355073928833, 0.37328392267227173, 1.534087896347046, -0.8449076414108276, -1.1490676403045654, -0.7613067626953125, -0.40503591299057007, 0.538020670413971, 0.8997906446456909, -1.8190754652023315, 0.7475229501724243, -0.6307653188705444, 0.8021575212478638, 1.5014889240264893, 2.6237053871154785, -0.30753767490386963, -0.7850286960601807, 1.405301809310913, -0.4127040505409241, -1.1919912099838257, 0.9118468165397644, 0.624090313911438, 0.78182452917099, -1.340045690536499, -1.0582057237625122, -0.26308873295783997, -1.1060843467712402, 2.3159236907958984, -0.4615221619606018, -1.399380087852478, 1.2156699895858765, -1.6357098817825317, -1.1992623805999756, -0.03771692514419556, 0.4906955063343048, 2.451525926589966, 0.987883985042572, 0.40637823939323425, -0.6979857683181763, 1.0657997131347656, -0.29447370767593384, 1.675737977027893, 0.6059776544570923, -1.0121062994003296, 0.9489986896514893, -1.7635372877120972, -1.0297720432281494, 1.2138371467590332, 1.0034687519073486, 0.8651611804962158, -0.09434366226196289, -0.40891122817993164, -0.6734285950660706, 0.4045853912830353, -0.4412773549556732, 0.9189210534095764, -1.7328733205795288, 0.749359667301178, 0.15192878246307373, -1.5818020105361938, 0.5381127595901489, -1.1830824613571167, -0.04309886321425438, -0.2692875266075134, -1.2621946334838867, 1.5884814262390137, 0.17743265628814697, -2.935185432434082, 0.25776758790016174, 1.2432100772857666, -0.06342202425003052, -0.3340735137462616, -0.4253769814968109, 0.5656189918518066, -0.12997844815254211, -1.000393271446228, -0.0003998642205260694, -1.2563557624816895, 2.6634371280670166, 0.224604994058609, -1.1989578008651733, 1.7911248207092285, -0.9744672775268555, -2.316765546798706, 1.002112627029419, 2.4554245471954346, -0.5839399695396423, -0.18300862610340118, 0.7040297985076904, -2.5924460887908936, -1.3734354972839355, -1.1341204643249512, -0.319435715675354, 1.1712678670883179, 0.5580894351005554, 0.8732467889785767, 2.1387665271759033, -0.7547187805175781, 0.3143038749694824, 0.38464727997779846, 1.8723493814468384, 0.8151834607124329, -0.6716868877410889, -0.19711393117904663, -0.3769861161708832, 0.041474297642707825, 0.6868665814399719, 0.9204365611076355, -0.5119025111198425, -0.4315067529678345, -1.063223958015442, 0.4477117955684662, -1.1565141677856445, 0.766986608505249, -0.36805763840675354, -0.7254326343536377, -0.21834169328212738, 0.5678995847702026, -1.1364800930023193, 0.21799248456954956, -0.19910092651844025, 1.1186788082122803, 2.5191538333892822, -1.6620872020721436, -0.4485277533531189, -0.584269106388092, -0.18978844583034515, -1.525490403175354, 0.1157265231013298, 0.4337047040462494, 2.978569269180298, -0.778562068939209, 0.4943441152572632, 0.10680465400218964, -0.7928845286369324, -1.4647443294525146, -0.6944039463996887, 1.578726887702942, 0.8200064301490784, -0.5415548086166382, 0.20680415630340576, 0.41682738065719604, 0.8839683532714844, -0.8141807913780212, 0.340754896402359, -0.7466642260551453, -0.6798798441886902, 1.41773521900177, -0.3064119219779968, -0.9570285081863403, -1.1857991218566895, -1.5062319040298462, 0.7556621432304382, -0.5871100425720215, -0.24045468866825104, -0.5976466536521912, 0.057009417563676834, 0.7499202489852905, 0.4504481256008148, 1.0276843309402466, 0.6539419293403625, -0.5911375284194946, 0.8672031164169312, -1.8562403917312622, -0.22738653421401978, 0.13580116629600525, 0.6275891065597534, -0.3674287497997284, 1.0676239728927612, -0.006629908923059702, -0.635842502117157, 1.4658303260803223, 0.8220378160476685, -0.30592581629753113, 0.5349497199058533, -1.1744482517242432, 0.016248399391770363, -1.6637821197509766, 0.2916114330291748, 0.4982561469078064, 0.40470728278160095, 1.148050308227539, -2.0594308376312256, 1.7192829847335815, 1.222087025642395, 0.12959448993206024, -0.8318332433700562, -0.8926771283149719, -0.00037189756403677166, -0.6252549886703491, -0.4367216229438782, 1.253466248512268, -0.3834892511367798, -0.41424232721328735, -0.5927780866622925, 0.4789433479309082, -0.4666774868965149, 0.02496153488755226, -1.116709589958191, -0.5189864635467529, 0.23741059005260468, 0.7644845247268677, -0.09536357969045639, -0.8483507633209229, -1.200828194618225, 0.3871600329875946, -1.6312140226364136, 0.6053641438484192, 1.0516546964645386, -0.002090450143441558, 0.8770073652267456, -1.969301700592041, -0.10010744631290436, -0.23890671133995056, -0.6351802945137024, 0.22031207382678986, 0.8578940033912659, -1.644797444343567, -0.38532888889312744, -0.15700238943099976, -1.7683464288711548, -0.6694075465202332, -0.2855662703514099, -0.16866013407707214, 0.031898170709609985, -0.281032919883728, -0.08016911149024963, 1.287555456161499, -0.18104290962219238, 2.1720383167266846, -0.4164324402809143, -0.21668967604637146, -0.9665735960006714, -0.5699725151062012, -1.0863698720932007, 1.1254061460494995, 2.5435256958007812, 0.5664865374565125, -2.059786796569824, 0.08704912662506104, 0.9043220281600952, 2.3763585090637207, -0.5535728931427002, -0.619717001914978, 1.0731925964355469, 0.9215898513793945, -0.8216153383255005, 1.1521412134170532, 0.5273988246917725, -0.6107574701309204, 1.706610083580017, 1.5433149337768555, 0.0668850764632225, -0.525732159614563, 0.7246038913726807, -0.21873047947883606, -0.6999914050102234, -1.149369239807129, -1.536700963973999, 0.4979188144207001, 1.2890329360961914, -0.9673747420310974, 0.038465794175863266, 0.8494148850440979, 1.527864694595337, -1.2513062953948975, -0.9216545224189758, -1.1341925859451294, -1.1124500036239624, 1.1151546239852905, -0.2626122534275055, -0.9600492119789124, 1.3635163307189941, 1.1909217834472656, -0.22361023724079132, 2.2614991664886475, -1.2426121234893799, -0.26263630390167236, -0.5221760869026184, -1.005895733833313, -0.5300105214118958, -0.08113182336091995, -0.3655242323875427, -1.2943531274795532, 0.7396886944770813, 0.3375234305858612, -0.43402647972106934, 0.3130691945552826, -0.9884618520736694, -0.5551072955131531, 2.1866912841796875, -0.34259915351867676, -0.8641636967658997, -0.0633622407913208, 0.2301650047302246, -0.2358252853155136, -1.173180103302002, 2.4490020275115967, 1.012599229812622], [0.33321812748908997, -0.8128640055656433, 1.0839433670043945, 0.14998292922973633, 1.2859132289886475, -0.011095528490841389, -0.34988582134246826, 0.0958513468503952, -0.060469526797533035, -1.0588557720184326, -0.40934401750564575, -2.349897623062134, 0.570467472076416, -1.0611001253128052, -1.8856056928634644, -1.0555444955825806, -0.14315007627010345, 0.31322944164276123, -1.3927708864212036, -0.12205886095762253, -0.24875356256961823, -1.0206888914108276, 0.6536548733711243, 1.2657238245010376, 0.06640429049730301, -1.2717863321304321, -0.29170843958854675, 0.9768953323364258, -1.068405032157898, 0.3188163936138153, -0.6094940304756165, 0.5928736329078674, 0.7468526363372803, 0.08904504030942917, 0.6479485034942627, 0.04569769650697708, -0.719230592250824, 0.4011062979698181, 0.4030105769634247, 0.5949308276176453, -0.15273354947566986, -0.3354189395904541, -1.1548006534576416, -1.6352109909057617, -0.7283707857131958, -1.9883606433868408, 2.0018625259399414, 0.6915844082832336, -0.6375200152397156, -1.102063775062561, -0.6529739499092102, 0.5888910293579102, 0.15703432261943817, -2.3406968116760254, -0.31129711866378784, 0.17994548380374908, 0.3011236786842346, -0.24534419178962708, 2.054112672805786, 1.26710045337677, -0.9047011733055115, 0.7374687194824219, 0.5575297474861145, 0.10342706739902496, -0.6608530879020691, -1.9489306211471558, -1.2644646167755127, 1.4292265176773071, 1.2241765260696411, -1.8825175762176514, -1.0019738674163818, 0.36237525939941406, -0.5850450396537781, -1.6139847040176392, -0.31731995940208435, 0.8773714900016785, 0.15198685228824615, 0.2137378752231598, 0.5448685884475708, -0.06652534753084183, 1.5949268341064453, -0.18863052129745483, -1.7803395986557007, 0.10048715770244598, 0.49923232197761536, 1.022810459136963, -0.4913020133972168, 0.2677576243877411, -0.43456047773361206, -0.8715128898620605, 0.034204576164484024, 0.5562282204627991, 0.3038405478000641, 1.0547385215759277, -0.31627628207206726, -1.2349698543548584, 0.9032300114631653, 0.09244230389595032, -0.7901777029037476, 0.9186608195304871, 1.2920989990234375, 0.2856411039829254, -1.730669379234314, 1.0196056365966797, -0.28962022066116333, 1.0528134107589722, -2.5664312839508057, 1.487841248512268, 0.5652517676353455, -1.4572091102600098, 1.524240493774414, -0.08887934684753418, -1.0527173280715942, 1.5186207294464111, 1.6495685577392578, -1.110302209854126, -1.0680011510849, 0.5986927151679993, 0.28069549798965454, 1.9495124816894531, 0.71285480260849, -0.2056661695241928, 2.2268261909484863, -0.6263148188591003, 1.3196073770523071, 0.9840389490127563, -0.946583092212677, -1.4343056678771973, -1.8560658693313599, 0.25547242164611816, 0.3632194697856903, 2.488863706588745, -0.48952972888946533, -0.14708486199378967, 0.25414398312568665, 0.01640528067946434, 0.2517145276069641, -0.023595238104462624, -0.3307579457759857, 0.12820354104042053, -0.548774242401123, -1.3965073823928833, 0.6068058013916016, -0.46795862913131714, -0.8935266137123108, 1.003395915031433, -0.4277447760105133, -0.830853283405304, 0.42839571833610535, -1.0153228044509888, -1.2421457767486572, -0.033606089651584625, 0.9300690293312073, -0.27016040682792664, -0.21426080167293549, 1.668200135231018, -0.002223012037575245, -0.1951693445444107, 0.724301815032959, 0.649879515171051, -2.0620546340942383, -0.7335015535354614, 1.8526307344436646, 0.4775349199771881, -0.16631938517093658, 2.0400800704956055, 1.2789145708084106, -0.1340983361005783, 0.9808357357978821, 0.2708708643913269, 0.4899751842021942, -0.32517072558403015, -2.3299989700317383, -1.288224697113037, -1.6879689693450928, -1.0482474565505981, -1.3366234302520752, -0.7168753743171692, 0.6563772559165955, 0.8918299078941345, -0.12090475112199783, -1.581394076347351, 1.9301389455795288, -0.21340888738632202, -0.41263914108276367, -1.0438841581344604, 0.18861635029315948, -1.5881284475326538, -0.5333850383758545, 0.774118959903717, -1.188684344291687, -2.0388731956481934, 0.7483245730400085, 1.191140055656433, -1.4373888969421387, 1.4651153087615967, -0.44071152806282043, -1.2396963834762573, -1.6139549016952515, 1.3599817752838135, 0.5729902982711792, -1.6909756660461426, -0.49359747767448425, -0.5901588201522827, 1.217248558998108, -0.16417735815048218, 2.0273263454437256, 0.613948404788971, 0.8417619466781616, 1.2989215850830078, -0.8629264235496521, 0.786523163318634, -0.39898616075515747, -1.5479227304458618, 0.25712454319000244, 0.6742176413536072, 0.8153076767921448, -0.3719637393951416, -0.9270429611206055, 0.895442545413971, -0.6008418798446655, 0.5430209636688232, -1.5543279647827148, 0.6936157941818237, 0.23192942142486572, -0.2917161285877228, -0.07032785564661026, 0.03108394518494606, 0.15426938235759735, -0.20278766751289368, 0.5116382837295532, -0.13426253199577332, -2.2414040565490723, -2.12894868850708, 1.1977710723876953, -1.3363533020019531, -2.1080210208892822, 2.0397608280181885, 1.7201765775680542, 2.2779831886291504, -1.4115949869155884, 1.328286051750183, 0.12478505074977875, 1.1953868865966797, -0.3614405393600464, -0.5746601223945618, -0.046700965613126755, 1.8154027462005615, 1.8518213033676147, -0.021214550361037254, 0.8089428544044495, 0.7215583920478821, 1.9347668886184692, 1.9612325429916382, -0.0906541720032692, 0.6279666423797607, -0.23635800182819366, -0.5703960657119751, 0.12881965935230255, 0.4105556905269623, -1.1704877614974976, -0.07191765308380127, -0.6244351863861084, -0.610149621963501, -1.199118733406067, 0.7343889474868774, 0.865265429019928, 0.6328552961349487, -0.507520318031311, 0.7277488112449646, -0.28736281394958496, 0.5464158058166504, 1.0671049356460571, -1.173895239830017, 1.931952714920044, 0.8517199158668518, -0.16102762520313263, 0.6446785926818848, 0.20713184773921967, -0.9731845855712891, -0.10346081852912903, 0.1665668785572052, 1.0430618524551392, 0.5552619695663452, -0.8343592882156372, 1.7727293968200684, -0.38829588890075684, -1.3892338275909424, -1.3758339881896973, -0.7192737460136414, 1.1279621124267578, -0.43041756749153137, 0.021327242255210876, -0.7377650737762451, -1.3651797771453857, 1.2098934650421143, -0.7257270216941833, -0.6557111144065857, 1.5005371570587158, -1.5837749242782593], [-1.3183293342590332, -0.242831751704216, 0.01261209324002266, 0.300968736410141, 1.8018172979354858, 0.4790547490119934, 0.4985244870185852, 0.6891417503356934, 0.7406290173530579, 0.840873122215271, -1.3805168867111206, 0.1686258763074875, -0.8138173222541809, -0.9844797253608704, 1.3390071392059326, 1.511421799659729, 1.1793874502182007, 0.5336053371429443, -0.37145495414733887, 0.0501011461019516, 1.5793681144714355, 1.8540899753570557, 0.5785751342773438, -0.1547110676765442, -0.2677646577358246, 0.5445168614387512, 0.007010092958807945, 0.3412981629371643, -0.8917513489723206, 0.4284805357456207, 1.92764413356781, -2.043835163116455, -0.2833549976348877, 0.7696985602378845, -0.2135447859764099, -0.9341709017753601, -1.7992820739746094, -1.9272187948226929, 0.9472530484199524, -0.14293068647384644, -0.23048771917819977, -1.137752890586853, -0.8854451179504395, 0.23748375475406647, 0.4702948331832886, 0.2665160000324249, 0.39423811435699463, 1.108317494392395, -1.317457914352417, -1.3781254291534424, 0.4945012032985687, 0.1412467360496521, -0.3612806499004364, 1.914688229560852, -1.226853370666504, -0.019867343828082085, -0.6143262982368469, 0.12342150509357452, 0.03314777463674545, 0.037586409598588943, -0.7971311807632446, -0.9844341278076172, 1.3156445026397705, -0.8024287223815918, 0.23888857662677765, -0.5661656856536865, 0.4860873818397522, -0.02879183739423752, 0.5596439838409424, 1.68976628780365, -1.1521376371383667, 1.076853632926941, -1.4746443033218384, 0.7715186476707458, 0.16242659091949463, 1.2835595607757568, -0.7506396174430847, 0.631465494632721, -0.3223249316215515, 1.8720065355300903, 1.5839126110076904, -0.018295694142580032, -0.4517439901828766, 2.392510414123535, 0.5190349817276001, 1.43415367603302, -0.3578333258628845, -2.8945119380950928, 0.231508269906044, 1.4319380521774292, -0.06338460743427277, -0.3074716031551361, 0.23101931810379028, -1.0497881174087524, 0.5689697861671448, -0.08671664446592331, -0.22159995138645172, -0.605699896812439, 0.5409238934516907, 0.41401398181915283, 3.068464517593384, 2.0634074211120605, -0.9046218395233154, -1.140913724899292, -0.43553104996681213, 0.02610512264072895, 0.8938183784484863, -0.5358671545982361, -1.3227325677871704, 1.0622148513793945, 0.517396867275238, -0.06805015355348587, 1.2348084449768066, 0.7004555463790894, 0.4202781617641449, -0.12129442393779755, 1.6866915225982666, -1.8898307085037231, -1.6047015190124512, 1.9196891784667969, -0.26788315176963806, -0.5411705374717712, -0.09091895818710327, -0.17907805740833282, 1.4729679822921753, -0.5811024308204651, -0.3535902500152588, -0.8828630447387695, -0.6983548402786255, -1.4651880264282227, -0.45103827118873596, 0.641475260257721, -0.6820361018180847, -1.4601788520812988, -0.8239927887916565, 0.36325332522392273, 1.543871521949768, 1.9143069982528687, 0.42792168259620667, -1.3440314531326294, -0.4898785948753357, -0.9593333005905151, -1.8400123119354248, 0.2677402198314667, -1.7608833312988281, -1.4279472827911377, 0.3284549117088318, 0.4336576759815216, 0.003227720968425274, -1.353236198425293, -1.3264530897140503, -0.36647361516952515, -0.12744301557540894, 1.3315882682800293, 1.291403889656067, -0.8695331811904907, -1.7240279912948608, -1.8464710712432861, 0.9122782945632935, -0.8622444272041321, 0.05027389153838158, -1.8248250484466553, -2.2146081924438477, -2.114424228668213, 0.979153037071228, 0.5623271465301514, -2.2351722717285156, -0.48275744915008545, -0.7189000844955444, 1.0754846334457397, 1.5442588329315186, 0.25152260065078735, 1.865700125694275, -1.411323070526123, -0.989868700504303, -1.5128763914108276, 0.3999766707420349, -0.6357410550117493, 0.5977541208267212, -0.4463382661342621, 0.49797552824020386, -0.03249260410666466, 0.20023536682128906, 0.4515393376350403, 0.8756994605064392, 2.2826035022735596, -0.57450270652771, 2.199051856994629, 0.04080111160874367, -0.05382406339049339, -1.1988638639450073, 0.18182824552059174, 0.006577331572771072, -0.0022921187337487936, 2.0715646743774414, -1.6801759004592896, -0.0990055650472641, 0.5473366975784302, 0.6155396103858948, 1.4403071403503418, -2.266993284225464, 0.1900789886713028, 1.808223009109497, 0.03216981515288353, -1.6874215602874756, -0.6413317322731018, -1.164282202720642, -0.541333019733429, 0.03554774820804596, -0.5594634413719177, 0.2892104685306549, 1.2888816595077515, -1.155348300933838, -0.5576749444007874, 0.2091929018497467, 0.35537853837013245, 1.001503586769104, -0.932744026184082, -0.6355579495429993, 2.2799489498138428, 0.9285957217216492, 0.3175741136074066, 1.4463603496551514, 0.1839444637298584, 0.4434279203414917, 0.83372563123703, -1.972977876663208, 0.7855547070503235, 1.9759700298309326, 0.6554564833641052, -0.019106945022940636, 0.15315347909927368, 1.4702879190444946, 2.801990509033203, 0.4193747639656067, 0.4013030529022217, 0.1606324464082718, -0.2863713204860687, -1.1097500324249268, 0.35879504680633545, 0.24979019165039062, 0.606744110584259, 0.5212329626083374, -0.6701646447181702, 0.37044280767440796, -0.7790271043777466, 3.064505100250244, 0.5042793154716492, -0.4987737238407135, -0.43220311403274536, 0.4062127470970154, -0.9290013313293457, 0.06516369432210922, -0.6140199899673462, 0.7139424681663513, -0.9541595578193665, -0.19342099130153656, 1.1139695644378662, -1.2155961990356445, -1.2925097942352295, -2.047896385192871, 0.7156396508216858, -0.4000293016433716, 1.2585806846618652, -0.45748355984687805, -0.6366528868675232, 0.6718745231628418, 0.48635268211364746, 0.22888992726802826, -0.806382417678833, 1.8829861879348755, -0.4179020822048187, -0.602705180644989, 1.6526004076004028, 1.582688570022583, 0.07820480316877365, 0.1271655112504959, 2.3975818157196045, 0.3284548819065094, -0.8457193970680237, 0.8116670250892639, 1.7500758171081543, -0.38264352083206177, 0.4024656116962433, 1.5660808086395264, -0.4593081474304199, -1.6505746841430664, 0.3715549111366272, 2.4635939598083496, -1.2509785890579224, 0.9820323586463928, -0.2932513654232025, 0.580834150314331, 1.923126459121704, 0.3613663911819458, 0.6566771864891052, 1.8545818328857422, 0.06845508515834808, -1.0035614967346191, 0.11890256404876709], [-1.0167362689971924, 0.9118075966835022, -1.385855793952942, -1.5301083326339722, -0.10108423978090286, 0.5919726490974426, 0.6915913224220276, -1.0116777420043945, -0.7382577061653137, 0.42949509620666504, -0.7171474695205688, 0.6616484522819519, -0.9844551682472229, 1.3653792142868042, -0.452288419008255, 1.2458465099334717, -1.4775633811950684, 1.0601623058319092, 0.8341898918151855, -0.4433470368385315, 0.10448410362005234, 0.0428503081202507, -1.3335378170013428, 1.6561576128005981, -0.5046522617340088, 0.45270565152168274, 0.38069725036621094, -0.546846866607666, -1.2479465007781982, 0.030318863689899445, -0.13459470868110657, -1.0155999660491943, 1.4254653453826904, -0.16256116330623627, 0.10950872302055359, -1.3763705492019653, 1.5891711711883545, 0.808770477771759, 0.2581883370876312, -0.06191159039735794, 1.3003671169281006, 2.067465305328369, -0.8821196556091309, -0.4666477143764496, -0.5590118169784546, -1.7516348361968994, 0.16845284402370453, -0.30106526613235474, -0.08032830059528351, 0.46454712748527527, 0.09216319769620895, -1.6137140989303589, 1.0083085298538208, -0.17318648099899292, 1.200676441192627, 0.12459661066532135, 1.07608962059021, -0.5861042141914368, 0.9104673266410828, -0.056189216673374176, -0.6236128807067871, -1.1894172430038452, -0.8802204728126526, -0.8387367725372314, -0.6682127714157104, 0.9275047183036804, 0.26154449582099915, 0.8063822984695435, -0.23611657321453094, 0.11694995313882828, -1.5081157684326172, 0.44605204463005066, -0.5942250490188599, 1.6328707933425903, 0.6627640128135681, -0.0640466958284378, -1.4383567571640015, 1.167522668838501, -1.7855066061019897, 0.010668357834219933, -0.9248291254043579, 0.9862866997718811, -0.9855376482009888, 0.6949174404144287, -0.5669447183609009, 0.018408263102173805, 0.03278699889779091, 2.1478569507598877, -2.1005630493164062, 1.311410903930664, -1.1776847839355469, 0.10734550654888153, 0.6850091814994812, -1.965707540512085, -0.6512247323989868, -1.5773335695266724, 1.8524667024612427, 0.5877775549888611, -1.3117257356643677, 0.2715320885181427, 0.8922353982925415, -0.43064364790916443, 0.7473517656326294, -0.6571215987205505, -1.9594782590866089, 0.2682815194129944, 3.366201639175415, -0.6929957866668701, -1.9482512474060059, 1.2745959758758545, 1.0224462747573853, 0.20016776025295258, -0.15477725863456726, 1.8489171266555786, 0.7725933194160461, -1.6042168140411377, -0.09199008345603943, -0.16633057594299316, -1.2654253244400024, 2.2128734588623047, 1.0500819683074951, -1.6299023628234863, 0.3031608462333679, 1.443593144416809, 0.5675544142723083, 0.2685563862323761, 0.15214253962039948, -1.5961735248565674, -0.12049167603254318, -0.1903344988822937, 0.9672735333442688, -1.770957112312317, 1.2652559280395508, -0.5621926784515381, 0.2678360342979431, -0.2854290008544922, -0.004731651861220598, -1.003965973854065, 0.12483712285757065, -1.3246737718582153, -0.29366761445999146, -0.7600445747375488, -0.19824819266796112, 0.3712090253829956, 1.2074390649795532, -0.08840823918581009, 0.35207295417785645, 0.46917909383773804, 0.10826122015714645, 0.17526090145111084, 2.0056111812591553, 0.17534010112285614, 1.5144236087799072, 1.2453429698944092, -0.10751179605722427, -1.1310752630233765, 0.20362235605716705, -0.2937266230583191, -2.3646531105041504, 1.5541399717330933, -0.13666123151779175, 1.310531735420227, -0.42329540848731995, 0.6576600074768066, -0.6025112867355347, -0.5423789024353027, 0.8649798035621643, 0.08787234127521515, 1.7828689813613892, 0.1702440232038498, -0.4762161374092102, -0.6901385188102722, -0.33298903703689575, -0.3894845247268677, 1.4644147157669067, -0.10088279098272324, -0.435878187417984, 1.8401391506195068, 0.16266465187072754, 1.155136227607727, 1.5337272882461548, -0.044183846563100815, 1.632132649421692, 0.7677455544471741, -0.9096975326538086, -0.6411930918693542, -0.32058224081993103, -0.5308199524879456, 0.1784314215183258, 0.017503639683127403, -1.1891849040985107, 1.5956995487213135, 0.052120476961135864, 2.0032427310943604, -0.4715360403060913, -1.2901235818862915, 0.2515833079814911, 1.2049353122711182, 0.17065750062465668, -0.24506501853466034, 1.0761991739273071, -1.1812547445297241, -0.5560457110404968, 0.5067436695098877, 0.5037943720817566, -1.9757113456726074, -0.8746798634529114, -1.1508681774139404, 0.3536633849143982, 0.9857938885688782, -0.23604191839694977, 0.03195034712553024, -0.7516792416572571, -0.33608078956604004, 0.07790148258209229, -0.0653086006641388, 1.096318244934082, -1.5322550535202026, 0.21785740554332733, 1.4712600708007812, -0.374350905418396, 0.15415601432323456, 0.9156120419502258, 0.7921218276023865, 0.5238972306251526, -0.8143703937530518, -0.903051495552063, 0.26966097950935364, -1.9198271036148071, 0.014534955844283104, -0.26653626561164856, -0.20744414627552032, 0.3253755271434784, 1.678888201713562, 0.614208996295929, 0.5782679915428162, -0.40773558616638184, -0.24576909840106964, 0.27416157722473145, 0.7842544913291931, -0.1769968867301941, 0.4565039873123169, -0.6140426397323608, -1.8416260480880737, -0.4192008078098297, 0.6612129211425781, -1.1243172883987427, -2.1888418197631836, 1.8791731595993042, -0.6323387622833252, -0.7901180386543274, -0.16457881033420563, 0.03375322371721268, -0.3850296139717102, -0.2391888052225113, 0.37280645966529846, -1.8982558250427246, -0.8453308343887329, 0.5057249665260315, -1.3135924339294434, 0.9117308259010315, -0.19128762185573578, 0.2702387869358063, 0.5296781659126282, 0.15040323138237, -0.18995629251003265, -0.529843270778656, -0.28707557916641235, 0.15578794479370117, 0.4485148787498474, -1.3922173976898193, 0.2550816833972931, -0.5619825720787048, 0.7193751335144043, -1.0678989887237549, 1.9550808668136597, 0.24502085149288177, 0.4022001326084137, 1.789393663406372, -0.9980694055557251, 0.13514922559261322, 0.3284480571746826, -0.5642049312591553, 1.3961278200149536, -1.1809558868408203, 0.09304739534854889, -0.792799711227417, 0.42528194189071655, -0.04611385613679886, -0.24847163259983063, -1.2433786392211914, -0.6688476204872131, 1.1756258010864258, 0.288149356842041, 1.0983006954193115, 1.112044095993042, -0.8561000227928162, 0.7443616986274719, -1.5952413082122803, -0.07556705921888351], [0.6287142038345337, 0.9967327117919922, -0.06554994732141495, -2.0889179706573486, -0.6379079818725586, -0.24815449118614197, 0.39008039236068726, -0.4634982943534851, -0.282044917345047, -0.5282955765724182, -0.2325185388326645, -1.794678807258606, 1.0318702459335327, -0.7783737182617188, 1.8570700883865356, -1.8737434148788452, 0.11068163812160492, -1.0436172485351562, 0.12733770906925201, 0.29089343547821045, -1.357055425643921, -1.9684836864471436, 0.10004385560750961, -0.5549697279930115, -3.1849820613861084, -1.1382261514663696, -1.741992712020874, -0.38609954714775085, -0.06460128724575043, -0.028902370482683182, 0.21889182925224304, 0.8113400340080261, -0.3922620117664337, 0.7407389283180237, -0.5524201393127441, -0.22846175730228424, -1.6634715795516968, 0.058979932218790054, 0.7366978526115417, -1.919007658958435, -0.2143159657716751, -0.554305911064148, 0.7170277237892151, -0.07213779538869858, 1.164767861366272, 1.696211576461792, 0.7305766344070435, -0.0682884231209755, 0.4645904302597046, 0.9280747175216675, 1.97493577003479, 0.24683330953121185, -0.520887017250061, -0.2931274175643921, 0.25087350606918335, -0.046376634389162064, -0.35060960054397583, -0.5090814232826233, -0.9044317603111267, -0.9880883097648621, -1.7789337635040283, 1.5230659246444702, -0.933721661567688, 0.33513203263282776, -0.630425214767456, -0.4252098798751831, -0.002677052980288863, 0.7163774371147156, -0.3608691096305847, 0.09007591754198074, -1.4021697044372559, -0.9569001197814941, -0.9405032992362976, 1.701066493988037, 0.9777267575263977, -0.18196696043014526, -0.21325717866420746, 0.8034727573394775, -0.662855327129364, -1.8466765880584717, 0.8399292230606079, 1.570316195487976, 1.7409693002700806, -0.12243258953094482, -0.4053053557872772, -0.41152554750442505, -0.5362851023674011, -1.2694569826126099, 2.0933704376220703, 0.07739664614200592, -0.9528590440750122, -2.019118070602417, 0.6285494565963745, -1.7257620096206665, 0.9696468710899353, 0.35935819149017334, -0.5156766772270203, 0.44523486495018005, 0.5965248942375183, 2.094900131225586, 2.152118444442749, 0.0731884092092514, 1.8567945957183838, 0.24104899168014526, 2.1132147312164307, 0.14368212223052979, 1.1755900382995605, 0.6339903473854065, 0.6110249757766724, -0.4442639648914337, 1.3699654340744019, 0.9231266379356384, -0.16557888686656952, 0.8443670868873596, 0.6479817628860474, 1.7875455617904663, -0.6801749467849731, -0.5584432482719421, 0.07089578360319138, 0.15762437880039215, -0.25645914673805237, 0.635598361492157, 1.0269873142242432, -0.11304101347923279, -0.49299731850624084, 0.30090513825416565, 0.26271870732307434, -1.5965090990066528, -0.23553557693958282, -0.08189799636602402, 1.410677194595337, -0.2294800728559494, -1.719010829925537, -0.9496504068374634, -0.26385876536369324, 0.564367949962616, 0.9065191745758057, 0.09182939678430557, -0.9174283146858215, -0.4743991494178772, -1.1930488348007202, 0.999012291431427, 0.3222973346710205, -0.0866982564330101, 1.74864661693573, -0.2803988456726074, -0.46208471059799194, -1.3332000970840454, -0.527718186378479, 0.14944973587989807, -0.6543304324150085, 1.0779000520706177, 2.124713897705078, -0.75644850730896, 0.912080705165863, -0.3890514671802521, 0.5947456359863281, 1.7458269596099854, -0.16480055451393127, 2.3253889083862305, 0.6229771375656128, -0.6209152340888977, -0.2202926129102707, -1.7638895511627197, 1.2709376811981201, -1.7621474266052246, 0.8062152862548828, -0.4347435534000397, 1.4240491390228271, -0.3956905007362366, -1.9883145093917847, -0.26280680298805237, -0.7861756086349487, 0.8085129857063293, 0.9191363453865051, -1.3347928524017334, 1.916797161102295, 0.13045796751976013, 0.8754913210868835, 0.7643594145774841, -0.7478770613670349, 0.02934756875038147, 2.162236213684082, 0.3447096049785614, 1.4137592315673828, -0.13921725749969482, 0.19334904849529266, 0.7524653673171997, 0.454550564289093, 0.6515942811965942, 1.602457046508789, -1.353738784790039, -1.6376181840896606, 0.16079604625701904, 0.6819830536842346, 0.03583239018917084, 0.03700244799256325, 0.010665159672498703, 1.5177242755889893, 0.19456352293491364, 0.04995904862880707, 0.6487442255020142, -0.4968666732311249, -0.2195330411195755, -0.48088884353637695, 0.62002032995224, -0.7888314127922058, -1.3283356428146362, -0.49465566873550415, -1.577372431755066, -1.0364971160888672, -0.349396288394928, 0.18333351612091064, -2.263983726501465, -0.6427596807479858, -1.0434315204620361, 1.8443231582641602, 0.7597872614860535, -0.012261150404810905, 0.3761475384235382, -1.8204751014709473, -0.6440842747688293, -0.5152974724769592, -1.7395131587982178, 2.188671827316284, 0.8323231339454651, 0.44551900029182434, -0.6689794063568115, 0.4519883692264557, 0.7448034286499023, -0.010561203584074974, 0.04802091792225838, 1.805924654006958, -0.03576751798391342, 0.6097032427787781, 1.1020948886871338, -1.606349229812622, -1.3132410049438477, -2.265272617340088, 0.21232187747955322, 0.2621671259403229, -0.6456538438796997, 0.33246535062789917, 1.465306043624878, -1.8466722965240479, -1.685585379600525, -0.5758363008499146, 0.8548386693000793, 0.03253817558288574, 0.737798273563385, -1.3992066383361816, -0.23292414844036102, -0.3783738911151886, -0.3250100910663605, -1.0993534326553345, -0.292253315448761, 1.6534970998764038, 0.4803648293018341, -0.007044432684779167, -0.4309980869293213, 0.6095631122589111, -1.7752306461334229, -0.018908584490418434, -0.4427575170993805, 1.428749442100525, 0.7822067141532898, 1.4725302457809448, -0.7333086133003235, -0.6405282020568848, -0.6233879923820496, -1.6848511695861816, 0.2667437195777893, -1.0009188652038574, 0.8424332737922668, 0.4267631471157074, 2.8384041786193848, -0.7922453880310059, 0.07054644823074341, -0.6641888618469238, 0.4577232003211975, -1.6285182237625122, -0.6630048155784607, 0.108489491045475, 0.029527992010116577, -1.3185921907424927, -0.8262457847595215, -0.2734951674938202, -1.8979530334472656, 0.7547102570533752, 1.540722131729126, -0.46424540877342224, -0.8323693871498108, 0.282978355884552, -0.311739444732666, 0.7165715098381042, -0.945264458656311, -0.2755420207977295, -0.2804858684539795, 0.6411945819854736, 0.10660412162542343], [-1.3761979341506958, -0.12546153366565704, -0.5659176111221313, 1.0662522315979004, 1.6728483438491821, 0.30707991123199463, 0.016905652359128, -1.086342453956604, -1.7649084329605103, 0.6802374124526978, -1.4046790599822998, -1.0714772939682007, -0.41265740990638733, -0.1840684711933136, 0.7699053883552551, -0.006797746755182743, -0.9030399918556213, 0.4841534495353699, -3.2389912605285645, -0.22919175028800964, -2.037208318710327, -0.4202035963535309, 0.330126017332077, 0.9590641856193542, 0.3150672912597656, 1.898173213005066, -1.4134832620620728, -0.5167627334594727, -0.8318324685096741, 0.2131652683019638, 1.4983710050582886, -0.7079852819442749, -0.28455132246017456, 3.169862985610962, -0.01931966282427311, -0.9416393041610718, 1.1158976554870605, 0.5516091585159302, 0.5243226885795593, -0.09324491024017334, -0.6196085810661316, 0.5281965732574463, -0.6663078665733337, -2.6899757385253906, 0.37260863184928894, 0.7079681158065796, -0.654094934463501, 2.3408002853393555, 2.7175121307373047, 2.258737564086914, -0.6628159284591675, 0.33683013916015625, -0.9804019927978516, -1.0102378129959106, 0.0033974896650761366, -0.7333583235740662, -0.09707169234752655, 0.9900134205818176, 0.3445044457912445, -0.6055113077163696, 0.8299364447593689, -1.4341368675231934, 0.6006748676300049, 0.19555917382240295, -0.6938336491584778, -0.6572081446647644, -3.2554562091827393, -1.105785608291626, 0.1620725393295288, 0.6616498231887817, -1.1880390644073486, 0.65335613489151, 0.16119608283042908, 0.3953203558921814, 1.0510414838790894, 0.5413463115692139, -0.8202161192893982, 1.233669400215149, 0.45784059166908264, -0.9402502775192261, -0.4468863010406494, -0.1991957724094391, 0.5255006551742554, 0.6804633736610413, -0.7590433955192566, -0.9398519396781921, 2.1773738861083984, -0.38693612813949585, 0.025129830464720726, 1.5397824048995972, -0.6441712975502014, -1.1285064220428467, -1.5800362825393677, 3.1258883476257324, 1.1951172351837158, 0.2642378807067871, 0.9801945090293884, 0.7692208290100098, -0.8056387901306152, -0.04178009554743767, 0.3795577883720398, -0.026147987693548203, 0.26166269183158875, -0.18766503036022186, -0.044090040028095245, 1.3028295040130615, -1.4761407375335693, -0.00037813937524333596, 1.6333928108215332, 0.7656478881835938, 0.25192058086395264, 0.6078231334686279, -0.19951829314231873, 0.34939834475517273, -1.761169672012329, 1.3921632766723633, 0.04408367723226547, -0.4762907326221466, -0.16928192973136902, -1.3806649446487427, -1.167691707611084, -0.9430537819862366, -0.632196843624115, -1.8294516801834106, 2.4677083492279053, -0.2626681923866272, 0.4553523361682892, -1.981327772140503, -0.34647372364997864, 1.1122368574142456, 0.7489652633666992, 0.8556405305862427, -0.35050415992736816, -0.920812726020813, 0.6391690373420715, -2.1030759811401367, 1.9053864479064941, -0.4208202660083771, 0.6132760047912598, 1.6022268533706665, 1.6162818670272827, 0.1737368106842041, -1.3133981227874756, -0.9022120833396912, 0.44712767004966736, 0.43786078691482544, -0.984177827835083, 1.6330279111862183, 0.058400530368089676, -1.7114983797073364, -1.5945870876312256, -1.7235958576202393, -0.3090493381023407, 1.8470532894134521, -2.2100510597229004, -0.05649096891283989, -1.3763427734375, -1.2344409227371216, 1.1822760105133057, -0.16657093167304993, 0.9691312909126282, 1.0480189323425293, -1.449479103088379, 1.723047137260437, 0.6372190713882446, 0.49823957681655884, -0.16554009914398193, 0.7100619077682495, 0.01663372479379177, -1.1663262844085693, -1.6215732097625732, 0.5912082195281982, 0.03762829676270485, 1.4995108842849731, -0.939538836479187, -0.6906369924545288, -2.00479793548584, -0.339306116104126, 1.3749301433563232, 0.2570483088493347, 1.9669709205627441, -0.9625357985496521, 0.27126774191856384, 0.2763214111328125, -0.7272459864616394, -0.32258883118629456, -0.1968059539794922, 0.10140414535999298, -1.0030361413955688, -0.06456254422664642, 0.3165832757949829, 1.9161431789398193, 1.1182245016098022, 1.6680680513381958, -0.4966577887535095, -1.6654773950576782, -0.4290864169597626, 0.6253319978713989, 0.21562530100345612, 0.41815945506095886, 1.1652405261993408, 1.863423228263855, 0.218309685587883, -0.5111664533615112, -0.5773435831069946, -1.640611171722412, -0.23849627375602722, -0.7845819592475891, 0.7096953988075256, -1.171834111213684, 0.47172027826309204, -0.46761956810951233, 0.041972700506448746, 1.2673799991607666, -0.05765998736023903, -0.4720323383808136, -0.4594344198703766, 0.7970782518386841, -0.6447253823280334, 1.2117799520492554, 0.21018382906913757, -0.24801060557365417, 0.4049263596534729, -0.35486406087875366, 1.3552707433700562, 2.2316067218780518, -2.0561604499816895, -1.5603630542755127, 0.5389848947525024, 0.48444315791130066, -0.3533633351325989, 0.30930495262145996, -0.504899263381958, 0.3415215015411377, 1.2282230854034424, 2.3690176010131836, -0.31517493724823, -0.8775359988212585, 2.042323589324951, 2.013324737548828, 0.18932275474071503, 1.8713805675506592, 1.2472288608551025, 0.19343119859695435, 2.194836139678955, 0.9263133406639099, -2.7140390872955322, 0.9376887679100037, 1.054327368736267, 0.7475379705429077, -1.690673828125, -1.4880046844482422, 1.0553871393203735, -1.8389049768447876, -0.08661176264286041, -1.0064820051193237, -1.1806968450546265, -0.19949354231357574, 1.1439507007598877, 1.0233334302902222, 0.5050414800643921, 0.6571298837661743, -0.5947818756103516, -1.137929916381836, 2.3757004737854004, 1.1255253553390503, 0.9324361681938171, -0.13896606862545013, -0.35516682267189026, -0.6875631213188171, -0.7146456241607666, -0.9204446077346802, 0.5116710066795349, 0.6928848028182983, -2.217189073562622, 0.04012693464756012, 1.2106271982192993, 0.9724863767623901, -1.0255084037780762, -0.2330855131149292, 0.03852435201406479, -2.015254497528076, -0.5360496044158936, 0.5819324851036072, 0.6916685700416565, -0.5962749719619751, 0.35122227668762207, 1.4770543575286865, 0.48424530029296875, -1.1708685159683228, -1.0672484636306763, 0.6702155470848083, 0.7909079790115356, -0.7358714938163757, 0.048216380178928375, 0.3182764947414398, 1.2952719926834106, -0.055463388562202454, 0.1197390928864479, -0.5495414733886719], [0.9617437720298767, -0.5399908423423767, 0.44321897625923157, -0.10143254697322845, 0.8159123659133911, 0.9133167862892151, -0.9543415307998657, 1.2661571502685547, -1.1507221460342407, -0.2389434427022934, -0.10646792501211166, 0.8729543685913086, 0.18692433834075928, 1.3029897212982178, -0.1194596141576767, -0.5488643646240234, -0.22701221704483032, 1.5297218561172485, 1.866032361984253, 0.9775896072387695, 0.8308786153793335, -0.5546851754188538, -0.02665850706398487, -1.9140727519989014, 1.1939815282821655, -0.5047440528869629, -0.9092130064964294, -1.4070717096328735, -0.3421127200126648, 0.7148935198783875, -0.4693017899990082, -0.01919550821185112, 0.29314103722572327, 0.6606400609016418, -0.3621094822883606, -0.34275007247924805, -0.7493197917938232, -1.0279912948608398, -1.2047340869903564, -0.742405354976654, -0.4198368787765503, 0.6768551468849182, -1.2034085988998413, 2.549041748046875, -0.8519192934036255, -1.942431092262268, -0.8289113640785217, -0.7875690460205078, 0.4396055340766907, 0.4127376079559326, -0.34389591217041016, -1.0627838373184204, -0.9943677186965942, 0.14419177174568176, -0.5548628568649292, -1.6468029022216797, 0.630339503288269, -1.3233238458633423, -0.481364369392395, -0.04499583691358566, 1.842856526374817, 0.010896400548517704, -0.698176920413971, -0.015888456255197525, -1.275637149810791, -0.2949739992618561, 0.2617429196834564, -0.03804728761315346, 0.9418514966964722, 1.215986728668213, 0.42105960845947266, 1.0503824949264526, 0.30239060521125793, 0.3397836983203888, -2.2485713958740234, 0.860370934009552, 1.3871384859085083, 0.1698998361825943, -1.371305227279663, -0.4605312943458557, 1.1630582809448242, 0.46313855051994324, 0.1032262071967125, -0.8060773611068726, 1.357282042503357, -0.3917511999607086, -1.383980631828308, -0.686425507068634, 0.10918811708688736, 1.5315934419631958, -0.5712907314300537, 0.5185850262641907, 0.22838422656059265, 0.08232578635215759, -1.4526934623718262, -0.06584279984235764, 0.2832910716533661, -2.315110921859741, 0.27913644909858704, 0.5532400608062744, 0.898141622543335, 0.24323609471321106, -0.08110495656728745, 1.0785036087036133, 1.0473651885986328, -1.122058629989624, 0.6602058410644531, 0.08720292896032333, -1.1570641994476318, -0.3854379653930664, 0.29199811816215515, -0.1704009622335434, -1.0390146970748901, 0.32801687717437744, -0.761450469493866, 0.8587574362754822, -0.6338890790939331, 0.6113470792770386, 0.2880413830280304, -0.4131253957748413, 0.3695698082447052, 1.7714937925338745, -2.008122444152832, -0.929659366607666, 0.26487410068511963, 0.6117285490036011, 0.3992651700973511, 0.3171616196632385, -1.0212043523788452, -0.5317583680152893, 1.4124040603637695, -2.1055727005004883, 0.34239673614501953, 1.2771830558776855, 0.3163793683052063, 0.003153106663376093, 1.034271240234375, 0.5093868970870972, 1.506038784980774, 0.12932232022285461, -1.439336895942688, -0.7036375403404236, -0.7630729079246521, 0.13174405694007874, 0.7403746247291565, -0.0026968412566930056, 0.5403016209602356, -0.36900225281715393, 0.34171614050865173, -0.9301636815071106, -0.6782853603363037, 0.9207878112792969, 0.06772872805595398, 0.366083562374115, 1.6436752080917358, -1.1206610202789307, -0.4089592695236206, 0.6737724542617798, 1.609885573387146, 2.278294563293457, -0.5516865253448486, 0.48804283142089844, 1.2108813524246216, -0.7999094724655151, 0.6083855628967285, 0.560875415802002, -0.24733980000019073, -0.7414599657058716, 0.872826337814331, -0.7116311192512512, 2.183650255203247, -0.7218573689460754, -0.8569917678833008, 3.198025941848755, 0.4682939052581787, 0.026778342202305794, -1.052099585533142, -0.6820113658905029, -0.18852801620960236, -0.6334004402160645, 0.30030199885368347, -1.0973750352859497, -1.1072221994400024, -0.6260297894477844, -0.22106117010116577, -1.6990464925765991, 0.8801581859588623, -0.8653631806373596, 0.18372642993927002, 1.6906970739364624, -1.3247913122177124, 0.944726824760437, 0.5430614352226257, -0.31828272342681885, 1.4597117900848389, -0.6923202872276306, -0.7936506867408752, -0.4988229274749756, 0.6554060578346252, 0.881766676902771, 2.8265249729156494, -0.8401063680648804, -0.4350424110889435, 0.6602345108985901, 0.981719970703125, -0.9381043314933777, -1.6074618101119995, 0.2269703447818756, 0.7607220411300659, 2.46050763130188, 0.45940661430358887, 0.9185733199119568, 0.3166183829307556, -0.666077196598053, -0.2558731138706207, -0.7739614248275757, -0.9786527752876282, 1.4457517862319946, -0.01733238250017166, -1.1882562637329102, 0.16463898122310638, 0.41842520236968994, -1.3800231218338013, -0.5661402344703674, 1.2901643514633179, 0.5155638456344604, 0.4399888515472412, -0.7431482076644897, 0.10574289411306381, 0.6735827326774597, 1.0377377271652222, -0.30915361642837524, -0.3550221920013428, 0.07280954718589783, -0.03830205276608467, 0.6378386616706848, 1.6538257598876953, -0.6960172057151794, 0.5448805689811707, -0.5405465960502625, -0.48769456148147583, 0.12842251360416412, 0.16697774827480316, -0.7272383570671082, -1.4473141431808472, 0.04718824103474617, 0.613044261932373, -0.14827637374401093, -0.6479765176773071, 0.2890571355819702, 0.4704608917236328, -1.2529927492141724, 0.34499120712280273, 1.7684394121170044, 2.0869367122650146, -1.6256043910980225, -1.1035879850387573, -1.135034441947937, 0.9062231779098511, 0.9230067133903503, -0.6674168109893799, 0.4488264322280884, -1.5565762519836426, -0.6896212100982666, 0.8833213448524475, -1.357999563217163, -1.3850948810577393, -1.1017204523086548, -0.14090496301651, -0.7921072840690613, 0.5219180583953857, 0.10612120479345322, -1.043367862701416, -0.8341348171234131, 1.146897554397583, -1.7567437887191772, -0.10866834968328476, -0.589126467704773, 1.5833699703216553, 1.0960251092910767, 1.9367225170135498, 0.44023850560188293, 1.115797996520996, -0.593478262424469, 0.6072346568107605, -1.467609167098999, 1.1591769456863403, 1.822977066040039, -0.6324353814125061, -0.8639771938323975, 1.4264165163040161, -0.10158714652061462, -0.5525993704795837, -0.5439680814743042, -0.18909315764904022, 0.6997995972633362, 1.4033952951431274, -0.3353070914745331, 1.0635758638381958, 0.36665499210357666], [0.8572831749916077, -0.6559584736824036, 0.6574180126190186, 0.3454350531101227, 0.44053393602371216, -0.4497106969356537, -0.6506494283676147, -0.7218025922775269, -0.7581762671470642, -0.7872970104217529, 1.6462230682373047, 0.4023037850856781, -0.5570181608200073, -1.0873140096664429, -0.3745386600494385, -0.7168425917625427, 0.3034891188144684, 0.21758228540420532, -1.1391099691390991, 0.04563641548156738, 1.0246264934539795, 1.148861289024353, -0.4470585882663727, 0.16947662830352783, -1.0530433654785156, -0.021671533584594727, -1.3985882997512817, -0.6194950938224792, 0.9190901517868042, -1.0490537881851196, 0.5910155773162842, -1.4252537488937378, -0.2828136086463928, -0.9711875319480896, 1.1923129558563232, 1.3256690502166748, 0.6987497210502625, 1.8273004293441772, -0.3729799687862396, 1.0764654874801636, -0.21685241162776947, -0.36731281876564026, -0.1510809063911438, -0.723924458026886, 0.8755714893341064, 2.8206627368927, 0.6955499649047852, -0.881176233291626, 1.0446356534957886, 1.0770546197891235, -0.21003851294517517, 1.7762972116470337, -1.1138637065887451, 0.0996960923075676, -1.4288028478622437, -0.6474870443344116, 0.2911107838153839, 1.2960196733474731, 1.0700117349624634, 1.1587013006210327, -0.9814130663871765, -0.04614318162202835, -1.1546211242675781, -1.0038065910339355, 0.49701711535453796, 1.3520199060440063, -0.16944216191768646, 0.7341502904891968, -1.8577858209609985, 1.503964900970459, -2.361104726791382, -1.4675655364990234, 1.4504225254058838, -0.09875766187906265, -0.6236652135848999, 0.4023064374923706, 1.4629822969436646, -0.7231503129005432, -1.535923719406128, -0.1753254383802414, -0.3566421866416931, -0.4818859398365021, 0.3462367057800293, 0.2486046552658081, -0.15388037264347076, -0.21247181296348572, 0.826888918876648, -1.1857874393463135, -0.2805873155593872, -0.2839098572731018, 0.35408803820610046, 1.464901328086853, -0.9083101153373718, -0.012807835824787617, -0.05452769994735718, 0.08688151836395264, -1.436198353767395, 0.44804254174232483, -0.7297919392585754, 0.2683408856391907, 0.5786229372024536, -1.0315898656845093, -0.5926612615585327, -1.9805164337158203, 0.11444631218910217, -0.3760582506656647, 0.6550511717796326, -0.6391322612762451, 0.4034721553325653, 1.1699074506759644, -0.4566456377506256, -0.7980949878692627, -0.08843538910150528, 0.5820102691650391, -0.8244190216064453, 2.02955961227417, 2.2284176349639893, 0.33335116505622864, -1.6406458616256714, 2.5812225341796875, 0.0708504244685173, -0.5457058548927307, -0.45402100682258606, -0.31230804324150085, -0.5473741888999939, 0.05763407424092293, -0.24177081882953644, 0.47820377349853516, 0.05106806010007858, 0.7606053352355957, -0.48915910720825195, -1.8562794923782349, -0.30224841833114624, 1.7623040676116943, 1.7141988277435303, 0.8688995838165283, -0.6585314273834229, -0.7156931161880493, -1.0316625833511353, 0.47052937746047974, -0.5880305767059326, 0.752660870552063, -0.6144006252288818, -0.022025248035788536, 0.95526123046875, 0.11517398804426193, -0.47923436760902405, 0.09935566037893295, -0.251817911863327, 1.6304877996444702, 1.688166856765747, -0.3259047269821167, -1.3193060159683228, 0.5649048089981079, 0.45376068353652954, -0.6086737513542175, 0.5048540234565735, -0.604285478591919, 0.777130663394928, -2.09870982170105, -0.898967444896698, -0.11217842996120453, -0.47363704442977905, 0.90329909324646, 1.5356332063674927, -1.729772686958313, 1.5995339155197144, 1.25406014919281, -1.7390271425247192, 0.8574345111846924, 0.6413251757621765, 0.579982340335846, -1.3290064334869385, 0.28032416105270386, -1.2980455160140991, 0.7523640990257263, 2.1690292358398438, -2.770594358444214, 0.3715158998966217, -0.31273743510246277, -0.0344892218708992, 0.557877242565155, 0.24737495183944702, 0.8489280343055725, 1.1945720911026, 0.26274725794792175, -0.2788327932357788, -1.7328346967697144, -0.661892831325531, 1.8070818185806274, 0.26264989376068115, -0.6950395703315735, -0.4011986255645752, 0.38912662863731384, -1.7190157175064087, -1.1549222469329834, -0.1314460188150406, -2.0858497619628906, 0.8922169804573059, 1.8917553424835205, -1.9167101383209229, -0.1576015204191208, 0.5723554491996765, 0.3504345118999481, 0.22031868994235992, 0.8954775333404541, 0.3882186710834503, -1.0821512937545776, 0.1691463589668274, 0.37372833490371704, 1.4060319662094116, 0.6118218302726746, -0.04627754166722298, 0.25121235847473145, -0.8367606401443481, -0.30589985847473145, 0.9343259930610657, -2.2040538787841797, -0.9074860215187073, 1.1332266330718994, 0.7638458609580994, -0.4198688566684723, -0.07022303342819214, 0.04972890019416809, 1.2411267757415771, 0.9344700574874878, 0.7438530325889587, -0.18245963752269745, 1.0304981470108032, 0.042649947106838226, 0.05666310712695122, 0.13395483791828156, 0.22475023567676544, -1.3637659549713135, -1.6198769807815552, 0.30481940507888794, 0.5877324342727661, 0.14783833920955658, 1.7322633266448975, -0.039988599717617035, 0.2544189393520355, 1.3765486478805542, -1.2929755449295044, -0.5724279880523682, 1.3164504766464233, 0.6082019209861755, 0.6035902500152588, -1.3892791271209717, 0.1866045594215393, 0.37706130743026733, 1.204317569732666, 0.6595227122306824, 0.16560718417167664, 0.2229466587305069, 0.12585805356502533, 2.291231155395508, 1.391431212425232, -2.049074411392212, -3.006091833114624, 0.6722577214241028, -1.3921151161193848, -1.4897783994674683, 0.3750797510147095, 0.5674303770065308, 0.15960600972175598, -1.0931814908981323, 0.3902668058872223, 0.4589601457118988, 0.4365972876548767, 2.000826120376587, 1.069167971611023, -0.870067298412323, 0.5349632501602173, -0.8217536211013794, -0.20928318798542023, -0.12447863817214966, -0.5922790169715881, -2.618542194366455, 0.9301362037658691, 1.5740795135498047, 0.6041120886802673, -1.2950481176376343, -0.290081650018692, -1.2513325214385986, -0.9980964064598083, -0.7933225035667419, 0.9763479232788086, 0.7100503444671631, -0.6830402612686157, -2.8447325229644775, -1.7057353258132935, 1.1527520418167114, 1.4671990871429443, -0.009250355884432793, 0.4724454879760742, 0.09724324941635132, 1.3784812688827515, -0.9925406575202942, 0.021418612450361252, -1.327437400817871], [-1.1818206310272217, 0.011449580080807209, 0.32812368869781494, -0.7412135004997253, 0.8123517036437988, 0.6316441893577576, 0.11439388990402222, -1.2811766862869263, -0.42723098397254944, -1.6008929014205933, -0.31693655252456665, 0.745657742023468, -1.8855262994766235, -0.45891717076301575, -0.10470010340213776, -0.3775497078895569, -0.9864417910575867, 0.8951833248138428, 0.8194006085395813, 1.4419939517974854, -0.29511451721191406, 0.9354382157325745, 1.0472183227539062, -1.336486577987671, -1.8896660804748535, -0.5063790082931519, 0.8622819185256958, 0.10372906923294067, -1.2426657676696777, -1.1709221601486206, -1.4619407653808594, 0.37243983149528503, -1.2288874387741089, -0.7991394996643066, -0.4990978538990021, 2.431645631790161, 0.8925133347511292, 0.05547995865345001, -0.19569022953510284, -1.6751772165298462, 0.5139122605323792, -1.1266334056854248, -0.9650925993919373, 0.04341517388820648, 0.14170387387275696, -0.09626638889312744, -0.9152624011039734, -0.39692622423171997, -0.8302322030067444, -1.1311415433883667, -1.2113183736801147, -0.30676180124282837, -1.3193198442459106, 0.5241836309432983, 1.3872238397598267, 0.8383278846740723, -1.6709920167922974, 0.07329945266246796, 2.4880030155181885, 1.726694941520691, -0.19279372692108154, -0.10162154585123062, 0.30115532875061035, 2.5793192386627197, -0.4651574194431305, -1.1634868383407593, 1.117996335029602, 0.6616749167442322, -0.04774110019207001, -0.11030800640583038, 0.7297179698944092, 0.07348889857530594, 0.3324601650238037, 0.6230078935623169, 0.36041396856307983, 0.908948540687561, 1.6196380853652954, 0.6300656199455261, 0.10287895053625107, -0.031048493459820747, 1.7344425916671753, 0.6226568818092346, 0.20975400507450104, -0.9276857376098633, -0.8466362357139587, 0.6805483102798462, -1.170253872871399, 0.0221832487732172, 0.40224939584732056, -0.3422102630138397, -1.0617645978927612, 0.8540657162666321, -0.6473857760429382, 0.6342178583145142, -1.3195289373397827, -0.1829567402601242, 0.7539294958114624, -1.1316792964935303, -2.8197834491729736, 2.7296104431152344, -0.22552001476287842, 0.07449807226657867, -2.393326997756958, -0.2331380397081375, -0.33036157488822937, 0.3941267430782318, -1.1825168132781982, -0.8367623090744019, 0.9153266549110413, -0.45459896326065063, -0.7406855821609497, -1.2063255310058594, 0.24453571438789368, 1.0377426147460938, 0.3331143856048584, -0.7127576470375061, -0.15911699831485748, 0.6281587481498718, -0.6042344570159912, -0.014555332250893116, 0.0922003984451294, 0.6219977140426636, -0.2228822112083435, 0.025349119678139687, 1.4121607542037964, 0.09353546053171158, -0.5351073145866394, 1.093355655670166, 0.13449908792972565, -1.0423283576965332, 1.1987513303756714, -1.0525630712509155, -0.6669541597366333, 2.377542018890381, -0.49196019768714905, -0.3482496440410614, -0.2501136064529419, -0.6253039240837097, -0.1998506486415863, -0.9296882748603821, -0.14668260514736176, -1.0890144109725952, 1.3400954008102417, 0.42747074365615845, -0.5284488797187805, -0.34395474195480347, -0.4751952290534973, -1.4068347215652466, -0.32524141669273376, 1.324569821357727, 1.4713494777679443, 1.8367668390274048, 1.697335958480835, -1.6617133617401123, -0.7559827566146851, 1.0125932693481445, -1.6869748830795288, 0.4945865869522095, -0.008485455997288227, -0.02513543888926506, -0.15694768726825714, 0.09241662919521332, 1.165462851524353, 0.04346223548054695, -0.991326093673706, -1.1818584203720093, -0.17845185101032257, -0.6044674515724182, 1.7037432193756104, -1.0099036693572998, -0.021889565512537956, 0.9188600182533264, 0.958013653755188, -0.32489344477653503, 1.4963876008987427, -1.7506463527679443, 0.85202556848526, 1.567986249923706, 0.5543521046638489, -0.6879287958145142, -0.9975699782371521, -1.7449954748153687, 0.12081616371870041, -0.4847066402435303, 1.193310260772705, -1.9796185493469238, 0.3981359601020813, -1.3815512657165527, 0.13751766085624695, -0.07218030840158463, -0.0318644717335701, -0.5002408027648926, -1.1082459688186646, 0.189815953373909, 0.6446916460990906, 0.35780173540115356, -1.0297998189926147, 1.3223698139190674, 1.3097864389419556, 0.7284687161445618, 0.6622193455696106, 1.0534093379974365, -1.3056809902191162, 0.5411460995674133, -1.3739557266235352, 1.761596918106079, -0.9917621612548828, -0.5274884700775146, 1.485291838645935, -1.1100468635559082, 1.7851877212524414, 1.82561457157135, 0.022191565483808517, 0.7861477136611938, -0.7060453295707703, 0.5556869506835938, 0.1169343963265419, -1.485704779624939, 1.8452973365783691, -2.742870807647705, 1.4159643650054932, 0.24052174389362335, -0.7857430577278137, 0.2266414612531662, 1.8473128080368042, -0.36229223012924194, 2.3185999393463135, 1.4741026163101196, 0.12389962375164032, -0.6301277279853821, 0.49840909242630005, -0.5061132907867432, -0.6091805100440979, 0.03461606800556183, 0.4883330464363098, 1.6055718660354614, -0.6578993201255798, 1.67811918258667, -2.1681830883026123, -0.473080575466156, -0.14736975729465485, 2.1177260875701904, 1.2392112016677856, -0.8106951713562012, -0.9989206194877625, -1.3593549728393555, 0.1571340709924698, -1.2573378086090088, -0.5136011242866516, 0.6921706199645996, -0.24649401009082794, 0.6898425221443176, 0.5174591541290283, -0.07583808898925781, 1.4634608030319214, -0.9741299748420715, -0.8207712769508362, -0.12407578527927399, 0.011813398450613022, -0.8537511229515076, -0.10977581143379211, 0.2795874774456024, 0.9756649136543274, 1.830394983291626, 1.6318793296813965, 1.1690828800201416, 0.48636823892593384, 0.3265874981880188, 0.467423677444458, -0.9765188694000244, -1.183988332748413, 2.7792816162109375, -0.03902430087327957, -1.2430963516235352, 0.3278595805168152, 0.01668645441532135, 1.4774386882781982, -1.9008454084396362, 0.9879769682884216, 1.395056128501892, 0.036923158913850784, -0.6654369235038757, -1.0851308107376099, 0.40567827224731445, -0.2031460702419281, 0.40461960434913635, 0.9877015352249146, 0.1583748310804367, -0.02041098289191723, -0.07463883608579636, 1.1206971406936646, -0.18840248882770538, -1.8164410591125488, 0.6813207864761353, 0.1580808311700821, 0.8569592833518982, 0.8517141342163086, -1.6881825923919678, 0.7318555116653442, -0.07080788910388947], [0.7664182186126709, -0.251231849193573, 0.29164376854896545, 1.5772284269332886, 0.3533765375614166, -0.15780217945575714, -1.9201840162277222, 0.3288024365901947, 0.7634708881378174, 0.033151209354400635, -0.020183734595775604, 1.4155884981155396, -0.27132582664489746, 0.27728161215782166, -2.138110399246216, 0.3900420367717743, -0.7800966501235962, 0.5304186940193176, -1.9485557079315186, 0.6751599907875061, -0.3261452913284302, 0.6612094044685364, 1.4395051002502441, 0.14399836957454681, -1.4093029499053955, 0.851722776889801, 0.1644946038722992, -0.6437919735908508, -0.5129702091217041, -0.9792539477348328, -0.4895576238632202, 1.4179741144180298, 0.832617998123169, -0.4244549572467804, 0.4774271845817566, -1.4464808702468872, -0.23833726346492767, -0.4243054986000061, 0.3167215585708618, -0.00683518685400486, -0.5220786333084106, 0.22745440900325775, 0.531219482421875, -0.3787909150123596, 0.7546489834785461, -0.6724331378936768, 0.8558847308158875, -0.5915343761444092, -1.4027937650680542, -1.8430289030075073, -0.8489270806312561, 0.9506696462631226, -0.7191263437271118, -0.2956593334674835, -0.1498001366853714, -0.39881065487861633, -1.0849090814590454, 0.7005668878555298, 0.08876089751720428, -0.27370738983154297, -2.183741807937622, -0.8236295580863953, 0.09231946617364883, 0.12757711112499237, -0.1413159966468811, -0.22085349261760712, -0.5085949301719666, -0.8496723175048828, 1.574284553527832, -1.947529911994934, -0.03432480990886688, -0.4935387372970581, -0.6951536536216736, -0.8073311448097229, 0.21412630379199982, 1.4183868169784546, 0.11501629650592804, 0.3989303410053253, 1.2259458303451538, 0.03584519028663635, -1.1289573907852173, -1.1607258319854736, 1.0777240991592407, 0.4434286057949066, 0.9068104028701782, -0.07190512120723724, -0.37476304173469543, 0.7194063067436218, -1.82010817527771, 1.2317781448364258, 0.6082490682601929, -1.529007077217102, -0.14817853271961212, -0.9391745328903198, 0.8475133776664734, 0.8923631310462952, -0.16317811608314514, -1.4116878509521484, 0.1676710695028305, -0.30867359042167664, 0.4519721567630768, 0.47811952233314514, -0.2049841433763504, 0.15213480591773987, -0.559849202632904, 0.1363295316696167, -1.5033971071243286, -1.1131435632705688, -1.0658682584762573, 1.4547597169876099, -1.3806676864624023, 1.2604830265045166, 0.8331480026245117, 0.11169667541980743, -0.16040945053100586, -0.8787010908126831, -1.1758214235305786, -1.2112845182418823, -0.6178715825080872, 1.0757001638412476, 0.12006945163011551, -0.042552098631858826, 1.205317497253418, 0.08394407480955124, 0.21042393147945404, 1.7327837944030762, 0.6771916747093201, 0.6389291286468506, -0.3409329354763031, -0.2550632059574127, -1.8181768655776978, -0.319768488407135, -0.4269081950187683, 0.47011327743530273, 0.552623987197876, -0.11272925138473511, -0.25207826495170593, -1.101259708404541, -0.7305020689964294, -0.1329517662525177, -0.4221227169036865, -0.3238846957683563, 0.25010308623313904, -0.7240136861801147, 0.022254958748817444, -0.7208473086357117, 0.20938488841056824, -2.5633022785186768, -1.6031843423843384, 0.354468435049057, -0.9543323516845703, 1.2744247913360596, -0.7350205183029175, -0.6572416424751282, 0.4299565553665161, 1.350609302520752, -0.8641127943992615, 0.16057196259498596, 0.945915937423706, -1.3657220602035522, 0.9961548447608948, -1.1559288501739502, -1.9815354347229004, -0.5034226775169373, 0.5132637023925781, 0.7760287523269653, 0.43980851769447327, -0.4656931757926941, -0.228037491440773, 0.919061005115509, 1.1653947830200195, 0.7100242376327515, 0.9219824075698853, -0.537325382232666, -0.6384836435317993, 0.3189823031425476, -0.7197686433792114, 0.8126120567321777, -0.03068569116294384, -0.46675029397010803, 0.8701092004776001, 0.8947856426239014, -0.037399038672447205, -1.4561110734939575, -0.6145642399787903, 0.7843495607376099, 1.0237582921981812, -0.19388383626937866, -1.7691651582717896, -1.5234614610671997, 0.6903042793273926, -1.1142886877059937, 0.21942958235740662, -0.2803892195224762, -1.6299628019332886, 0.2312234491109848, 0.7828354835510254, 1.0978678464889526, -1.3135504722595215, 0.7592916488647461, 0.04608280584216118, -0.1407930850982666, 2.5696842670440674, 0.15301954746246338, -0.4022597074508667, 0.24187518656253815, -0.11908610165119171, -0.8622758388519287, 0.36477071046829224, -0.4773448705673218, -0.1733698695898056, 1.2522587776184082, -0.7472184300422668, -0.9344321489334106, -0.7061668038368225, 0.37179142236709595, -0.08789893984794617, 2.5065057277679443, -1.2829838991165161, -1.1260311603546143, 0.5723977088928223, -0.40744560956954956, 0.0960460752248764, 0.9443312883377075, -0.18260812759399414, -0.5836196541786194, 0.36435452103614807, -1.4127755165100098, 0.00833695475012064, 1.4051932096481323, -0.06356334686279297, 0.7357204556465149, -0.3089073896408081, -0.3751683235168457, 0.2595784366130829, 0.4172423779964447, 2.6875782012939453, -1.6393476724624634, 0.27613431215286255, -0.08326342701911926, -0.7802510261535645, 0.5182955265045166, -0.3259362280368805, -2.336519718170166, -0.7146531343460083, -0.7592102885246277, 0.7531347274780273, 0.8063024282455444, 0.2846541106700897, -0.6152858734130859, -0.7161061763763428, -0.9872059226036072, 1.5854613780975342, 0.33950114250183105, -1.2722172737121582, 0.5989426970481873, -0.37133094668388367, 1.3328558206558228, -0.09112781286239624, -1.652675986289978, -1.666591763496399, 2.0536746978759766, 0.4429899752140045, 1.4468629360198975, 1.3097245693206787, -1.1113865375518799, -0.3380643427371979, -2.3342998027801514, -1.25956130027771, 0.7209253907203674, 0.09674974530935287, -0.46534717082977295, 1.0465794801712036, -0.05960476025938988, 0.2507961094379425, -0.8232457041740417, -0.7201312184333801, -0.22459423542022705, 0.5716686844825745, -0.7829197645187378, -0.7521633505821228, -0.10655990988016129, -0.7080931663513184, 1.1300283670425415, 0.598391056060791, 0.7454164028167725, -0.27096959948539734, 0.40816375613212585, 0.28557127714157104, 1.190825343132019, -0.2708323001861572, -1.1747063398361206, 1.073675274848938, -0.429909348487854, 1.4506864547729492, -0.4117996394634247, -0.3995339274406433, 0.4202878475189209, 0.7205033302307129, -0.6053645610809326], [0.05230000615119934, -0.3914337456226349, 0.4930489659309387, 1.0541929006576538, 2.178027629852295, -0.16881310939788818, -1.3168143033981323, 0.20390541851520538, 0.6554695963859558, -0.7842931747436523, 0.48180073499679565, 0.5055417418479919, 1.7873555421829224, -0.7762755751609802, 0.5516970753669739, -0.6353408098220825, 0.024909459054470062, -0.311728835105896, 0.42350220680236816, -0.38930848240852356, 0.1404595971107483, 0.18683210015296936, -2.063765048980713, -0.8655643463134766, -0.2763844430446625, -0.801207959651947, 0.41434669494628906, 0.36596763134002686, -0.9970620274543762, 0.3023213744163513, -0.8116372227668762, -0.8202614188194275, 0.4252905249595642, -0.5330556631088257, 0.05665499344468117, 0.5590884685516357, -0.16252613067626953, 1.3469524383544922, 0.14920222759246826, 0.44633936882019043, 0.0107875419780612, -2.2447946071624756, 0.49567463994026184, -1.2351936101913452, 0.5609138011932373, 0.6980574727058411, -0.014198288321495056, 0.03925245255231857, -0.001832842011936009, 0.18940410017967224, 0.6148689985275269, -0.1456165909767151, -1.2391057014465332, 0.6365894675254822, -0.9301849007606506, -0.16207659244537354, -0.5359769463539124, 0.7025684714317322, -1.5785412788391113, -0.7238991260528564, -0.5318224430084229, 2.2306315898895264, -0.8373919129371643, 0.14184865355491638, -0.4035890996456146, 0.31397461891174316, -1.3957624435424805, 0.4301796555519104, 0.8934370875358582, 0.956125795841217, 0.5510298013687134, -1.0183194875717163, 0.20898841321468353, -1.65169358253479, 0.6107447147369385, 0.5068519711494446, -0.5190248489379883, 0.22582727670669556, -2.1609609127044678, -0.903509259223938, -0.31203317642211914, -0.5850777626037598, 2.781912088394165, -0.04456567391753197, 1.2973984479904175, 0.2505144774913788, 0.7212512493133545, 1.7478272914886475, -1.0990724563598633, -0.7621431350708008, -0.3475914001464844, 2.5547547340393066, 0.14842632412910461, 0.49155664443969727, -0.5160494446754456, 1.3967936038970947, -0.1131637915968895, -0.6070042252540588, -0.6600976586341858, 0.17203274369239807, 0.3571757674217224, -0.8208174705505371, -1.6057167053222656, 0.003961234353482723, -0.7754502892494202, -0.8514733910560608, 0.4003298878669739, -1.7007008790969849, -1.430392861366272, 2.11039400100708, 0.21408025920391083, -0.13053560256958008, 1.1326104402542114, -1.2124450206756592, -0.47914162278175354, -0.830315113067627, -0.13664813339710236, -1.6192295551300049, 0.8317649364471436, 1.2096312046051025, 0.3584446310997009, 0.6536890864372253, -0.7740839123725891, 1.4584218263626099, 1.4688664674758911, -0.9816280603408813, -1.573866367340088, 1.3420625925064087, 0.4532167315483093, -0.7934620380401611, 0.0012939650332555175, -0.18600156903266907, -0.5735836029052734, -0.8899922966957092, -0.10741035640239716, 0.865656852722168, -1.2066980600357056, -0.5974741578102112, -0.4082176983356476, -1.394697666168213, 1.4752368927001953, -1.4695311784744263, 0.8013975620269775, 1.4192583560943604, -0.5593843460083008, 2.770702362060547, 0.9274042844772339, -0.3938370943069458, -0.4972088038921356, -0.8206879496574402, -1.9468926191329956, -0.7040151953697205, 0.7995085716247559, -0.4943488538265228, -0.49133366346359253, 1.4298696517944336, 0.017833109945058823, -0.37597227096557617, -1.5537488460540771, -1.0031872987747192, -0.7560527324676514, 0.8055587410926819, -1.1101160049438477, -0.01595940627157688, -0.7398870587348938, -0.5907246470451355, -0.2584112584590912, 0.532336413860321, -1.2408888339996338, -1.2410452365875244, 0.38285109400749207, 1.0190805196762085, 0.742196261882782, -0.45878326892852783, 0.36334237456321716, 0.21114513278007507, 0.6700230836868286, -0.8918599486351013, 0.7719100713729858, 0.2030429244041443, 1.0363643169403076, 1.823194146156311, -1.4118996858596802, -0.25453105568885803, -0.16859902441501617, -0.1876964569091797, -0.1108085885643959, -0.6993270516395569, 1.3470278978347778, -1.1612980365753174, -1.815045952796936, -0.9105157852172852, 0.9845193028450012, 1.232033610343933, -1.1658555269241333, 0.25317901372909546, -1.5025558471679688, -1.017387866973877, -1.2556757926940918, -0.39423757791519165, 2.1204302310943604, 0.08496297895908356, -0.0879083052277565, 0.9917702078819275, -0.928158164024353, 0.025205522775650024, 0.3678736984729767, 0.6714056134223938, 0.2682712972164154, -1.152260422706604, 1.045614242553711, 0.6103375554084778, -1.3655030727386475, -0.36876240372657776, -1.792632818222046, 1.6255592107772827, 0.8215446472167969, -2.867065668106079, -1.4177892208099365, -0.7748941779136658, 0.4508512020111084, 0.7273348569869995, 1.9633371829986572, -0.9294049143791199, 1.5735934972763062, -1.0284056663513184, -1.2631415128707886, -0.12491902709007263, -1.335942268371582, 0.0876883864402771, -1.074361801147461, -0.8477408289909363, -0.6049839854240417, -0.20042355358600616, 0.1348361372947693, 1.5146348476409912, -0.5283530354499817, 1.7188535928726196, -0.4372533857822418, -0.8136842846870422, 0.5450616478919983, 1.9778589010238647, 1.1842448711395264, 0.10139457136392593, 0.5927857160568237, -0.5840700268745422, -0.7547258734703064, -0.5610859990119934, 0.5084648728370667, 1.491807222366333, 1.5888296365737915, 1.3040231466293335, 0.2833106517791748, -0.9703692197799683, 0.48911407589912415, 0.42314520478248596, -0.8970329761505127, 1.766476035118103, -0.6020069122314453, 0.3994339406490326, 0.5842064023017883, 0.11123247444629669, 0.306559294462204, -1.025434970855713, 1.5801148414611816, 1.2072741985321045, -0.9682087898254395, -0.14091090857982635, -1.6851844787597656, -0.6651679277420044, -0.8463342785835266, -0.6872117519378662, -0.7718290090560913, -0.2918127775192261, -0.1760013997554779, -0.8042129874229431, 0.0314902737736702, -0.41043511033058167, 1.7185324430465698, 0.7406255602836609, -0.2636355459690094, -0.7060844302177429, 2.242619514465332, 1.1975983381271362, 0.004413760267198086, -0.7447933554649353, -1.13039231300354, 0.5360314846038818, -0.052909232676029205, 0.8513217568397522, 0.9746655225753784, 0.31573382019996643, 1.1384038925170898, -1.046656608581543, -0.36992523074150085, -1.8676421642303467, -1.0530030727386475, 0.9894202351570129, 2.078007698059082, 0.7069722414016724], [0.828596830368042, -0.3231957256793976, -0.39927053451538086, -1.978337049484253, -2.0997202396392822, -0.12547695636749268, 1.6838164329528809, -0.5345077514648438, -0.9854959845542908, 0.741774320602417, -1.0901581048965454, -0.09655005484819412, -1.2122397422790527, 0.4091418385505676, 0.8460150361061096, 1.0966330766677856, 1.104789137840271, 0.15779711306095123, -0.10766319185495377, -0.8429470062255859, -0.2322472333908081, -1.1154569387435913, -0.24256357550621033, -0.9540651440620422, -0.9033239483833313, -0.34270623326301575, 1.014464020729065, 0.34176772832870483, 0.18124353885650635, -0.8364008069038391, 0.08071663230657578, 0.9330597519874573, 0.37698009610176086, 0.35696837306022644, -0.807794451713562, 0.8671205639839172, -0.41911780834198, -0.27572545409202576, -0.10090743750333786, 1.018174648284912, 0.6890292167663574, -0.08335611969232559, -0.17945025861263275, -1.1668667793273926, -0.2646713852882385, 0.16504420340061188, 0.2304590791463852, 0.372699111700058, -0.6528737545013428, 0.7372355461120605, 2.483616590499878, 0.6465498208999634, 0.5785656571388245, 2.153228759765625, -1.229099988937378, 0.4442795217037201, 0.5653525590896606, -0.30940690636634827, -1.4848524332046509, -0.42034775018692017, 1.2452090978622437, 0.031177811324596405, -0.6337195038795471, 1.18773353099823, -1.2973576784133911, 0.9606345295906067, -0.7261508107185364, 0.04640712961554527, 0.10266660898923874, 0.8049744963645935, 0.01616724580526352, -1.5494639873504639, -0.5018795728683472, 0.5500550270080566, -0.09916984289884567, -0.10066933929920197, -0.5724895000457764, 1.096486210823059, -1.00511634349823, 0.4808443486690521, -1.2761667966842651, 0.5770536065101624, 0.16032801568508148, -0.5689356327056885, -1.1774306297302246, 0.29297810792922974, -0.12479400634765625, -0.8625969290733337, -1.5205795764923096, -0.2837377190589905, -0.8791490197181702, 0.7219401597976685, -0.19533690810203552, -0.6500862240791321, 2.285546064376831, 1.5987906455993652, -0.782981276512146, -0.20263801515102386, -0.3565692603588104, 0.5031185150146484, -0.45911556482315063, 1.3443236351013184, -1.6862579584121704, -0.13907794654369354, -0.5321827530860901, 0.6377497315406799, 0.32754555344581604, 1.383370041847229, 1.0623886585235596, -0.6731513738632202, -0.6456760168075562, -0.1010681614279747, 2.0451977252960205, 2.63608980178833, -1.1228231191635132, -0.2462816685438156, 0.06691620498895645, 0.05519960820674896, 1.907137393951416, 0.3987429141998291, 1.3404799699783325, -1.8814340829849243, 0.13357044756412506, 0.5509048700332642, 1.1203968524932861, -0.644757091999054, -0.8774623870849609, -1.0311700105667114, -0.031944841146469116, 1.715220332145691, -1.061942219734192, -1.54768967628479, 0.9335762858390808, 1.2623975276947021, -1.4480524063110352, 1.7169063091278076, 1.5624977350234985, -0.6797962784767151, -1.6232056617736816, 1.0640664100646973, -1.130548357963562, -0.1671810746192932, -2.222289800643921, -1.6119152307510376, -0.8863876461982727, -1.0827780961990356, 0.08147832751274109, -1.0035924911499023, -0.3103127181529999, 0.3425109088420868, -0.544265627861023, 0.6793134212493896, -0.6500798463821411, 0.7290208339691162, -0.22845010459423065, -1.0748318433761597, -0.657063901424408, -0.3202976584434509, -0.4574209749698639, -1.4642210006713867, -1.8092188835144043, -1.2743213176727295, -1.5203616619110107, 1.6663156747817993, 0.12295744568109512, -0.8955554962158203, -0.06781898438930511, -0.014797287061810493, 1.077433705329895, -0.2597983479499817, -0.0731678232550621, 0.24899528920650482, 0.5587425827980042, -0.23978276550769806, 0.4485844373703003, 0.08878176659345627, 0.10700913518667221, -0.36959636211395264, -1.805114984512329, 0.24289357662200928, -0.07993460446596146, 0.3708268702030182, -0.12228525429964066, -0.8600167632102966, 0.08282896131277084, 2.6722991466522217, 1.1811350584030151, -0.012534376233816147, 1.5408097505569458, -0.09863092005252838, -0.9062899947166443, 0.44443729519844055, 0.4758986830711365, 1.2966792583465576, 0.8910225033760071, -2.2751190662384033, 0.9406191110610962, -1.8511914014816284, -0.18805505335330963, 0.10614070296287537, -1.8081490993499756, 1.1394819021224976, 1.275349497795105, -0.43883779644966125, -0.7973580360412598, -1.4446593523025513, 0.12989749014377594, 0.49968090653419495, 0.4610018730163574, -0.4226848781108856, 0.547954797744751, -0.1774459034204483, -0.5216381549835205, -0.13839370012283325, 0.15844638645648956, 1.367642879486084, -0.5388674139976501, -0.40730321407318115, 0.02786196582019329, -1.0038025379180908, -1.7079657316207886, 1.3147413730621338, -1.7005436420440674, 0.8898913264274597, -0.1944943368434906, -0.6799176931381226, 0.07188718765974045, -0.3194613754749298, 0.44703078269958496, 0.005645299330353737, 0.026813330128788948, 0.4238949716091156, -0.684911847114563, -1.629880428314209, 0.46537384390830994, -0.6483979821205139, -2.58510684967041, 1.6471366882324219, -0.5266736149787903, 2.841364860534668, -0.3894135653972626, 0.10645615309476852, -0.6352457404136658, -0.15411785244941711, 0.4152540862560272, 1.7269717454910278, 1.2403695583343506, -0.19830112159252167, 0.9830169677734375, -0.4425041377544403, 0.8779338002204895, 0.134100079536438, 0.2254188358783722, -1.9080520868301392, 0.25514549016952515, 1.100714921951294, 1.173444390296936, -1.2761374711990356, 0.7820279598236084, 0.3072216510772705, -0.48372575640678406, 1.668199896812439, -0.3308964967727661, 0.4218674302101135, 0.5389378070831299, 0.2560392916202545, 1.8089162111282349, 0.46499744057655334, -0.7005211114883423, -0.22878910601139069, 0.6081510186195374, -0.47324708104133606, 0.7141885161399841, 0.574444591999054, -1.1301805973052979, -0.7715244889259338, -0.6563304662704468, -1.388644814491272, -2.115471839904785, -0.34548696875572205, -0.5192767977714539, 2.450984239578247, 1.3331199884414673, 0.3065887689590454, -0.20557284355163574, -1.506058692932129, -0.7901032567024231, 0.6187490224838257, 0.8151662349700928, 1.350226640701294, -1.034289002418518, -0.7919755578041077, -0.21798351407051086, -1.5745947360992432, 0.3210970461368561, 0.9239341616630554, 0.718914270401001, -0.623127818107605, -1.6663682460784912, 1.9058359861373901], [0.6321626305580139, -0.03683320805430412, 0.8708083033561707, -0.46594667434692383, 0.41683754324913025, -0.2175043374300003, 0.10180205851793289, -0.2831418812274933, 0.7357286214828491, 2.7597122192382812, -1.1438748836517334, 1.7833075523376465, -1.1943552494049072, -0.09119892120361328, 0.8064537048339844, -0.36064866185188293, -0.8949722051620483, 1.7894750833511353, -0.1053469106554985, -1.017777919769287, 1.1635221242904663, 0.820631742477417, -1.0984675884246826, -1.0290844440460205, 2.3779351711273193, -0.1280396431684494, -0.21929150819778442, 0.603626549243927, -2.4204063415527344, 0.8031936883926392, 0.984000563621521, 0.053342822939157486, -0.1600344479084015, -0.34190529584884644, 1.2886450290679932, 0.4464598000049591, 0.6504173278808594, 1.6305526494979858, 0.22030963003635406, 0.8001032471656799, -1.4532809257507324, -0.4090340733528137, 0.5419178605079651, 1.2943493127822876, 0.26200833916664124, 0.5025293827056885, -0.47107332944869995, -1.2132542133331299, 1.0371155738830566, 0.2465105950832367, -0.30887502431869507, -0.18149982392787933, 1.1001670360565186, -0.024146482348442078, -1.2473024129867554, -0.011520754545927048, -0.6107596755027771, -0.14842893183231354, -0.7506569027900696, 1.500960111618042, 0.3646172285079956, -0.44765791296958923, -0.8864531517028809, 2.4990506172180176, 2.1291279792785645, -0.4092949628829956, -1.6835013628005981, -0.7412343621253967, 0.8587244749069214, 1.574835181236267, 0.9388485550880432, 0.049700330942869186, -1.3881731033325195, -0.28364449739456177, 0.557529628276825, -0.4549542963504791, -1.3652678728103638, 1.2193599939346313, -0.8545605540275574, -0.6661247611045837, 1.4189403057098389, -1.6602246761322021, 0.6619109511375427, 0.79514479637146, 1.8090156316757202, -0.8064892292022705, 0.927844762802124, 0.22664161026477814, 0.023288704454898834, -0.40279003977775574, -3.0743837356567383, -0.14121422171592712, -0.9645417332649231, -0.12510161101818085, 0.04434806853532791, 0.214324951171875, -0.35569682717323303, 1.3810174465179443, -1.1556025743484497, -1.002631425857544, 2.517422914505005, -0.6456519365310669, 0.6445565223693848, -1.165297031402588, 1.1028451919555664, 1.4323900938034058, 1.0015400648117065, -1.0403311252593994, 0.5052669048309326, -0.9989360570907593, 0.5381615161895752, 1.3748129606246948, 1.3883618116378784, 2.127852201461792, -0.6419365406036377, 1.6224098205566406, 0.9905492067337036, 1.4769974946975708, -1.6852465867996216, 1.2081222534179688, 1.0663318634033203, -0.5751829147338867, 0.40510106086730957, 0.9463664293289185, 1.0481301546096802, -0.02725759893655777, -0.6663222312927246, 0.881662130355835, -0.22131554782390594, -0.649806797504425, -0.7575890421867371, 1.2243192195892334, -1.5080742835998535, -1.15740966796875, 1.6546218395233154, -1.3000348806381226, 1.0425019264221191, -0.05495409294962883, -1.022833228111267, -2.776135206222534, -0.5572950839996338, 1.6600548028945923, -0.397920161485672, -0.41745245456695557, 0.076532281935215, 0.5150566101074219, 0.24120962619781494, -0.04704398661851883, 1.0141714811325073, -0.957248866558075, 1.245727300643921, -1.1961264610290527, -2.9782795906066895, 2.2485427856445312, -0.3432597815990448, -1.2877192497253418, -0.19475343823432922, -0.3238489031791687, -1.0984907150268555, -0.9359292387962341, -0.07591905444860458, -0.2041628509759903, -1.6611475944519043, -0.5495073199272156, 0.48316171765327454, -0.09363006055355072, 0.2771790027618408, 0.759328305721283, -0.4985272288322449, 0.9151521921157837, 1.6610888242721558, -1.4780378341674805, 0.28658366203308105, -0.13155584037303925, -0.3796645998954773, -0.18986600637435913, -0.2766271233558655, 1.0253756046295166, -1.1692464351654053, 1.964629054069519, -1.7959986925125122, -2.336414098739624, -0.3344304859638214, 0.2346215695142746, 0.6630730032920837, 0.2161998450756073, 0.41935598850250244, 3.396519422531128, -1.0576729774475098, 0.15942144393920898, -1.4179404973983765, -0.5398172736167908, -1.1516550779342651, 1.760601282119751, 0.7561303973197937, 0.20376628637313843, 0.8240718841552734, 0.227247416973114, 0.7972134351730347, 0.5469719767570496, -0.06184477359056473, -0.16077657043933868, -1.281322956085205, -0.6489749550819397, -0.8876652121543884, -1.535801887512207, 0.512322187423706, 0.15153485536575317, -0.046928323805332184, -0.12243755906820297, 0.0874902606010437, 0.085906483232975, -2.3695735931396484, -0.5500913858413696, 0.23917709290981293, 2.9643917083740234, 1.3460379838943481, 0.26905137300491333, -0.4792494773864746, 0.4166214168071747, -2.3538613319396973, -0.05242075398564339, 2.08309268951416, 0.9851500391960144, -0.6057724952697754, -0.5604071617126465, 0.3954073190689087, 0.12371885776519775, -0.39499324560165405, 0.17571184039115906, -0.11571656167507172, 0.8299739360809326, 0.3255804181098938, -0.6887028813362122, 0.9260048270225525, -0.767686665058136, -1.0193638801574707, 0.24778974056243896, 0.33820033073425293, 1.4655660390853882, 0.25988414883613586, 1.119736909866333, 0.3217795789241791, -1.0809428691864014, 0.47436097264289856, 0.08293413370847702, 1.2516412734985352, 0.8734104037284851, 0.9509223103523254, 0.02869797684252262, -1.5672301054000854, 0.8376833200454712, -0.1411396712064743, 0.16965699195861816, -0.03630055487155914, 0.4035024642944336, 0.4200253486633301, 1.1212294101715088, 0.0969751849770546, -0.5623955726623535, 0.35903200507164, 0.7733158469200134, -0.5995681881904602, -0.341557115316391, -1.0086356401443481, 1.7512168884277344, 0.7782683372497559, -0.3625352680683136, -0.37740471959114075, 0.33650046586990356, 0.5373134613037109, -0.10942310839891434, -0.9753347635269165, -0.8168108463287354, -0.2678786516189575, 1.0197031497955322, -0.927191436290741, 1.2186968326568604, -1.6981269121170044, -0.22690467536449432, -1.1918015480041504, 0.3467901051044464, -1.1556812524795532, 0.7618391513824463, 1.536599040031433, -0.30252939462661743, -2.0661940574645996, 1.0853906869888306, 0.5788764357566833, 0.0246924739331007, 1.0683408975601196, -1.364757776260376, 0.07587166875600815, -1.9346808195114136, 2.2960240840911865, 2.1002721786499023, 1.0158298015594482, 0.43217018246650696, 0.12171714007854462, -0.4366198480129242], [0.13853202760219574, 1.6453624963760376, -1.4800593852996826, 0.14394164085388184, -1.7098747491836548, 0.17334432899951935, -0.2854655683040619, 2.1098225116729736, -0.41423311829566956, 0.9788335561752319, -0.6474937796592712, -1.5092536211013794, -0.3862805664539337, 2.7910678386688232, 0.2182701975107193, 1.434701919555664, 1.3237295150756836, 0.40032103657722473, -0.5907508134841919, -2.3940370082855225, -1.1187491416931152, 1.1243094205856323, -0.2910248935222626, 0.6375479102134705, 0.8625105619430542, 1.0055949687957764, -0.4749302864074707, 1.6399160623550415, -0.25561583042144775, 0.7452903985977173, -0.518234372138977, 1.8641523122787476, 0.5304409861564636, -1.1390511989593506, -0.18789614737033844, 1.045218825340271, 0.5110050439834595, -0.02222074195742607, 0.7599869966506958, 0.7086610794067383, -0.15984080731868744, 2.270583391189575, 0.8598524928092957, 0.0018479535356163979, 0.35030049085617065, -0.02699470706284046, 0.8901179432868958, 0.7511482238769531, -0.7952020168304443, 2.883845329284668, -0.35548919439315796, -0.33032524585723877, -1.4792934656143188, -0.9326227903366089, -1.4893407821655273, -0.987596333026886, -0.8503815531730652, -0.10253353416919708, 1.2347619533538818, 0.741399347782135, -1.845857858657837, -0.704106867313385, 0.5939220190048218, -0.5901458263397217, 0.50666743516922, 0.3379793167114258, -0.9222466349601746, 0.6837478876113892, 0.12474599480628967, -0.7020168900489807, 1.5331027507781982, 0.5045005679130554, 0.31381991505622864, 1.2517508268356323, 0.9658911228179932, 0.9855337142944336, 0.6997858881950378, 1.447371006011963, 0.08291208744049072, 1.1165094375610352, 1.1574125289916992, 1.4812977313995361, 1.6048624515533447, 0.9860141277313232, 0.1947924792766571, 0.05814357101917267, 0.29861366748809814, 0.5934551954269409, 2.1331522464752197, -0.3918646275997162, 0.9278489351272583, -0.4675600528717041, -0.5195876359939575, 0.7092090845108032, 1.18002188205719, -1.0213137865066528, 0.6784104108810425, 0.88676518201828, 0.566390335559845, -0.497916042804718, 0.629046618938446, 0.13395465910434723, 0.5635290741920471, 0.7283423542976379, -2.20043683052063, 0.8092454671859741, -0.6868849992752075, 0.2715339660644531, 0.31767168641090393, 0.9461684226989746, 0.1567608267068863, 0.723097026348114, -1.2007241249084473, 1.744482398033142, 1.7851536273956299, -0.5850751399993896, -2.3379290103912354, -1.0316959619522095, 0.6722044944763184, 0.17402240633964539, -0.6131332516670227, 0.15463963150978088, -1.9484657049179077, 0.24008937180042267, 0.13169297575950623, -1.7092281579971313, -0.4253116846084595, -1.028753399848938, 0.5853395462036133, 1.157231330871582, -0.9950016140937805, -0.6153737902641296, -0.8605098724365234, 0.7606477737426758, 1.2446036338806152, -1.5295106172561646, 1.959003210067749, 0.041784368455410004, -0.196940615773201, 1.640022873878479, 0.03034311719238758, -2.001051902770996, 0.4714440405368805, 0.9951751232147217, -1.1520415544509888, 0.5982352495193481, 0.09099650382995605, -0.14787164330482483, 0.1597355604171753, -0.9953212141990662, 0.510554850101471, -2.2293927669525146, -0.7883526682853699, -0.222269669175148, 1.1369343996047974, -2.1069507598876953, 0.8886755108833313, 0.3223288655281067, 0.46939122676849365, 0.6957923769950867, -0.6444101929664612, -0.1053488701581955, 0.41158804297447205, 0.6274454593658447, 0.9954638481140137, 0.1274574249982834, -0.7846585512161255, 0.9402154088020325, 0.9860574007034302, 0.8128917813301086, 0.5658771395683289, 0.4165993630886078, 0.7807151079177856, -0.4597064256668091, 0.8164213299751282, 0.032527074217796326, 0.2900441884994507, -0.5952615737915039, -1.7498602867126465, 0.7184985876083374, 1.3201313018798828, 0.7926871180534363, 0.13893088698387146, -0.06853577494621277, -0.21335062384605408, -0.7862116098403931, 0.03784581273794174, 1.1580874919891357, -0.3690594732761383, 0.8176584243774414, -0.18589210510253906, 0.5083103179931641, 1.7232142686843872, 0.5612616539001465, -0.6350478529930115, -0.34755390882492065, -1.5678809881210327, -0.2984713912010193, -0.6230552792549133, 0.7596270442008972, -0.38703879714012146, 0.3223840892314911, -0.20759624242782593, 0.48910096287727356, -1.3389108180999756, 0.953746497631073, 1.0508482456207275, 0.6103636026382446, -0.9372819662094116, 0.6987083554267883, 1.2480148077011108, -0.379731148481369, -0.8750891089439392, 1.264325499534607, 0.522661566734314, -1.3143552541732788, 0.3416370153427124, -0.9182901978492737, -1.2741029262542725, -0.6304090619087219, -0.40151914954185486, 0.34658730030059814, 0.5879523158073425, -0.6403090357780457, 1.0443753004074097, -0.7836583256721497, 0.018374811857938766, -0.44194239377975464, 0.15889744460582733, -0.18631580471992493, -1.3201024532318115, 0.6895497441291809, 2.9631619453430176, 2.031461477279663, -0.46752870082855225, -1.172348976135254, -0.6752853989601135, -0.029188167303800583, 1.0885869264602661, -1.4079134464263916, -1.6691228151321411, -0.15423691272735596, 1.5691922903060913, -0.019227227196097374, -0.06182435527443886, 1.2479084730148315, -0.053456004709005356, 0.7649759650230408, 1.5712393522262573, 0.5794530510902405, 1.9546551704406738, -0.2053917497396469, -1.3353629112243652, 1.0651880502700806, 0.32338541746139526, -0.0012788566527888179, -0.9892748594284058, 1.1174023151397705, -0.7922816276550293, -0.3972890079021454, 0.024839118123054504, -0.9850338697433472, 1.6197603940963745, 0.5009328722953796, -0.7086576819419861, 1.4692535400390625, -0.24662642180919647, 1.5201561450958252, 1.4441328048706055, -1.1660408973693848, -0.24510788917541504, 0.681329071521759, 0.6519190669059753, -0.37464553117752075, 0.5001474618911743, -0.8207846879959106, -0.9531376957893372, -1.0479846000671387, -1.240073800086975, 0.2616855502128601, 0.955686628818512, 1.064616084098816, -0.033828672021627426, 2.0199131965637207, -0.4923328459262848, 0.2070457637310028, -0.6485636234283447, -0.023727817460894585, 1.7342243194580078, 0.963930070400238, -1.4296470880508423, -1.655901312828064, -0.3443467915058136, -0.3201277554035187, -0.27636510133743286, -1.2993507385253906, -2.1398963928222656, 0.6522590517997742, -0.7055879831314087, -1.4009817838668823], [0.6510966420173645, 0.02855815924704075, -1.4841622114181519, 0.5945427417755127, -0.01717858947813511, 0.5638147592544556, 0.4410180151462555, 0.07157858461141586, -0.5226203799247742, -0.14556051790714264, -0.9742072820663452, 1.0033180713653564, 0.013187930919229984, -0.5897383689880371, 0.896530270576477, -0.6351686716079712, -0.8991097211837769, -1.4731475114822388, 0.32253164052963257, 0.9972331523895264, -0.844504714012146, 0.8454722762107849, 0.6877341866493225, 0.011800948530435562, 1.22278892993927, -0.827114999294281, 1.192165732383728, 0.8222264647483826, 0.9431532621383667, 0.08262621611356735, -0.26384708285331726, 0.1883840411901474, 0.4152510166168213, 0.34536412358283997, -0.12905025482177734, 0.5309967994689941, -0.14459453523159027, -0.310608834028244, 1.3187779188156128, 0.5133257508277893, -0.15769043564796448, -1.4444406032562256, -1.3136892318725586, 1.6195807456970215, 1.24290931224823, -0.23612183332443237, -0.32684245705604553, 0.04781469702720642, -0.26693475246429443, -1.1140246391296387, 0.6018779277801514, 2.560044288635254, -0.33691805601119995, -2.000441551208496, 0.5401883125305176, -0.45164090394973755, -0.7504687309265137, 0.26493188738822937, 0.263555109500885, 1.7249788045883179, -0.713322639465332, -0.6349008679389954, -0.5969691872596741, -2.1447958946228027, 2.594459056854248, -2.7415099143981934, -0.8128402829170227, -0.8070945143699646, 0.5266317129135132, -0.38073021173477173, 1.3768099546432495, 0.753593385219574, -0.001989521784707904, -3.0637433528900146, -1.0768542289733887, 0.7906679511070251, 0.5472184419631958, -0.294865220785141, 1.0195132493972778, -0.4434168040752411, 0.22386352717876434, -0.8362492322921753, 0.7963955998420715, -0.7011894583702087, -0.2485189586877823, 0.012369164265692234, -0.5104535818099976, 0.30639737844467163, 2.760188579559326, 0.002854923252016306, -0.4157201051712036, 0.27893683314323425, -1.0403425693511963, 0.7059741020202637, -0.9211871027946472, 0.8400060534477234, 0.13192524015903473, 0.7185014486312866, 0.21864232420921326, 0.27715209126472473, 1.1235133409500122, 0.2332206666469574, 2.0523641109466553, 0.040530409663915634, 1.0370895862579346, -0.1895885467529297, 1.0970321893692017, -0.49777260422706604, 0.2656554877758026, -0.7527549862861633, 1.5580753087997437, -0.2545195519924164, 0.04948042705655098, 0.3149847984313965, -0.6681764125823975, -0.4446084201335907, 0.42471426725387573, 0.7420307397842407, -0.5321077108383179, -0.30826860666275024, 1.2418303489685059, 0.6213150024414062, -0.024733876809477806, -0.006349972449243069, 3.174807071685791, 1.0701836347579956, -0.8024179339408875, -1.0596106052398682, 2.7395193576812744, -1.1427197456359863, -0.9381340742111206, 2.338611602783203, -0.01321343146264553, 1.2231229543685913, -0.8671700358390808, 0.4059467613697052, 0.2064344584941864, 0.1400795727968216, 0.08582551032304764, -0.17068994045257568, -0.12900179624557495, 1.014221429824829, 0.4836641252040863, -0.23952622711658478, 0.00013695325469598174, 0.8984639048576355, 0.4365435242652893, -0.18791484832763672, 0.12157370150089264, -0.2735999822616577, 2.3001708984375, -1.5533623695373535, -0.7058388590812683, 0.8183152079582214, 1.9971128702163696, 0.057230494916439056, -2.4014523029327393, 0.5901901125907898, 1.1186977624893188, 0.46675392985343933, 1.2786520719528198, -0.36433619260787964, 0.0494462251663208, 0.48880353569984436, 0.9178622961044312, 1.453322172164917, -1.193292260169983, 1.1710469722747803, -0.5117506384849548, 0.7190952897071838, -0.218508780002594, 1.5869218111038208, 0.8200969099998474, -0.021008802577853203, 0.29450851678848267, 0.4526686668395996, 0.014975924976170063, 1.3513870239257812, 0.010885776951909065, -0.6721875667572021, 0.49353963136672974, 1.3573009967803955, 0.1152675598859787, 1.7170544862747192, 1.0213923454284668, -2.0610201358795166, 0.5137820839881897, -0.6231407523155212, 1.6561591625213623, -0.654771089553833, -0.4321388602256775, 1.3065592050552368, -0.5330204367637634, -0.31667616963386536, 1.0078626871109009, -0.0009499051375314593, -0.43779757618904114, -0.5713681578636169, 0.32391437888145447, 0.5328746438026428, -0.4190136194229126, 0.2517063021659851, -1.7382206916809082, 0.06192811205983162, 1.0988023281097412, -2.0444421768188477, -0.30501922965049744, 1.573489785194397, -1.8043292760849, -0.040859609842300415, -0.6321310997009277, 0.7948406934738159, -0.09135385602712631, 0.678688108921051, -0.8803607821464539, 1.1195884943008423, -0.5945169925689697, 0.14152482151985168, -0.024124357849359512, -0.6073561906814575, 0.2793180048465729, 0.4228864312171936, 0.620786190032959, 1.2379156351089478, -0.13467620313167572, 1.4565138816833496, -0.5601211190223694, -1.5662821531295776, 1.3430545330047607, -1.8449217081069946, 0.2956118881702423, 0.2629123032093048, -1.4480303525924683, -0.3819836676120758, -1.1117945909500122, 0.2736630439758301, -0.12368594110012054, 0.7236145734786987, -0.17265155911445618, 2.31040096282959, 3.3289337158203125, -0.06655280292034149, -0.6456125378608704, -0.7226440906524658, -0.8700419068336487, 0.37768131494522095, -1.5183018445968628, 0.960398256778717, -2.3557441234588623, 0.22650383412837982, -0.5518962740898132, 1.3371798992156982, 1.7004854679107666, -0.20995686948299408, 1.9255374670028687, 1.2543305158615112, -0.18602420389652252, 0.7079802751541138, 2.0688369274139404, 0.3157028555870056, 0.5653254389762878, 0.07742305845022202, 0.3194112181663513, -0.22238686680793762, 0.46794506907463074, -0.42227259278297424, 1.562169075012207, 0.4998481571674347, -1.3317348957061768, 0.4930032193660736, 0.13549409806728363, -0.22073857486248016, 1.650187373161316, -0.4987807273864746, -0.8752531409263611, -1.7233378887176514, -0.34574538469314575, -0.44015640020370483, 1.2590243816375732, -1.1437047719955444, 0.8449171781539917, 1.1501331329345703, -0.015546664595603943, -0.6760055422782898, 0.6207386255264282, 1.559788465499878, 0.2584106922149658, -0.4658432900905609, 0.7040603756904602, 0.7765402793884277, 0.8988320827484131, 0.29359015822410583, -0.5372837781906128, 1.145622730255127, -0.22352828085422516, 1.4779393672943115, 0.2718089818954468, 0.8427427411079407, -0.004082500468939543, -0.6696736812591553], [-2.09631609916687, 0.9957324862480164, -1.0780014991760254, -0.07958418875932693, 0.31552654504776, 0.692755937576294, -0.10030143707990646, -0.4357108175754547, 0.7473392486572266, -0.1917172223329544, -0.3231084644794464, -0.09871169924736023, 0.39530593156814575, -1.268080711364746, 1.1457242965698242, -0.8448004722595215, 0.8900518417358398, 1.8323630094528198, 1.2904298305511475, -0.47417184710502625, 1.2920106649398804, 0.46766039729118347, -1.5288212299346924, 0.9198833107948303, -1.1294108629226685, -1.2502516508102417, -0.9963420629501343, -0.13097797334194183, -0.3763013780117035, -1.0629299879074097, 1.1213127374649048, 0.4032578766345978, -0.6005430817604065, 0.6186543107032776, 0.6743009090423584, -0.2644226551055908, 1.1396435499191284, 0.1316705048084259, 0.06081448495388031, 0.3162306547164917, 0.0901881605386734, -0.18313881754875183, -0.4714079797267914, 0.6265572309494019, -0.9997396469116211, 0.26860886812210083, 0.01867409236729145, -0.7326966524124146, 0.09140857309103012, 0.5967804789543152, 0.7555542588233948, 0.08713708817958832, -1.2432540655136108, -1.0424141883850098, 0.14795325696468353, 0.26077666878700256, 1.374749779701233, 1.513411045074463, 0.5429508686065674, -0.18963998556137085, 0.8752598762512207, 1.7291569709777832, -0.9466404318809509, 0.8330159783363342, -0.9187070727348328, 1.1311308145523071, -0.61287522315979, 1.182039737701416, 0.0049491943791508675, 0.9708176851272583, 1.3091349601745605, 0.7761229872703552, 0.3908156156539917, 0.1676890254020691, -1.2518495321273804, 0.39830154180526733, 0.48904845118522644, 0.5958425402641296, 1.1731663942337036, 2.433614492416382, 0.14894458651542664, 1.4244799613952637, 0.6648232340812683, 0.5432590842247009, -1.0012842416763306, 0.11826777458190918, 1.1251899003982544, -0.3181058466434479, 1.143186330795288, -0.4901858866214752, 0.7731929421424866, -1.1396714448928833, -2.2412796020507812, -0.28151941299438477, 0.42087724804878235, 0.35268139839172363, -1.1783406734466553, -0.1053735613822937, 1.5670392513275146, -0.6835967898368835, -1.0868810415267944, -0.479941189289093, 0.08590015023946762, -1.8655463457107544, -1.5958374738693237, -1.0142064094543457, -0.9939577579498291, 0.10764901340007782, 0.8396178483963013, -0.21184681355953217, -4.065252780914307, 0.004716923926025629, 0.44709089398384094, -0.15415681898593903, 0.8121773600578308, -0.43293923139572144, 0.9916632175445557, 0.44550859928131104, 0.0034709693863987923, -0.6965740323066711, 1.0862177610397339, 0.21270181238651276, 0.5989804863929749, 0.6703776121139526, -0.7667949199676514, -0.25367069244384766, -0.27850937843322754, -0.48729756474494934, -0.839916467666626, 0.08292561769485474, 0.865134596824646, -1.904051423072815, 1.0629414319992065, 0.9767745733261108, -1.0834823846817017, 0.6345327496528625, -0.25741642713546753, -0.1546912044286728, 0.06943684071302414, -0.1952471137046814, 0.31120583415031433, 0.6221733093261719, 0.9056877493858337, 0.7440751791000366, 0.9525882601737976, -0.24143511056900024, -0.66153484582901, 2.2702081203460693, -1.58637273311615, 0.16850882768630981, -1.3322031497955322, 0.3989463150501251, 0.674709141254425, 0.19459199905395508, -0.661447286605835, -0.2737496793270111, 1.4635688066482544, -0.7127470970153809, -0.4660155773162842, 0.5197567343711853, -2.316002368927002, 0.8456994295120239, 0.05999842286109924, -2.50764536857605, -0.4332486093044281, -0.6036094427108765, 1.2694168090820312, 1.7360270023345947, 0.46093952655792236, -1.025149941444397, 1.5323448181152344, 1.1602319478988647, -0.6725923418998718, 0.49840500950813293, 0.10104026645421982, -0.3878360092639923, -1.0357125997543335, 0.7290241122245789, 0.22057119011878967, -0.40379852056503296, 0.5221741795539856, 1.143494725227356, -1.5629847049713135, -2.116812229156494, 0.9139163494110107, -0.24611559510231018, 0.2328355312347412, 1.5192413330078125, 0.9316744208335876, 0.5573031306266785, 0.5379431247711182, -0.916623592376709, 1.6561172008514404, -0.1783248335123062, -0.975123405456543, 0.9690548181533813, 0.07157590240240097, -1.6431782245635986, -0.1420600712299347, -1.6226472854614258, -1.496130108833313, 1.79189133644104, 0.04338899999856949, 0.5695454478263855, 0.056411877274513245, 0.34969642758369446, -1.3921492099761963, 0.28643494844436646, -0.8483312726020813, -0.3850133419036865, -1.6624422073364258, -0.6982555985450745, -0.12326229363679886, -0.7035870552062988, -0.9778863191604614, 2.204010486602783, -1.3071824312210083, -0.265535831451416, 1.0588148832321167, 0.893008828163147, 0.6020441651344299, 0.601564884185791, 0.6098842620849609, -1.4833024740219116, 0.9216119647026062, -0.05701841786503792, 0.45071715116500854, -0.2874952554702759, -2.2995967864990234, -0.1358909010887146, -1.4374618530273438, 0.935592532157898, -0.2191172093153, -1.0984758138656616, 0.5641159415245056, 0.6068699359893799, -1.413878083229065, 0.5950332880020142, 0.9916254281997681, -3.1150972843170166, 3.24857234954834, -0.05592700466513634, 0.2641925811767578, -1.001160740852356, 0.5318626761436462, 1.4223552942276, -0.6449617147445679, -0.7202780246734619, -1.4468603134155273, 0.16661910712718964, -0.7438501715660095, 0.3444550633430481, -0.010481355711817741, -1.6784628629684448, 0.8182400465011597, 0.09003832191228867, 0.8187088370323181, -0.8565661311149597, -0.45834341645240784, -1.3515242338180542, -2.9508583545684814, -0.6030258536338806, -0.39155811071395874, 0.05369020253419876, -0.04106961563229561, -1.1677781343460083, -2.6154580116271973, -0.23667235672473907, 0.5040592551231384, 2.9550700187683105, 0.8415142893791199, -1.389164686203003, 1.4988042116165161, 2.4570021629333496, -0.8474159240722656, 0.695811927318573, -0.2962454855442047, 0.18372471630573273, 0.10242018848657608, -0.747712254524231, 1.332337498664856, -0.7288191318511963, 1.1127361059188843, -0.9616401791572571, 0.3690478801727295, -0.8103638887405396, 0.33445802330970764, 0.860641360282898, -0.28777363896369934, 0.16186393797397614, -0.9362356066703796, 0.9153929948806763, 0.8581641912460327, 0.3424699604511261, 0.7709596753120422, -1.051513433456421, -0.022123755887150764, -1.4811152219772339, 1.7344839572906494, 0.8531411290168762], [-0.8766454458236694, 2.0554165840148926, 1.5973851680755615, 1.0555616617202759, 0.45538049936294556, 0.8333100080490112, -0.47109001874923706, 0.42399367690086365, -0.07457908242940903, 0.2038676142692566, 0.06348942220211029, -1.7798963785171509, -0.40789398550987244, 0.6404164433479309, 0.7552226781845093, 0.3698979616165161, -0.758661150932312, 0.5515491366386414, 0.023741770535707474, 1.1638075113296509, -0.7108796238899231, -0.8231298327445984, 0.6865683794021606, 0.4046552777290344, -0.69439297914505, 1.117435097694397, 0.26305827498435974, 0.002610594965517521, -0.4568009078502655, -0.10671499371528625, -1.0158569812774658, -2.619370937347412, 0.3347245752811432, -0.6770002841949463, 0.1579100787639618, -1.7852660417556763, 0.2003888487815857, 0.13859684765338898, -0.6009181141853333, 1.0512231588363647, -0.05621073767542839, -0.2509126663208008, 1.300660490989685, -2.680753469467163, 2.02630352973938, 0.5484431385993958, -0.19620731472969055, -1.5440499782562256, 1.3112748861312866, -1.4737974405288696, -1.436241626739502, 1.507311463356018, 1.3653969764709473, 0.5381242632865906, 0.8060507774353027, 1.5150644779205322, 0.09331723302602768, 0.31807559728622437, -0.2873225510120392, 0.04837113618850708, 1.7227448225021362, -0.5363305807113647, 0.07517929375171661, -0.1516764760017395, 0.799127995967865, -0.729221761226654, -0.6507960557937622, -0.6568390130996704, -0.9166689515113831, -1.1571954488754272, 1.6363035440444946, -0.5691519975662231, 0.017109805718064308, 0.6254744529724121, -0.24224400520324707, -0.7879347205162048, 0.7121990919113159, 1.2944663763046265, 0.9331279397010803, 0.0665101632475853, 0.44326308369636536, -0.06648018211126328, -0.5210353136062622, 1.0588884353637695, 0.08571220189332962, -1.2530667781829834, 1.6338000297546387, 0.3448428809642792, 0.8943673968315125, -1.6260305643081665, -1.226737141609192, 0.3354402482509613, -0.237654447555542, -0.7308619022369385, -1.6043206453323364, 0.5447636246681213, 0.05370072275400162, 0.3484531044960022, -0.15345901250839233, -0.9619995951652527, -0.9305391907691956, 1.0816015005111694, 1.310909390449524, -1.1298377513885498, -2.98002290725708, -1.0420979261398315, 0.7755392789840698, 1.0184346437454224, -0.4251917004585266, -0.4955100417137146, -0.5859208106994629, 0.4618068337440491, 0.13375328481197357, -1.3151400089263916, -0.15953496098518372, -0.551601767539978, -0.03569367527961731, 0.7622309923171997, -1.2062472105026245, 0.9411388635635376, -0.3744516670703888, 2.243922472000122, 0.5210356712341309, -1.3052617311477661, -0.5762454867362976, 0.7531289458274841, -1.541098713874817, -0.5649362206459045, -0.20993651449680328, 0.38573452830314636, 0.37646201252937317, 0.5996224284172058, -0.2692239582538605, -1.6350743770599365, -1.0085898637771606, -1.0047221183776855, -0.338490754365921, 0.589202344417572, 2.439714193344116, 1.2810125350952148, 0.07086370885372162, -2.380829334259033, -1.2795771360397339, 0.8902153372764587, 1.365846037864685, 0.7275493144989014, -1.1302769184112549, 0.9498444199562073, 0.5686237812042236, 0.8643062114715576, -0.3589733839035034, -0.2510015070438385, -0.8560221791267395, -1.699480652809143, 1.3752093315124512, -0.16631868481636047, 0.995335042476654, 1.7464535236358643, -0.540328323841095, -0.7370206713676453, -0.58670973777771, 0.7718777656555176, -1.977556824684143, 1.0070548057556152, 0.39141395688056946, 0.3402973711490631, -0.32915639877319336, -0.7019036412239075, -0.9663720726966858, -0.06811865419149399, -1.897218108177185, 0.18976713716983795, -0.24138978123664856, -1.1242529153823853, -0.08016076683998108, 0.9940282702445984, -0.38049718737602234, -1.7990132570266724, -1.012351632118225, 0.7817182540893555, -0.566463828086853, -0.9964393973350525, 1.6063332557678223, -0.39308884739875793, -0.1710715889930725, 1.1568998098373413, 0.342274010181427, 0.7807657718658447, 0.08170729130506516, -0.018178287893533707, -0.4597613513469696, 0.38657280802726746, -0.46536165475845337, 0.7816603183746338, 0.4529278576374054, -0.9887470006942749, -2.9568824768066406, -0.04665086045861244, 1.2700759172439575, -0.24897809326648712, -0.13398440182209015, 0.6149998903274536, -0.17740239202976227, -0.662004292011261, 2.1436991691589355, -0.9463014006614685, -0.6976811289787292, -1.7342489957809448, 0.4989292025566101, 0.00973579939454794, 0.548560380935669, 1.0384705066680908, 0.11607307940721512, -0.007654465269297361, 0.7794070243835449, -0.2783595621585846, 1.905328631401062, -0.8171471357345581, -0.8316044211387634, -0.9736666083335876, 0.22922545671463013, 0.026648201048374176, 0.7747315764427185, 0.41415563225746155, -1.7403526306152344, 0.49023890495300293, -0.19641900062561035, -0.011244035325944424, 0.16575151681900024, 0.48082399368286133, 1.0894993543624878, 0.10282661020755768, 0.5806591510772705, 2.8409271240234375, -2.7028911113739014, -0.68147212266922, 1.049849271774292, 0.483910471200943, -0.17336031794548035, 2.2808938026428223, 0.031167814508080482, -0.9823884963989258, -0.35011300444602966, -0.21687914431095123, -1.0319173336029053, -1.623308777809143, -0.9629982709884644, 0.7015101909637451, 1.2749176025390625, -0.08551819622516632, -0.6657597422599792, 1.7034317255020142, 0.4858250916004181, 1.557011365890503, 0.011800583451986313, -0.6947616338729858, 0.34176206588745117, 0.38827064633369446, 1.4629930257797241, 0.049807485193014145, -0.0642429068684578, -0.21863888204097748, -0.4813242256641388, 0.508674681186676, -0.9866486191749573, -1.2680176496505737, 0.5788417458534241, 0.5044228434562683, -1.6285995244979858, 0.5287421941757202, -2.146423816680908, -0.4887985289096832, 0.3839740753173828, -1.6678937673568726, 0.03421184793114662, -0.4042964279651642, -0.5955629944801331, 0.7472178936004639, -1.1373845338821411, -0.38316649198532104, -0.16524943709373474, -0.8825733065605164, -0.1340823918581009, 0.5611228942871094, 0.5757374167442322, 1.250603199005127, -0.8078489303588867, 1.0504920482635498, 0.6447382569313049, 1.2938153743743896, 2.768935441970825, -0.9607098698616028, -0.6631892919540405, -1.1506174802780151, 1.7856258153915405, 1.4745839834213257, -0.01154317520558834, 0.07832483947277069, 1.0058538913726807, 0.3064313232898712], [-0.7028698325157166, 0.28321459889411926, -1.855015754699707, -0.12758180499076843, -0.513119101524353, -1.3432823419570923, 0.44090840220451355, 0.37845057249069214, 0.28081372380256653, -0.03975987434387207, -0.5703821778297424, -0.13237078487873077, -0.5468067526817322, -0.19849511981010437, -0.38350874185562134, -0.19180378317832947, -2.0259687900543213, -1.168416142463684, 0.8597298264503479, 0.856898307800293, 0.42398038506507874, 0.6656256318092346, 1.9568065404891968, -0.17781049013137817, -0.7398166656494141, 1.1573257446289062, 1.052027702331543, -1.6384825706481934, 0.7739351391792297, 0.20804846286773682, -0.44901013374328613, 0.3480828106403351, 0.36432281136512756, 2.2766897678375244, 0.06203272566199303, 1.6819192171096802, 0.905364990234375, 2.229440450668335, -1.4663043022155762, 0.35013270378112793, -0.5566781163215637, 0.3332814574241638, -2.0241429805755615, -0.569721519947052, 0.24921682476997375, 0.40178990364074707, -0.8786877393722534, 0.26097986102104187, 0.7923377156257629, -1.0444953441619873, 0.6493039727210999, 0.7797902226448059, 0.6315390467643738, -0.3412575125694275, -0.07232589274644852, 0.6036013960838318, 0.309436172246933, 0.0027977905701845884, -1.4461151361465454, 1.2964890003204346, -0.18838512897491455, 0.9310901761054993, 0.6968091726303101, -1.1555508375167847, 0.0140765979886055, -0.9890986680984497, -0.08692159503698349, 0.4436313509941101, 0.7644811868667603, -0.7781615257263184, -0.027721310034394264, 1.1873892545700073, -0.0999823808670044, 0.6621373295783997, 0.33941954374313354, -0.495474249124527, -0.9823209643363953, -0.7380045652389526, 1.3869212865829468, 1.002487301826477, 1.0274596214294434, -0.9889755845069885, 1.2532618045806885, 1.8696227073669434, 1.848940372467041, 1.1416068077087402, -0.3540458381175995, 0.3125990927219391, 0.26528361439704895, -1.0532071590423584, 1.5781018733978271, -0.2970108091831207, -0.5568886995315552, -1.0152643918991089, 2.0201191902160645, 1.8076722621917725, -0.9729203581809998, -0.8445635437965393, 0.9853964447975159, -0.9985213279724121, 1.5445998907089233, -2.0358686447143555, -0.18644940853118896, 1.0983588695526123, 0.7909570336341858, -0.05920834466814995, -0.14588792622089386, 0.7151893973350525, -0.10856850445270538, -1.132444143295288, -1.3708492517471313, 1.7355067729949951, -0.12533533573150635, 0.24112215638160706, 0.06587933748960495, 1.321303129196167, -0.508397102355957, -0.4943666160106659, 0.3770599365234375, 1.7049484252929688, -0.12883993983268738, -1.3849581480026245, -0.9744652509689331, 0.48266172409057617, 0.4016123414039612, -0.36563125252723694, -0.03541373088955879, 0.29053372144699097, 0.1975684016942978, 0.9019299149513245, -1.160133719444275, 0.3420467674732208, 0.11013283580541611, 0.27317389845848083, -0.34723883867263794, -0.29869142174720764, -0.3145776391029358, 1.8908175230026245, 0.7752258777618408, -0.0322762131690979, -2.045534610748291, -0.49144771695137024, 0.4636726677417755, 0.7171754240989685, -0.3642623722553253, -0.35279878973960876, 2.2583093643188477, 0.31851422786712646, 1.47849702835083, -0.23249982297420502, -0.08741067349910736, 0.07145468145608902, -0.257144033908844, -0.7207959890365601, -0.9600775837898254, 1.3425769805908203, 0.2296959012746811, -0.590288519859314, 0.26043587923049927, 0.239040806889534, -0.38688793778419495, 1.5542643070220947, -0.15061154961585999, 0.5078268051147461, -0.27798476815223694, 1.2961695194244385, 1.4253716468811035, -1.01659095287323, -0.3239797353744507, 0.6735953688621521, -1.5509884357452393, 0.3502756357192993, 0.2551172375679016, 0.9764425158500671, -0.05786651372909546, -0.328205406665802, 0.8554509878158569, -1.0832464694976807, -0.34340691566467285, 1.5101680755615234, 0.4604358375072479, -0.7253515720367432, 0.5418123006820679, 0.34372764825820923, 0.8715465664863586, -1.930794596672058, 1.3772341012954712, 1.2429416179656982, 0.4450155198574066, -1.8511290550231934, -0.9973707795143127, 1.069990634918213, -1.568458080291748, 0.3323930501937866, 2.023045539855957, -0.8481351137161255, -0.1330183446407318, -1.3350898027420044, 0.7301603555679321, -0.6367860436439514, -2.179810047149658, -0.29370763897895813, 0.27470308542251587, 1.5903300046920776, -0.2551187574863434, 0.5036599040031433, -0.6385977864265442, -0.45273324847221375, 0.4369736909866333, 1.9528437852859497, 0.24401599168777466, -0.43336614966392517, 0.26511573791503906, 0.21941356360912323, 0.14345648884773254, -0.5448479652404785, -2.907580852508545, 1.9164406061172485, 0.2978668808937073, -0.008035577833652496, 0.5104669332504272, -0.331531286239624, 1.0685433149337769, 1.215834140777588, -0.9091046452522278, -1.6996300220489502, -1.1574406623840332, 0.5895468592643738, -0.2378796637058258, -0.10093673318624496, 1.5004347562789917, 0.08482059836387634, -1.2737466096878052, 1.2077059745788574, 0.009987599216401577, -0.4188457131385803, -0.33154043555259705, 0.6734306216239929, 0.33186373114585876, 0.571398138999939, -0.2023029625415802, -0.6130667924880981, 0.9871384501457214, -1.2956796884536743, -0.13928468525409698, -0.2037336677312851, -0.3023889362812042, -2.7286481857299805, -1.02714204788208, 1.487443208694458, -0.26325029134750366, -0.04120888188481331, 0.7678273916244507, 0.6343544125556946, 0.844961404800415, 0.3659266233444214, 0.26877596974372864, -1.1731144189834595, -0.2618718147277832, -0.13791511952877045, 0.04960636422038078, -0.1751413643360138, 0.2826564311981201, 0.7680971622467041, -1.0452146530151367, 0.9854556322097778, 0.6984745860099792, -0.7238457798957825, 0.6638976335525513, -0.6574856042861938, 0.9577255845069885, 0.3176126778125763, 0.5286318063735962, 0.036686234176158905, 1.5109620094299316, 0.9797042608261108, -0.5576645731925964, -0.4096081256866455, 0.7356002330780029, 0.21805521845817566, 0.07410004734992981, 0.6607082486152649, 3.3174142837524414, -0.48778706789016724, 0.24392440915107727, -0.26304328441619873, 1.9526654481887817, -0.7574477791786194, 0.33815690875053406, 1.9810444116592407, 0.504588782787323, -1.141303300857544, 0.27036890387535095, -0.9296530485153198, -0.11267212778329849, 0.7573724389076233, -0.8544114828109741, -0.551621675491333, -0.7931559681892395, -0.9237858653068542], [-0.17951206862926483, 1.1628985404968262, 0.26959630846977234, 0.18180522322654724, 0.6636492609977722, -1.1990970373153687, -0.17052768170833588, 0.038060225546360016, 0.011154799722135067, 1.0120970010757446, 1.8301931619644165, 1.3778493404388428, -1.9769397974014282, -0.16355763375759125, -2.1434803009033203, -1.7266557216644287, 1.651228666305542, -0.16222825646400452, 0.16404512524604797, -0.658041775226593, -0.005346566904336214, -2.2433621883392334, -2.665894031524658, -1.3121308088302612, 0.3127569854259491, 0.09804248064756393, 0.6114755272865295, 0.25278621912002563, 0.5829166769981384, 1.483738899230957, -1.4284082651138306, 1.015252709388733, 0.04055023565888405, -0.6782702803611755, 0.18489263951778412, -1.07796049118042, -0.2150745391845703, 0.1946902871131897, -0.6373265981674194, 0.8223177194595337, -1.4670439958572388, -0.5426697134971619, 0.7138345837593079, -1.4487719535827637, -0.3054395914077759, -1.6593035459518433, 0.8265621662139893, -0.11857083439826965, -0.350183367729187, 1.2641828060150146, 0.15970160067081451, -0.579437792301178, -1.038934588432312, 0.24869535863399506, -0.2171795815229416, -0.0646136924624443, -1.013623595237732, 0.6735622882843018, -0.6934846639633179, 2.224191427230835, -1.2258082628250122, 0.888985812664032, 0.047567714005708694, -0.9128875136375427, 0.38434919714927673, 0.33011871576309204, -0.18846575915813446, -0.49000316858291626, -1.5653934478759766, 1.6306463479995728, 0.5179516077041626, -0.07669232040643692, 0.21947555243968964, -1.6015154123306274, 1.3521060943603516, 0.49518951773643494, -0.5387958288192749, -0.2042304277420044, 0.2897944450378418, 0.19753161072731018, -0.8655673861503601, -1.04661226272583, 0.3646990656852722, -2.5023653507232666, 0.22889640927314758, -0.47360050678253174, 0.5146478414535522, -0.44165536761283875, 1.7153321504592896, -0.3853505551815033, 1.0077593326568604, -1.2459708452224731, -0.561224102973938, 0.46749523282051086, -1.7920643091201782, 0.935293972492218, 1.433375597000122, 1.77448570728302, 0.42118188738822937, -0.6210981607437134, 1.3136566877365112, 0.5576884746551514, -1.354765772819519, -0.19479666650295258, 0.8591669201850891, 0.5099278092384338, -0.534905731678009, 1.1177947521209717, -1.5592845678329468, 1.4707915782928467, 0.650946319103241, 0.9425016641616821, 1.2802411317825317, 0.2921912968158722, 0.20555411279201508, -0.3203085660934448, -1.5459522008895874, 0.48203426599502563, 0.7219627499580383, -1.7079826593399048, 1.2549922466278076, -0.15340639650821686, 0.16375957429409027, -1.577453374862671, -0.3819870352745056, 0.9643800854682922, -1.0223909616470337, -1.2341392040252686, 0.010199462063610554, -0.6558744311332703, 0.3551027476787567, -1.6228266954421997, 0.1572050005197525, 0.8346897959709167, 0.04034122824668884, -0.7851327061653137, 2.220510721206665, -0.18226642906665802, 0.9252564311027527, 0.5095324516296387, -0.1385640949010849, -0.8798716068267822, 2.016925573348999, -0.711215615272522, -0.705300509929657, 1.4909229278564453, -0.13357709348201752, -0.2998654246330261, -0.02932535484433174, -0.876204788684845, 0.04633009061217308, -1.425153374671936, 0.38740071654319763, -1.1010783910751343, -0.1446305811405182, 0.6888186931610107, -1.728823184967041, 1.2906996011734009, -0.5321825742721558, 1.9087886810302734, 0.1894541084766388, 0.07600133866071701, 2.124267816543579, 0.16895771026611328, -1.1661152839660645, 3.4135875701904297, -0.06767427176237106, 1.9106696844100952, 0.7170260548591614, -0.6615990996360779, 0.14150580763816833, -0.6021562814712524, -0.8686085343360901, -1.2523603439331055, -0.4171510636806488, 1.1059600114822388, -0.4089280664920807, 0.4262222945690155, -0.3183380663394928, -0.549124538898468, -0.6148969531059265, 0.9436916708946228, 1.271992802619934, 0.2814680337905884, 0.8894490599632263, 0.6537482738494873, 0.26239892840385437, 0.16333214938640594, -0.27506303787231445, -0.04644463211297989, -0.8956834077835083, 0.12379468232393265, 0.8162810206413269, -1.403353214263916, 0.3460425138473511, 0.0511651337146759, 1.0429551601409912, -1.3214879035949707, -0.23346687853336334, -0.4044519066810608, 0.1083189845085144, -1.3785468339920044, 0.8439922332763672, -2.0257480144500732, 0.6542012691497803, -1.2236335277557373, -0.5908054113388062, 2.983506202697754, 1.1128082275390625, -1.0822227001190186, -0.36989182233810425, 1.4082605838775635, -0.11186634749174118, -0.033181749284267426, -0.5484806895256042, -0.22409746050834656, 0.4089832305908203, 1.81163489818573, -0.770206093788147, 0.20849262177944183, -0.03937690705060959, -1.0660793781280518, 1.2265057563781738, 1.1842865943908691, -0.49702146649360657, 0.9332167506217957, 0.04199584946036339, -1.3476018905639648, 0.13125653564929962, -1.647937536239624, 0.00675759045407176, 0.20117199420928955, -2.50542950630188, 0.010031145997345448, 0.7160393595695496, 0.4569195806980133, -0.40441855788230896, 0.28849467635154724, -0.5393558740615845, 0.12902666628360748, 0.15981459617614746, -0.8061212301254272, -0.506934404373169, -1.759995698928833, 0.15661470592021942, -0.08920657634735107, -0.8220378160476685, -0.6346116065979004, 0.2524265646934509, 1.000814437866211, -0.7007268071174622, -1.1653534173965454, -0.8990181088447571, 1.5023730993270874, 2.5395336151123047, -1.0612410306930542, 1.2756140232086182, 1.504090666770935, 0.3385818898677826, -1.0707058906555176, -0.14613521099090576, -0.7651292085647583, 0.695656955242157, 0.46065327525138855, 0.19492563605308533, -0.5862880945205688, -0.8056941032409668, -0.1256524622440338, 0.16287125647068024, -2.0886285305023193, 0.42426905035972595, -0.3674139678478241, 0.6923901438713074, -0.18111544847488403, 0.4702354371547699, -0.1794314682483673, 0.5495894551277161, 0.16943927109241486, 1.1221246719360352, 0.5803617835044861, 0.6264479160308838, 1.2652137279510498, 1.3394606113433838, 0.4468787610530853, -1.1552854776382446, -0.24647372961044312, 0.6620007157325745, 1.3524935245513916, -0.37854164838790894, -2.163534641265869, -2.0014400482177734, -0.23298560082912445, 0.02745475433766842, -1.1597520112991333, 1.333432674407959, 1.2108479738235474, -0.8159474730491638, 0.6039162278175354, 1.9274126291275024, -0.3961374759674072], [-0.6175726056098938, -1.5483351945877075, 0.4134857654571533, 0.6826269030570984, 0.6631066203117371, 1.4724799394607544, 0.5603710412979126, 0.46563637256622314, -0.2317156344652176, -1.5806795358657837, 1.444314956665039, -2.4707767963409424, 0.5480788946151733, -0.03491436690092087, -1.2337534427642822, -2.608551263809204, 0.38572022318840027, 2.3464956283569336, 0.04450102150440216, 0.3659626543521881, -0.6656791567802429, -1.3156789541244507, 1.4894044399261475, -1.2344950437545776, 1.7117350101470947, 0.6163210272789001, -0.18972136080265045, -1.2960504293441772, -0.5143980383872986, 0.740051805973053, 0.7103160619735718, -1.9429142475128174, 1.630184292793274, 0.30468228459358215, 0.06381329894065857, -0.9844447374343872, -1.226144552230835, 0.009585212916135788, 1.089644432067871, -0.9029496312141418, 0.9256142377853394, 0.3478243052959442, -0.23314014077186584, -1.300646424293518, -1.2632898092269897, -2.3823397159576416, -0.00024612960987724364, -1.1319986581802368, -0.7261434197425842, 1.2885538339614868, 0.11309701949357986, -0.4598110020160675, 1.18467378616333, -0.21821174025535583, -0.641600489616394, 0.7045295834541321, -1.3335641622543335, -0.21192507445812225, -0.11091325432062149, -2.1848549842834473, 0.1047697514295578, 0.22663941979408264, 0.2172733098268509, -0.35291069746017456, 0.7244908809661865, 1.258368968963623, -0.15066879987716675, -1.2848929166793823, -0.4313299357891083, 0.1710757613182068, 1.1210229396820068, -1.3239825963974, 0.1071353405714035, -0.2879463732242584, -1.1593034267425537, -0.013741766102612019, -0.7747467756271362, -1.0814557075500488, 2.169590711593628, 0.48354947566986084, -0.21574577689170837, 0.33876490592956543, -1.6469182968139648, 0.4815818667411804, -0.3350071609020233, 0.4701592028141022, 1.498636245727539, 0.3533508777618408, -0.05700602009892464, -0.718757688999176, -0.294856458902359, 0.27647581696510315, -0.45869845151901245, -0.06529337167739868, 0.8269652128219604, -0.20757657289505005, -1.9956328868865967, -0.5290673971176147, -0.045128971338272095, 0.3998878598213196, 2.036768913269043, -0.48716413974761963, -0.1631103754043579, 0.03411214426159859, 1.0712063312530518, -0.1933065801858902, 1.184108853340149, 0.5855537056922913, -0.26694563031196594, 2.7244105339050293, -0.6650639772415161, -1.2939950227737427, 1.0633115768432617, -0.23606207966804504, -1.0183953046798706, 0.4357939064502716, 0.2660491168498993, 1.2166545391082764, 0.9490641951560974, 0.9703658819198608, -0.37921929359436035, -1.0308572053909302, -0.6025713682174683, -0.51485675573349, -0.03751024976372719, -1.8165656328201294, 0.9342212677001953, -1.014288306236267, -1.1070291996002197, 1.1322450637817383, -0.8495607972145081, -0.4750746786594391, -0.5182591676712036, -1.0145635604858398, 0.276870459318161, -0.7874624729156494, -0.3757862448692322, 0.14766176044940948, -0.034195348620414734, 0.10302816331386566, 1.1338222026824951, -0.3149445652961731, -1.0701411962509155, -0.7374378442764282, 0.9797585010528564, -0.7046020030975342, 0.9747846126556396, 0.3039373755455017, -0.30658113956451416, -0.8247795701026917, -0.9030230045318604, 0.5358114838600159, -1.2965015172958374, 0.027052141726017, 0.2951759696006775, -0.12129618972539902, -0.9825812578201294, 0.11938504874706268, -1.974974274635315, -0.6512503623962402, -1.1922986507415771, 1.365294098854065, 0.5617356896400452, 1.4871727228164673, -0.5425792336463928, -0.45573997497558594, 0.793411374092102, -0.5443466901779175, -0.11410596966743469, 1.4106978178024292, 0.32951998710632324, 0.7365012168884277, -0.6822042465209961, 0.10478955507278442, -0.223493754863739, 0.8970013856887817, 0.5442274808883667, -0.20887508988380432, 0.49901100993156433, 0.9968447089195251, -0.23634417355060577, -1.6462777853012085, 0.7455008029937744, 0.7773455381393433, 1.0750787258148193, -0.24568191170692444, -1.7925200462341309, -0.7123748660087585, 0.2611951231956482, 0.7291277050971985, -1.3081259727478027, -0.48290351033210754, 1.111450433731079, -1.2933671474456787, -0.8104108572006226, 0.7981808185577393, 0.052799105644226074, 1.222123384475708, -0.8900610208511353, 0.5653661489486694, -0.39269107580184937, -0.21246224641799927, -0.18703457713127136, 0.9346672296524048, 1.5131199359893799, 0.6670681238174438, -1.369889259338379, 0.033870406448841095, 0.6415642499923706, -1.5479344129562378, -0.13826113939285278, 0.5870406031608582, -1.0596768856048584, 0.21406447887420654, 0.4962788224220276, 0.1193949282169342, -1.7152631282806396, 0.802412748336792, -1.309782862663269, -0.4411305785179138, 0.11118815094232559, -0.0862247422337532, 1.6471797227859497, -0.05879286676645279, 0.4814564883708954, 0.7735106348991394, 0.7910135388374329, -0.7108726501464844, -1.96756112575531, -0.803314208984375, -1.2787253856658936, -0.2863783538341522, 1.4178229570388794, 0.1584639847278595, -0.31815290451049805, 0.45570889115333557, -0.28645968437194824, 0.18851310014724731, 1.6517112255096436, 0.9703808426856995, 0.4258017838001251, -0.9318215250968933, 0.9091481566429138, 1.1558854579925537, -1.1521012783050537, -0.8991280794143677, -0.4643607437610626, 0.006200230214744806, -0.12185720354318619, 0.2836076319217682, 0.2199965864419937, 2.1307361125946045, 1.06828773021698, 0.634431779384613, -0.5048024654388428, 1.4274675846099854, -2.1017565727233887, 1.7667317390441895, 0.856194794178009, -0.11013422161340714, -1.0086350440979004, -0.20657044649124146, 0.6599348783493042, -1.7119519710540771, 0.9900175333023071, 0.7959080934524536, -0.2313780039548874, 0.8330793976783752, 0.9560381174087524, -0.9329383373260498, 0.10642096400260925, -1.4451674222946167, -0.28873100876808167, -2.234346628189087, 0.006983295548707247, 0.00968253891915083, 0.20880001783370972, -1.0669687986373901, 1.3913798332214355, 0.5910063982009888, -1.061797022819519, 1.4095271825790405, 0.5385315418243408, -0.5560684204101562, 2.254595994949341, 0.17326657474040985, 2.657102108001709, 0.5306262969970703, -0.7601807713508606, 0.6399245262145996, 0.9496422410011292, -1.816471815109253, -0.2292279154062271, 0.059317149221897125, 1.729209065437317, -1.422715663909912, 0.7263320684432983, 1.221146821975708, -0.6178471446037292, -0.09808673709630966]], "dtype": "float32"}], "e": {"__nd__": [[-0.032493073493242264, 0.0039137620478868484, 0.02516399696469307, -0.021498942747712135, -0.05084877461194992, 0.023555388674139977, -0.04630620777606964, 0.04044891521334648, -0.05383297801017761, 0.041169170290231705, 0.026224782690405846, 0.0036796964704990387, -0.05077696964144707, 0.11068727821111679, -0.07448073476552963, 0.08777455240488052, 0.03843960538506508, -0.05138538405299187, 0.08916645497083664, -0.008115471340715885], [-0.05249346047639847, 0.024316826835274696, -0.04494786262512207, 0.013792116194963455, 0.03409196436405182, -0.028286214917898178, 0.020365305244922638, 0.016857154667377472, 0.0818488672375679, -0.1477227807044983, 0.08480876684188843, -0.0780487135052681, -0.12536147236824036, 0.030261045321822166, -0.049554433673620224, -0.0855088159441948, -0.07215138524770737, -0.00228602415882051, 0.08078201115131378, 0.04824335500597954], [0.055653613060712814, -0.017865510657429695, -0.024370331317186356, -0.03227986395359039, -0.0930311381816864, -0.1247943863272667, -0.048021260648965836, -0.010818216018378735, 0.05430984869599342, 0.000590043084230274, 0.015203398652374744, 0.0012310676975175738, -0.03754264488816261, 0.00946411956101656, 0.04619801044464111, -0.06198501214385033, 0.0734512060880661, -0.02636738307774067, -0.0186325553804636, -0.0018922771560028195], [-0.050134360790252686, 0.07400798797607422, -0.036982350051403046, 0.09877005964517593, -0.0437735877931118, 0.029477210715413094, 0.010909557342529297, 0.08501216024160385, 0.03171643614768982, 0.11329400539398193, -0.04325486347079277, -0.008631670847535133, 0.0398225337266922, 0.0009637270704843104, -0.09777697175741196, -0.056847043335437775, 0.06613299250602722, -0.008837263099849224, -0.0035542924888432026, 0.02860628068447113], [-0.06225435063242912, 0.004052395466715097, -0.03282356634736061, -0.05149738863110542, -0.0035552054177969694, -0.013034123927354813, -0.03744401037693024, 0.009050002321600914, -0.06716708093881607, -0.024559589102864265, 0.02189088985323906, 0.030342578887939453, 0.08144718408584595, -0.046254198998212814, 0.024168994277715683, 0.09730006754398346, 0.0047050947323441505, -0.0019006620859727263, 0.1231769397854805, 0.036928825080394745], [0.10470204055309296, -0.027856020256876945, 0.038201045244932175, 0.009771013632416725, -0.07446300983428955, -0.04211178794503212, -0.052707914263010025, -0.0929974615573883, 0.00749296136200428, 0.028003007173538208, 0.013513620011508465, 0.018594998866319656, 0.04982537776231766, 0.024282170459628105, -0.04889050871133804, -0.0296315997838974, 0.10882226377725601, -0.01712707430124283, 0.07858923822641373, -0.039271239191293716], [0.01731303706765175, 0.010545683093369007, -0.10083160549402237, 0.07806774228811264, -0.021713392809033394, -0.07146628201007843, -0.0023797128815203905, 0.032354045659303665, 0.014377143234014511, 0.07991661876440048, 0.0810449868440628, -0.04904884472489357, 0.04985474422574043, -0.057459697127342224, -0.020586339756846428, -0.018992457538843155, 0.06107713654637337, 0.02542339451611042, 0.04474315792322159, -0.03845962509512901], [-0.05664370208978653, 0.0716804563999176, 0.022217007353901863, 0.022172868251800537, -0.0363331101834774, 0.07019087672233582, -0.002855525119230151, -0.01769770309329033, 0.009347707964479923, -0.03638011962175369, 0.030004965141415596, -0.0638570711016655, 0.023605523630976677, -0.029906494542956352, 0.15212634205818176, 0.08177470415830612, -0.03329678997397423, -0.007403792813420296, 0.0820874571800232, -0.09824216365814209], [0.038313817232847214, -0.013277068734169006, 0.11970118433237076, -0.059502024203538895, -0.03702940791845322, 0.06085024029016495, -0.0855555385351181, 0.01769200712442398, -0.00857832282781601, -0.021279839798808098, -0.02644091658294201, -0.006387257482856512, 0.030318809673190117, 0.021290041506290436, -0.09558424353599548, -0.08426833897829056, -0.05359348654747009, -0.07435226440429688, -0.06674596667289734, -0.11546508222818375], [-0.04872829094529152, -0.04634338617324829, -0.026829764246940613, 0.031561993062496185, -0.010911286808550358, -0.09311192482709885, 0.0937521904706955, 0.003808631096035242, 0.05036051943898201, -0.016622427850961685, -0.06791695207357407, -0.015012292191386223, 0.05144912376999855, -0.02349831908941269, 0.018779398873448372, -0.07523929327726364, 0.039091963320970535, 0.06257400661706924, -0.011247607879340649, 0.03364017978310585]], "dtype": "float32"}}, {"a": [{"__nd__": [[0.23848049342632294, -0.4832882881164551, -0.7634581923484802, 1.1760739088058472, 0.41787585616111755, -0.7103368639945984, 0.4117606580257416, 0.2235649824142456, -0.32937464118003845, 1.0496011972427368, -0.42861947417259216, 2.0333518981933594, 0.6812925338745117, 0.36248329281806946, 1.7847166061401367, 0.2566189169883728], [-0.8342156410217285, -0.7079607844352722, 0.6803518533706665, 0.6423999667167664, -0.7656562924385071, 1.508826494216919, -0.41168922185897827, 0.7284155488014221, 1.1376246213912964, 0.28185945749282837, 0.05857692286372185, -0.6593783497810364, 0.9215133190155029, -1.6143951416015625, -0.3631381094455719, 1.2340565919876099], [-0.9385424852371216, 1.2706584930419922, -1.2849304676055908, 1.715596318244934, 0.14867481589317322, -0.045314617455005646, 0.9076012372970581, -1.0543487071990967, 1.6140265464782715, 0.7650076746940613, 0.7795764207839966, 1.6931358575820923, -0.530164361000061, -0.7867743968963623, -0.22785648703575134, -1.308772087097168], [1.197519063949585, -0.46811026334762573, 0.4635811448097229, -0.7054866552352905, -0.6279017329216003, -0.8591184020042419, 0.893126904964447, -1.1527210474014282, -0.10813502222299576, 0.7459633350372314, -1.1584787368774414, 0.3765902519226074, -0.599187433719635, -1.4184056520462036, 0.1936519742012024, -0.9024530053138733], [-0.7697250843048096, 0.9294171929359436, -1.4722589254379272, -0.9765574932098389, -0.03343677520751953, -0.03075786866247654, 0.3684298098087311, -1.1933495998382568, -1.0384631156921387, 0.34262409806251526, -2.4307138919830322, -0.15800741314888, -0.6769605875015259, -0.7047545909881592, -0.8842859268188477, 0.585407018661499], [-0.39641568064689636, 1.748509168624878, -0.6769401431083679, 0.4802231192588806, -0.5043807625770569, 0.9513050317764282, 0.37820860743522644, -0.027462925761938095, 0.8815294504165649, 0.6933022141456604, -0.3787180483341217, 1.131721019744873, -0.7386857867240906, -0.02925441600382328, 0.625018298625946, -0.9160914421081543], [1.5171719789505005, -0.38145822286605835, -1.1875216960906982, -1.3298718929290771, -0.18998730182647705, 0.19555602967739105, -0.7815266847610474, 1.548505187034607, -0.21676155924797058, -1.1431490182876587, -0.821000337600708, -0.19893938302993774, -0.5739496946334839, 0.8293114900588989, -0.5959873199462891, 1.7318276166915894], [0.4714512825012207, -0.21474362909793854, 0.8792034983634949, -0.011538495309650898, 0.27428746223449707, 1.3753834962844849, -0.7996343970298767, 0.8033854961395264, 0.16743215918540955, -0.9073575735092163, -0.8269063830375671, 1.5293389558792114, 0.280908465385437, -0.3780008554458618, 1.4011130332946777, -0.9218126535415649], [-0.6867924332618713, 0.8293694853782654, 1.046170949935913, -0.11022715270519257, -0.44661837816238403, 0.37445637583732605, 0.5484429597854614, 0.17963609099388123, -0.3911232352256775, -0.8044005632400513, -0.2609180808067322, -1.297398567199707, 0.01681439019739628, -0.15217433869838715, 1.0565603971481323, 0.2319510281085968], [0.24184468388557434, -1.0874570608139038, -0.05296580493450165, -1.5647143125534058, 1.752712607383728, -0.4645240008831024, -0.10495753586292267, -0.9192573428153992, 0.954494297504425, 1.0676724910736084, -0.13733027875423431, 1.6495153903961182, 0.3861674964427948, 1.2855523824691772, -1.5965492725372314, -0.2751336097717285], [-0.8315312266349792, -0.22076639533042908, 0.07020303606987, -0.5300068259239197, -0.4079517126083374, -0.06662657856941223, 1.3864389657974243, -0.17312724888324738, -0.9541371464729309, 0.6508323550224304, 1.5169211626052856, 1.8966392278671265, -0.8171764016151428, 0.17992019653320312, 1.2877205610275269, 1.2919921875], [-0.7151235342025757, 0.7821674346923828, -0.21939711272716522, -0.9778686165809631, -1.4782140254974365, 0.003412277903407812, -0.42139652371406555, 0.5486621856689453, 0.6930205821990967, -2.0740833282470703, -0.6399340629577637, -0.9430901408195496, -0.48322388529777527, 0.9695661067962646, 0.07614786177873611, -0.13221968710422516], [-0.46841728687286377, 0.8538142442703247, 0.15214672684669495, -0.19259178638458252, 1.338125228881836, 1.238246202468872, -1.8997068405151367, -0.8774564862251282, 0.2006816416978836, -0.7811278700828552, -0.5770610570907593, 0.5098931789398193, -0.06780799478292465, -2.3012454509735107, 0.6991307735443115, 0.3609873652458191], [-1.347263216972351, 0.29451876878738403, 0.877480685710907, 1.9364607334136963, -1.4081721305847168, -0.11364518105983734, 0.6074166297912598, -0.9132093787193298, -0.04313277825713158, -0.573419451713562, 0.6408289670944214, 1.463808298110962, -1.5109978914260864, -1.2041287422180176, -1.580120325088501, -0.808295726776123], [-1.5437524318695068, -1.4637664556503296, 0.4895268976688385, -0.06644627451896667, -1.620875358581543, -0.9526219367980957, 0.552876889705658, -0.8949363827705383, -0.9700847268104553, -0.7421562671661377, 0.971666693687439, -1.6564674377441406, -0.8751175403594971, 0.48549705743789673, 1.2802586555480957, -0.633975088596344], [-0.6138206124305725, 1.253431797027588, -0.03444823622703552, -0.6277682781219482, 0.23391887545585632, -0.32983338832855225, -0.8616100549697876, -1.2448115348815918, 0.13794972002506256, -1.356668472290039, -1.3174562454223633, 0.41365867853164673, 1.4986441135406494, 0.32401272654533386, 1.2719899415969849, -0.5964935421943665], [1.069387435913086, -0.21076545119285583, -1.2595571279525757, 0.26169365644454956, 0.7343641519546509, 2.6573870182037354, -0.1511206179857254, -1.2880629301071167, 1.318021297454834, -1.250253438949585, 1.8964214324951172, 0.5670844316482544, 1.2315119504928589, 0.2239452451467514, -0.14077593386173248, -1.3100199699401855], [0.4399529695510864, -0.5655620694160461, -0.3749736249446869, -0.4200398027896881, -0.8605195879936218, 0.5784524083137512, -1.8603423833847046, -0.4531145989894867, -0.11161571741104126, -0.3293919563293457, 1.5679203271865845, -0.6546561121940613, -0.48664191365242004, 0.1703617125749588, 0.2852309048175812, -0.2607896625995636], [1.4694581031799316, 1.0572668313980103, -0.3611997365951538, 0.3902209997177124, -0.7698937654495239, -0.3219417333602905, -1.5796244144439697, -1.159203290939331, -1.7172198295593262, -0.33865463733673096, 0.5292490124702454, -1.185584306716919, -1.7241301536560059, -0.14290642738342285, -0.7135668992996216, -1.062186360359192], [0.5507820248603821, 0.810926079750061, 1.038210153579712, -0.6953814625740051, -0.024287430569529533, 0.7823530435562134, -0.5381788611412048, -0.7354835867881775, -0.6121058464050293, 1.2926453351974487, 0.016485359519720078, -0.5719611644744873, 0.5805202126502991, 1.4751060009002686, 0.6457774639129639, -0.23020991683006287], [-2.8079869747161865, -0.034720756113529205, -2.2659783363342285, -0.1617170125246048, -0.39717206358909607, 1.2579121589660645, -0.34323403239250183, -0.6482111811637878, -0.5302125811576843, -0.605038046836853, 0.3518036901950836, 0.429848313331604, 0.1193346157670021, -1.6261568069458008, 2.4025957584381104, 1.2228285074234009], [-0.1631821244955063, -0.6891315579414368, -0.8314619660377502, 0.08161935210227966, -0.41358765959739685, 0.27348852157592773, 0.8848642110824585, -0.8425782918930054, 1.3599973917007446, 1.9415309429168701, -0.8282104134559631, 1.3284785747528076, 0.9784475564956665, -1.856074571609497, 2.0853869915008545, -1.4500207901000977], [0.643243134021759, 2.173614978790283, -0.04645783081650734, -0.750328540802002, -1.2127456665039062, -1.2111823558807373, 0.24465125799179077, -0.04491724073886871, 1.8888740539550781, -1.6500999927520752, -0.2669057250022888, -0.9580071568489075, 1.0620756149291992, -0.4817711412906647, 0.3223773241043091, -0.7527814507484436], [-2.096045970916748, 0.5928891897201538, -0.9901378750801086, -0.9801643490791321, -0.27805647253990173, -0.040030043572187424, -0.7342625856399536, 1.267684817314148, -0.267703115940094, 0.7493983507156372, -0.8567696809768677, -0.5746533274650574, 0.9057580232620239, -1.414885401725769, -1.9893128871917725, -0.6982224583625793], [-0.8986155986785889, 0.20529066026210785, 0.7390395402908325, -0.2318921983242035, 0.5759193897247314, -0.8357696533203125, -1.4810869693756104, -0.590276837348938, 0.9178324937820435, -0.3952352702617645, 0.8914845585823059, 0.30250969529151917, -0.25953325629234314, 1.3598443269729614, 0.3349419832229614, 2.1049883365631104], [-2.093796491622925, 0.315382719039917, 0.4924253821372986, -0.42447373270988464, -1.6656253337860107, -0.8801103830337524, 0.5992578864097595, 1.6495351791381836, 0.8811649084091187, -0.04346329718828201, 0.4833238422870636, 0.4079417884349823, 0.7800378799438477, 1.7455676794052124, -1.067702293395996, -1.0563101768493652], [-0.48275265097618103, -1.2272721529006958, 1.502830147743225, -0.09325967729091644, -0.5975261330604553, -1.0852580070495605, -0.37164783477783203, -0.8920836448669434, -0.03436390310525894, -0.034659143537282944, 0.09095964580774307, -1.5502722263336182, -0.7332051396369934, 1.0740303993225098, 0.5367764830589294, -0.016942046582698822], [-0.873221755027771, 0.18553580343723297, -1.7159541845321655, -0.724567711353302, -0.18422330915927887, -0.7204720973968506, -0.6915022134780884, 0.8480733633041382, 1.3340191841125488, 1.055423617362976, -1.2549937963485718, 0.9264158010482788, -1.5897570848464966, 1.2124602794647217, 0.30903691053390503, 0.9793779253959656], [-0.4637283682823181, -0.3362864851951599, -0.7916208505630493, 1.652903437614441, -1.6575595140457153, 1.0993664264678955, -0.1929621696472168, 2.3011631965637207, 2.224858283996582, -0.461862176656723, 2.235344171524048, -2.561755895614624, 0.10867711901664734, -1.0132596492767334, -0.7596040368080139, 1.0559977293014526], [0.7486045360565186, 1.578397274017334, 1.4132846593856812, -1.1243293285369873, 0.466572105884552, -1.0751984119415283, 1.1392333507537842, 1.3516716957092285, 0.29628893733024597, -1.2972527742385864, -0.3115229308605194, 0.007966574281454086, 0.3744734227657318, 0.6158877611160278, 1.2017966508865356, -0.33875441551208496], [-1.3994028568267822, -0.7219441533088684, -1.3914140462875366, 1.6166006326675415, -0.9763758778572083, 0.1594509482383728, 0.9857625365257263, -0.2656317949295044, -0.6298372745513916, 0.6535532474517822, -0.10369177907705307, 2.023496627807617, 0.5291489958763123, 0.14236244559288025, -0.9703422784805298, -2.0824294090270996], [0.4186219871044159, -0.23224002122879028, -0.49641236662864685, -0.8379625082015991, -1.28163480758667, 1.2086282968521118, -1.7376611232757568, -0.9899666905403137, 0.04732817783951759, 0.5700623989105225, -0.41144558787345886, -2.2430410385131836, 0.4641868472099304, -0.8621312379837036, -0.5086528658866882, 0.4802514612674713], [-1.6976500749588013, -0.4335033595561981, 1.2951010465621948, 0.37505877017974854, 1.0649306774139404, -0.7063664793968201, 1.9869939088821411, 0.13390009105205536, 2.1074373722076416, 0.18285292387008667, 0.7878060340881348, -1.178256630897522, -0.6751907467842102, 0.2760264575481415, 0.8618685603141785, -0.9328495860099792], [0.5870180726051331, -0.6723112463951111, 0.9938997030258179, -0.16935820877552032, -0.6350282430648804, 0.015032070688903332, -0.6491402983665466, 0.2598027288913727, 0.21362099051475525, 1.5237178802490234, 0.5432907938957214, 0.9186481237411499, -1.1377209424972534, 0.03563477843999863, -1.1023850440979004, -0.8050452470779419], [-0.2088792324066162, 1.1356111764907837, -1.3166269063949585, 0.3348209857940674, 0.19818052649497986, 0.060196708887815475, -0.9057989120483398, -1.5157121419906616, 1.1581782102584839, -0.013257183134555817, 0.898632824420929, 0.6746232509613037, -1.3950307369232178, -2.115708112716675, -1.0031347274780273, -1.055659294128418], [-0.4811515212059021, 0.9010764360427856, 0.5445286631584167, 1.451622724533081, -1.3773255348205566, 0.0016978206112980843, 0.07933451235294342, -1.0808439254760742, -0.6043189167976379, 0.7153549194335938, -0.7647241353988647, 0.46954646706581116, -1.3114807605743408, 1.7348867654800415, -1.3986462354660034, 0.4012012779712677], [0.11288285255432129, 1.5458760261535645, -0.383919358253479, -0.33804452419281006, 1.407114863395691, 0.8186647891998291, -0.00976694654673338, 0.7214834094047546, 1.034593105316162, -0.6846198439598083, -1.898973822593689, -0.6686002016067505, -0.5398460030555725, 0.5301589369773865, -0.8537160754203796, -0.2694127559661865], [0.22319914400577545, -0.7056965827941895, -1.027312994003296, 0.7545021176338196, 0.9913774132728577, 0.7979219555854797, 0.4315902292728424, 0.161434605717659, -1.5892624855041504, 0.14336632192134857, -1.1511681079864502, -0.9553172588348389, -0.07643720507621765, 0.9980420470237732, 0.3306240439414978, -1.3838238716125488], [0.8516183495521545, -0.8720885515213013, -0.3862650692462921, -0.5685303211212158, -1.0198915004730225, -0.6878097057342529, 0.4358944296836853, -0.3070667088031769, 0.5175856351852417, 0.4749826490879059, -1.278032660484314, 0.47909408807754517, -0.9741579294204712, 1.6216728687286377, 0.555234432220459, -2.1893932819366455], [1.6818524599075317, 0.462066113948822, -0.1043662577867508, 0.1634608954191208, -0.7525202631950378, -0.05182609334588051, 0.1892247200012207, 0.6907051205635071, -0.3408239185810089, 0.3271995186805725, 1.741161823272705, -0.4370688199996948, -1.6324260234832764, -0.8352192044258118, 0.46290549635887146, -0.06403367966413498]], "dtype": "float32"}, {"__nd__": [[-0.40805375576019287, 0.3601214289665222, -0.3309919834136963, -0.5949813723564148, -0.648205041885376, -1.0738811492919922, -1.5679374933242798, -0.42322924733161926, 0.0040208022110164165, 0.0031451829709112644, 1.5743796825408936, 0.3956158757209778, -1.9408894777297974, 1.1708029508590698, -0.3308744430541992, 0.794818103313446], [0.2267540842294693, -0.5066660642623901, -1.2553709745407104, 0.20288363099098206, -0.9261227250099182, 0.9479244947433472, 0.09968037158250809, 0.16211450099945068, -1.2677778005599976, -0.345149964094162, -0.5743218660354614, -1.1648833751678467, -1.571283221244812, 1.8769830465316772, -1.2492687702178955, -0.8368026614189148], [-1.562414526939392, 0.09656881541013718, -0.49815505743026733, -0.18112356960773468, 0.6670639514923096, 0.4151685833930969, -0.806211531162262, 1.1604138612747192, 1.0857890844345093, -0.1447635293006897, 0.5393736362457275, 0.7044271230697632, -0.01551984902471304, -1.0760577917099, 0.6727721095085144, 0.11738847196102142], [0.6476448178291321, -2.2509374618530273, -0.7762830257415771, 0.4858700633049011, 0.35636430978775024, 0.4385770559310913, -1.1346685886383057, 0.5632225871086121, -1.876201868057251, 1.1790050268173218, -0.6890742778778076, 0.8707712292671204, 0.5530484318733215, 0.04341646656394005, -0.1428091824054718, -0.8533684015274048], [0.5660690665245056, 1.984560251235962, 0.40615925192832947, 0.2035011649131775, -0.3249509036540985, 0.49899473786354065, 0.08486580848693848, 0.34059634804725647, -0.6067445874214172, -0.1624123454093933, -2.244194984436035, 0.5434486865997314, 0.7902084589004517, -0.16392146050930023, -0.25064319372177124, 0.2732580602169037], [-1.884843111038208, 0.6706886887550354, -0.7000691890716553, -0.7766413688659668, -1.3626807928085327, -1.6609333753585815, 1.3361212015151978, 0.5471643805503845, 0.8226277232170105, -0.12558554112911224, -0.0005706221563741565, -1.7548574209213257, 0.2912122309207916, 0.12262780219316483, 1.9350894689559937, 0.7670531272888184], [1.5224579572677612, 0.5541281700134277, -0.638049304485321, -1.952021837234497, -1.216761589050293, 1.083053469657898, -1.7111908197402954, -1.8567947149276733, 1.1740479469299316, 0.04185274988412857, -0.14780639111995697, 0.38622725009918213, -2.318288564682007, 0.25053057074546814, -0.45717644691467285, -1.6334995031356812], [0.07259532064199448, 0.038802143186330795, 0.05338152498006821, 0.7234300374984741, -0.36558470129966736, -0.36657801270484924, 1.1818989515304565, 0.04077070578932762, 0.14359083771705627, 1.3922171592712402, 1.3407483100891113, -0.6949124932289124, 1.2081563472747803, -1.6392537355422974, -1.004576325416565, 0.5046823620796204], [0.8552480340003967, 0.30249032378196716, -0.016661757603287697, -1.662634253501892, -1.2939850091934204, 0.22258898615837097, -0.809488832950592, -1.0831010341644287, 0.6604712009429932, 1.4488699436187744, 0.24866147339344025, -1.137764811515808, -0.41269323229789734, 1.1557514667510986, 3.39876127243042, 1.2374948263168335], [0.5636993646621704, 1.2249306440353394, 0.9347566366195679, -1.10081946849823, 0.8973174691200256, 2.01216721534729, 0.04451775923371315, 0.6699905395507812, -0.10962263494729996, -0.19358862936496735, -0.799649178981781, -0.40759873390197754, -0.40875333547592163, -1.556564211845398, 0.8345750570297241, -0.11965833604335785], [-0.9287824034690857, -0.4006732106208801, 0.4399060904979706, -0.5748036503791809, -0.9970589280128479, -0.2044159322977066, -0.3438865840435028, 0.9531038403511047, -0.13253790140151978, -0.06480647623538971, 1.6274874210357666, 0.1398577094078064, 0.42791658639907837, -0.3990013897418976, 0.6670339107513428, 0.45907461643218994], [-0.028207959607243538, -0.5185310244560242, 0.7918710112571716, 1.2924598455429077, 0.6021300554275513, -0.2625313401222229, -0.3288743495941162, -1.954485535621643, 1.5913090705871582, 0.10019475966691971, -0.48121458292007446, -2.322901487350464, -1.1374119520187378, -0.38329026103019714, 1.4175420999526978, -1.41961669921875], [-0.008635930716991425, 0.10664919763803482, -0.12437225878238678, -0.8668662309646606, 0.4625207185745239, 0.515166163444519, -0.5105819702148438, 0.020741350948810577, 0.044247500598430634, 2.0557141304016113, 0.20253725349903107, 0.539341390132904, 2.1088387966156006, -0.7861403822898865, -1.7517129182815552, -0.6481993794441223], [0.1243850588798523, -0.4237968921661377, 1.5686837434768677, -0.9445138573646545, -1.4773887395858765, 1.0786916017532349, 1.4635593891143799, -0.15106211602687836, -0.9939553141593933, 1.4527337551116943, 1.2991456985473633, -1.1346862316131592, 1.1832523345947266, -1.0340510606765747, 0.8467342257499695, -0.621319055557251], [0.6218350529670715, 0.0772891491651535, -0.3898802697658539, 0.093685582280159, -0.48898619413375854, 0.4509054124355316, -0.1672118455171585, 0.22265830636024475, 0.6820597052574158, -2.255526065826416, -0.2963534891605377, -0.18009839951992035, 1.7337749004364014, -1.1307669878005981, 0.22213290631771088, -0.1236940249800682], [0.9477186799049377, 2.1693825721740723, 2.084944009780884, 0.22412432730197906, -0.35729923844337463, -0.17179137468338013, 1.2050338983535767, -0.4865865111351013, 0.4578597843647003, -0.375969797372818, 1.947297215461731, 0.3959061801433563, 0.56211918592453, 1.7220951318740845, 1.2030425071716309, -0.170443594455719], [0.4658031165599823, 1.3413619995117188, 0.5327946543693542, -0.3825472295284271, -0.8015733361244202, 0.12376265227794647, 0.07889465242624283, -0.14830262959003448, -0.9284928441047668, -1.5782854557037354, 0.2624954879283905, -0.6864302158355713, -1.4364289045333862, 0.020492279902100563, 0.3985721170902252, -0.2805757522583008], [0.9688255786895752, -0.09586101770401001, 0.8237082958221436, 0.007962572388350964, 0.8759562969207764, 0.9605487585067749, 1.3551872968673706, 0.00013359934382606298, 0.5642422437667847, 0.023885298520326614, 0.15882495045661926, -0.41117945313453674, -0.18095439672470093, 0.16530542075634003, -0.07743334025144577, -0.5664383769035339], [0.3612186908721924, 1.3837605714797974, 1.0559800863265991, -0.28843799233436584, 1.8381924629211426, 0.8013409376144409, 0.15537981688976288, -1.268824577331543, -0.0568213015794754, -0.4312801957130432, 0.09233588725328445, -0.48272505402565, -1.6509687900543213, 0.04779982566833496, -0.2182736098766327, -0.23098307847976685], [-0.08295639604330063, -0.1081569716334343, 0.1480022370815277, 0.04446052387356758, -0.4848778545856476, -1.296412467956543, 0.9872086644172668, -0.7709508538246155, -1.48129403591156, -0.47385135293006897, 0.6874891519546509, 0.1792830228805542, -0.6844971776008606, 1.5129687786102295, -3.2495008781552315e-05, -0.21973927319049835], [-0.24695195257663727, -0.2614491581916809, -1.2398864030838013, -0.6546132564544678, 0.5697066187858582, -0.6355512738227844, 0.38066259026527405, 0.6326429843902588, -1.2066892385482788, -0.32032057642936707, 0.9835380911827087, 1.238883137702942, -1.5323807001113892, 1.1949234008789062, 0.6751844882965088, -0.2568027973175049], [1.2117950916290283, -0.06406436115503311, 0.31539186835289, 0.31347817182540894, -0.418374240398407, -0.30730023980140686, 0.6853099465370178, -1.409650444984436, 0.21729503571987152, 0.23874537646770477, 1.5361683368682861, -1.3994207382202148, 2.4721553325653076, 0.8203406929969788, 0.20325785875320435, 0.0026318386662751436], [0.17733386158943176, 1.035316824913025, -0.9642050266265869, -1.0572046041488647, 1.4356670379638672, 1.4738249778747559, 0.4594367444515228, 0.06751463562250137, -0.5811830163002014, 0.6405680179595947, -0.5234786868095398, -2.1329803466796875, -1.1955597400665283, -0.2873041331768036, 0.2517102062702179, 0.5157509446144104], [0.2909051477909088, -1.2631202936172485, -2.5973846912384033, 1.9544062614440918, 0.0737610012292862, -0.2846701145172119, 0.25100573897361755, 0.9815253019332886, -0.41257256269454956, -1.0624059438705444, -0.7093926072120667, 0.9235628843307495, 1.9422096014022827, -1.2482895851135254, 0.2381214052438736, -0.7953217625617981], [-1.7382841110229492, 1.4596712589263916, 0.548046886920929, -0.12036991864442825, -0.19646525382995605, 2.3523755073547363, 1.3271722793579102, -0.2716284692287445, -0.5610874891281128, 0.3483292758464813, 1.4956756830215454, -0.4176144003868103, 0.2089947611093521, 0.7088873982429504, 1.378005027770996, -0.3065899908542633], [-1.5832310914993286, -1.3235466480255127, 0.9314098358154297, 0.47086459398269653, -0.4903004765510559, 0.21515880525112152, 0.08393759280443192, -1.1815996170043945, -0.3006207048892975, 0.7375626564025879, -0.21568071842193604, 2.017948627471924, 0.6280507445335388, 1.110731840133667, -0.5181100964546204, 1.0868728160858154], [-2.0165200233459473, -0.6694391965866089, 0.24308352172374725, -1.7073700428009033, -0.6840364336967468, -0.9347877502441406, -0.6998632550239563, 0.7074499726295471, -0.5849238634109497, -0.3801303803920746, -0.5893152952194214, 1.2185789346694946, -0.028111016377806664, 1.373421311378479, -0.45233356952667236, 1.1251956224441528], [-0.6638523936271667, 0.3852170705795288, 1.428570032119751, -0.1066281720995903, -1.962623119354248, -0.08325131237506866, -0.1616852581501007, -0.8908769488334656, 0.6745282411575317, -2.0505433082580566, 0.18853911757469177, 0.11506904661655426, 1.0536047220230103, -0.4416753351688385, -1.4869544506072998, -0.9434322118759155], [-0.2239450067281723, -1.7407349348068237, 0.7010902762413025, 0.6808730959892273, 0.2005768120288849, -1.185892105102539, 0.572425127029419, 1.9348993301391602, -2.122251033782959, 2.5162510871887207, -0.7253069877624512, -1.6946030855178833, -0.22870676219463348, 0.28597113490104675, -1.6633059978485107, 0.515110433101654], [0.0037995348684489727, -1.5504741668701172, -0.26083970069885254, 0.7592610120773315, -0.04507851600646973, 1.441478967666626, -0.3659849762916565, 0.9881129264831543, 1.706638216972351, 0.879252016544342, 0.19262847304344177, -0.8703444600105286, 0.6837919354438782, 0.6017675995826721, 0.5593991279602051, 1.085141658782959], [-0.842488169670105, 0.19639603793621063, 1.1248538494110107, 0.8063439726829529, -0.8434193134307861, 1.3539681434631348, -1.2128833532333374, 0.4379425346851349, -1.04046630859375, -0.16331058740615845, 1.2568929195404053, 0.5839557647705078, 0.3443574011325836, -0.605169415473938, 0.4701973795890808, -0.6193910241127014], [1.5583155155181885, 0.38851436972618103, 0.2662295401096344, -0.4708925485610962, 1.283105731010437, 0.6505438685417175, 0.7731580138206482, -0.9248195886611938, 0.9453230500221252, -1.136025071144104, 0.6957346200942993, 0.7466678023338318, 1.1638679504394531, -0.7424926161766052, 0.38914334774017334, 0.9056934118270874], [-0.1591428518295288, 0.028801145032048225, 0.7662601470947266, -0.17882569134235382, -0.045675553381443024, 0.39407145977020264, 0.09635453671216965, 0.6274088621139526, 0.869034469127655, 2.063075542449951, -1.4225318431854248, 0.24881374835968018, 1.070077657699585, 0.610029935836792, 0.5010862946510315, -0.1380033940076828], [-0.5167948603630066, 1.9816731214523315, -1.2085758447647095, -0.8759337663650513, 1.0102005004882812, 0.8295931816101074, 0.8747267127037048, -0.5306808948516846, -0.3794420659542084, 0.23015227913856506, -0.6818360090255737, -0.7809693217277527, -0.29654255509376526, 1.5225305557250977, -0.683373212814331, 0.8164800405502319], [-0.21913652122020721, 0.841067910194397, -1.1368129253387451, -0.02289724163711071, 0.051453400403261185, -1.1865363121032715, -0.215708926320076, -1.3184871673583984, -0.31872305274009705, 0.9412849545478821, -0.14622831344604492, -0.7890089154243469, 0.779697060585022, -0.21647536754608154, 0.33270230889320374, -0.03315967321395874], [-0.8491114377975464, 0.7410939931869507, -1.2448517084121704, 0.8016711473464966, 0.9358403086662292, -0.15659554302692413, -0.09286467730998993, 0.7626467347145081, 1.802614688873291, 0.1093813106417656, 0.6844797134399414, 0.7574135661125183, -0.18709124624729156, -0.618847131729126, -0.31021010875701904, 1.2653586864471436], [0.0036741977091878653, 0.8955148458480835, -0.4214778244495392, 0.7586365938186646, -2.3068649768829346, 0.30334410071372986, 0.5048425793647766, 0.8168175220489502, 0.44113653898239136, 1.1947158575057983, 0.7505670189857483, -1.4441181421279907, 1.3885093927383423, -0.7732797861099243, 1.8151884078979492, 0.7901832461357117], [1.1123050451278687, 1.9704959392547607, -1.6351726055145264, -0.48037344217300415, 1.74130380153656, 1.0033185482025146, 0.24753884971141815, -1.345909595489502, -1.9041646718978882, -0.7795820832252502, -1.0009214878082275, 1.5574045181274414, -1.2161730527877808, -0.20830942690372467, 0.14916658401489258, 1.0243690013885498], [0.32746753096580505, 1.03524649143219, -1.4154738187789917, -0.9707256555557251, -0.8068369626998901, 1.7935289144515991, 0.2585498094558716, -0.8093374371528625, 0.30864763259887695, 1.0476840734481812, 0.885555624961853, 1.3251804113388062, -0.28915345668792725, 0.6321231126785278, -0.6919819712638855, 2.168034553527832], [0.8140003085136414, -0.5561140179634094, 1.2978699207305908, -0.2827039062976837, -1.807384967803955, -0.867143988609314, 0.4634557068347931, 0.875264048576355, -0.24237504601478577, 0.016296254470944405, -0.1108994260430336, -0.1243007555603981, 0.18014080822467804, 1.4289008378982544, -1.5830602645874023, 1.9396600723266602], [-1.3238608837127686, 0.12485720962285995, -1.0591520071029663, -0.4428020119667053, 1.464181661605835, 0.31110575795173645, 0.08279745280742645, 1.2871131896972656, 0.37605395913124084, -0.1491524577140808, -1.7207294702529907, 0.33883389830589294, -0.7072713375091553, 0.15141020715236664, 0.5345860123634338, 0.5380921959877014], [1.368439793586731, -0.2070576399564743, -0.006638699676841497, -1.002677321434021, 0.858829915523529, 1.1575934886932373, 0.5003918409347534, -0.7582955360412598, -0.2696857452392578, 0.3442952334880829, 0.02572738751769066, -1.3639732599258423, -1.9070017337799072, -1.0482432842254639, 0.17028284072875977, -0.3664906322956085], [1.0662175416946411, -2.740422248840332, -0.38304248452186584, -0.5958897471427917, -1.418861746788025, -0.7661231160163879, -0.29888322949409485, 1.7185653448104858, -1.2176975011825562, 1.1229361295700073, 1.5058189630508423, 0.9324004650115967, 1.2273286581039429, -0.6017386317253113, 0.9563547968864441, 0.2886505722999573], [-0.48567062616348267, 0.5008842349052429, 0.4669712781906128, -0.05517861247062683, 0.33109670877456665, -1.1036744117736816, 0.239803746342659, 0.842261791229248, -0.28685763478279114, -1.9295178651809692, -0.6496959924697876, -1.7835917472839355, -2.110457420349121, 0.07945132255554199, 0.26035401225090027, -1.741196870803833], [0.167211651802063, -0.49014654755592346, -0.171650230884552, -0.5364668965339661, 1.0240952968597412, 0.17678169906139374, -0.5801364779472351, 0.3746674954891205, -0.9246684908866882, 0.7908735275268555, 0.9952630996704102, -0.25956714153289795, 0.2977685034275055, 0.2386985868215561, -0.8029230833053589, -1.0468626022338867], [-1.4305890798568726, 1.1466482877731323, -0.4061509370803833, 0.8120940327644348, 1.1589171886444092, 0.1629323959350586, -0.6647283434867859, 1.5098581314086914, -0.8446604609489441, 0.005533324088901281, -0.06153102591633797, -0.3499264419078827, 0.18671831488609314, -0.13578058779239655, 1.2359936237335205, 0.9435057640075684], [0.2954082787036896, -0.9602016806602478, 0.49950605630874634, 1.4213669300079346, -0.5028476715087891, -0.9425635933876038, -0.3845059871673584, -1.1926378011703491, 0.6237744092941284, 0.10467877238988876, -0.7654278874397278, 1.1681289672851562, -0.22039446234703064, 0.8960393667221069, 0.27627190947532654, -0.09479628503322601], [-0.5234492421150208, -0.49910175800323486, 0.17541125416755676, 0.8039579391479492, -0.702841579914093, 0.7945255637168884, 0.27910053730010986, -1.568674921989441, -2.3652617931365967, -1.1972934007644653, -0.6445640921592712, 0.23715956509113312, 0.4691195785999298, -0.12937882542610168, -0.7866150736808777, -1.680223822593689]], "dtype": "float32"}], "e": {"__nd__": [[-0.16305802762508392, -0.3161001205444336, 0.11149056255817413, 0.4127434194087982, 0.07711627334356308, 0.003338049864396453, -0.3524104654788971, -0.0489242821931839, 0.12530367076396942, -0.2619001269340515, -0.023221783339977264, -0.1656658798456192, -0.025429092347621918, -0.13831539452075958, -0.14588455855846405, 0.04603145271539688, -0.4632737636566162, -0.1736380010843277, -0.3856605589389801, 0.09380397945642471, 0.3232828974723816, -0.0372503325343132, -0.3495379686355591, 0.4701254069805145, -0.07318781316280365, 0.3495399057865143, 0.027371767908334732, -0.49802133440971375, -0.0019271757919341326, 0.10475905984640121, -0.03464769199490547, 0.06228949502110481, 0.3413006365299225, -0.20419813692569733, 0.14841482043266296, 0.18046239018440247, 0.1278214156627655, 0.16324055194854736, 0.033033862709999084, -0.16922424733638763, 0.20444755256175995, -0.39347246289253235, 0.38499730825424194, -0.4181710481643677, -0.14417286217212677, 0.24291600286960602, 0.47396713495254517, -0.1020849421620369], [-0.3111962378025055, -0.3021833300590515, 0.42030560970306396, -0.06280568987131119, -0.005007400177419186, 0.05552167817950249, -0.2636149525642395, 0.4527449905872345, -0.06493914127349854, 0.24224087595939636, 0.3631216585636139, 0.01619666814804077, 0.24147813022136688, 0.2930589020252228, 0.333676278591156, -0.2617850601673126, -0.31808075308799744, -0.014851205050945282, -0.2914441227912903, -0.6973870992660522, -0.6666075587272644, 0.014880100265145302, -0.031257402151823044, 0.09518374502658844, 0.1096215546131134, 0.1343948394060135, -0.08311399072408676, 0.2202058732509613, 0.08161527663469315, 0.6293941736221313, 0.3551334738731384, 0.06880679726600647, 0.22542452812194824, -0.30122891068458557, -0.2635134756565094, 0.30256688594818115, 0.44834235310554504, -0.4413060247898102, 0.09943421185016632, 0.11021601408720016, -0.008550689555704594, -0.09481973201036453, 0.129621684551239, -0.34271669387817383, -0.21697159111499786, 0.14370562136173248, -0.18568331003189087, -0.17358998954296112], [0.01712808944284916, -0.1873808056116104, 0.2997930943965912, -0.21333006024360657, -0.0729605183005333, -0.03853856399655342, 0.12232699990272522, 0.23911193013191223, -0.26208481192588806, -0.18610621988773346, -0.17860740423202515, 0.1649988740682602, 0.11216941475868225, -0.19777812063694, -0.0967925488948822, 0.10559189319610596, -0.19951952993869781, 0.030544402077794075, 0.08737196028232574, -0.07143047451972961, 0.06047401577234268, -0.0985160693526268, -0.1970549076795578, 0.22363434731960297, 0.17298734188079834, 0.08897330611944199, -0.33811500668525696, 0.047343697398900986, -0.38214758038520813, -0.13318949937820435, 0.02746225707232952, 0.05777844041585922, -0.028767380863428116, 0.01718411222100258, 0.20520545542240143, 0.5382292866706848, 0.022078663110733032, 0.10503970086574554, 0.17858777940273285, -0.48897063732147217, 0.005474611185491085, -0.13801677525043488, -0.3076600730419159, -0.13075850903987885, -0.1456819623708725, -0.05644087493419647, 0.23867720365524292, 0.01016716193407774], [-0.24134066700935364, -0.1272992640733719, -0.3924514651298523, 0.10954736918210983, 0.1310908943414688, -0.06849535554647446, 0.34217286109924316, 0.11399726569652557, 0.09794944524765015, 0.1366785317659378, -0.30965474247932434, 0.2563146948814392, 0.06929495185613632, 0.2776430547237396, -0.07145821303129196, -0.1253749281167984, 0.060267191380262375, 0.17289884388446808, 0.0625314861536026, 0.07141894847154617, -0.1488436609506607, -0.04518214985728264, -0.059219010174274445, -0.03612641245126724, -0.35311272740364075, -0.08304837346076965, -0.2389192134141922, 0.05450816452503204, 0.04773259535431862, -0.4039810299873352, -0.30272719264030457, 0.05751125141978264, 0.11748860776424408, -0.2903665602207184, 0.18521936237812042, -0.4270738363265991, -0.07828005403280258, 0.08047239482402802, -0.17984822392463684, -0.05655105412006378, -0.23603323101997375, 0.42058590054512024, 0.1075345054268837, 0.062453266233205795, -0.15423408150672913, -0.5970017313957214, 0.2512289583683014, 0.18166778981685638], [-0.06579795479774475, 0.259262353181839, -0.15958206355571747, 0.004784350749105215, 0.4628632366657257, 0.0933133214712143, 0.1869528442621231, -0.1256638914346695, -0.056912053376436234, 0.1500939130783081, -0.487606018781662, -0.10216698050498962, 0.08461818099021912, -0.2227635383605957, -0.1246795654296875, -0.5090168714523315, 0.08704575896263123, -0.2874889075756073, 0.15556950867176056, -0.0011049264576286077, -0.014734596014022827, -0.3801838457584381, 0.4477575719356537, -0.0012705553090199828, -0.12866288423538208, 0.017538584768772125, 0.19554896652698517, -0.09930811077356339, 0.07923299074172974, -0.4033742845058441, -0.4083224833011627, -0.19519789516925812, 0.0013333156239241362, 0.5619571208953857, 0.42658138275146484, -0.04682568833231926, -0.18902482092380524, 0.5602331161499023, 0.24488890171051025, -0.10233448445796967, 0.41443341970443726, 0.21308356523513794, -0.3974054753780365, 0.056485116481781006, -0.19201961159706116, 0.02953442558646202, -0.13692353665828705, 0.2504810690879822], [-0.019667189568281174, 0.0006089842063374817, 0.27577075362205505, -0.24127799272537231, 0.3384205400943756, 0.040189847350120544, 0.34115496277809143, -0.1406536102294922, 0.09261031448841095, 0.24692955613136292, -0.2002713978290558, 0.04458361491560936, -0.009836530312895775, -0.10169653594493866, -0.12030284851789474, 0.20371319353580475, 0.10005208104848862, 0.061038538813591, 0.15818049013614655, -0.19647721946239471, 0.08438432216644287, -0.3230784237384796, 0.08528535068035126, -0.011940117925405502, 0.40148231387138367, -0.0649818703532219, -0.22010968625545502, -0.06396753340959549, -0.45160597562789917, -0.09388402849435806, 0.12855002284049988, -0.07756766676902771, 0.2606601119041443, 0.24865692853927612, 0.0370880551636219, 0.3099188506603241, 0.1894671767950058, 0.2661314606666565, 0.32840660214424133, -0.4251208007335663, 0.23159728944301605, -0.01568710245192051, -0.38323110342025757, -0.036036308854818344, -0.2949806749820709, 0.11105819791555405, 0.04468486085534096, -0.0671912431716919], [0.22346867620944977, 0.35499271750450134, -0.07961700856685638, 0.10924246907234192, 0.1559019535779953, -0.06260059028863907, 0.15604807436466217, -0.3700767457485199, 0.11134179681539536, 0.0835895836353302, -0.05329527333378792, -0.4538660943508148, -0.19366499781608582, -0.38887810707092285, 0.1804269403219223, -0.2853669822216034, 0.1683233231306076, -0.14744701981544495, -0.1389804482460022, -0.07739777117967606, 0.23369121551513672, -0.24590101838111877, 0.24037766456604004, 0.03809300437569618, -0.4179328978061676, -0.24255158007144928, 0.32265162467956543, -0.20939117670059204, 0.08081918209791183, 0.1633855700492859, -0.34863293170928955, 0.058144863694906235, -0.16313765943050385, 0.2276199907064438, -0.27887362241744995, 0.04787494242191315, -0.14404906332492828, 0.26801422238349915, 0.2852438688278198, 0.4462284445762634, 0.3115083575248718, 0.1005309671163559, 0.16998602449893951, 0.04216001555323601, -0.05464530363678932, 0.033072080463171005, -0.2211829423904419, -0.3010968565940857], [-0.33993205428123474, -0.238101989030838, 0.3313411772251129, 0.2587953209877014, 0.3011571168899536, -0.3552975058555603, 0.1227104663848877, -0.5013941526412964, -0.000593031058087945, 0.5150505900382996, 0.023755749687552452, -0.03839585930109024, -0.08074338734149933, -0.033529121428728104, 0.3768594563007355, 0.057856254279613495, 0.0575382225215435, 0.15693548321723938, 0.022287210449576378, -0.3815355598926544, -0.01625065691769123, -0.3235432505607605, -0.2100738286972046, 0.20985282957553864, 0.06788788735866547, 0.005421485751867294, -0.03938835486769676, 0.06375650316476822, -0.4149722456932068, 0.07439194619655609, 0.43942829966545105, 0.24085325002670288, 0.21452464163303375, -0.37555521726608276, -0.5386144518852234, -0.138190358877182, -0.12987540662288666, 0.11221937835216522, -0.19697123765945435, -0.3238283395767212, 0.20921289920806885, -0.04297734424471855, 0.09497074782848358, -0.0005930135957896709, -0.10850447416305542, 0.11712595820426941, 0.11633270233869553, 0.1219915896654129], [-0.2164091020822525, -0.044445138424634933, -0.02398226410150528, -0.5121351480484009, 0.24963116645812988, 0.5437220931053162, -0.24823832511901855, -0.07403920590877533, 0.27011650800704956, 0.44920945167541504, 0.22301669418811798, 0.25765460729599, -0.4550003707408905, 0.3960827589035034, 0.2113579511642456, 0.34297069907188416, 0.5821851491928101, 0.13181662559509277, 0.2245875746011734, 0.06446394324302673, -0.24061986804008484, 0.08549768477678299, 0.30394232273101807, -0.3059718608856201, 0.5440016984939575, -0.24534673988819122, -0.034205496311187744, 0.22522681951522827, -0.07413869351148605, -0.0687277615070343, 0.2368842214345932, -0.03736814856529236, -0.029280710965394974, 0.15094980597496033, -0.047849975526332855, -0.2845674455165863, 0.409747838973999, -0.10038638859987259, -0.26857659220695496, 0.00936860777437687, 0.03788735345005989, 0.10756207257509232, -0.2940714359283447, 0.45297157764434814, -0.3930085599422455, 0.36516425013542175, -0.3429941236972809, 0.10416167229413986], [0.1241922378540039, -0.07769802212715149, -0.05703498050570488, 0.22417782247066498, -0.186930850148201, -0.40430623292922974, 0.21638768911361694, -0.11777027696371078, -0.14277604222297668, -0.2541998326778412, -0.28083544969558716, -0.1838649958372116, 0.4892506003379822, -0.2643100321292877, -0.3221621811389923, -0.09736181050539017, -0.5500600337982178, 0.14914701879024506, 0.10347069799900055, 0.03185982629656792, 0.16536526381969452, 0.007094993721693754, -0.14677076041698456, -0.1582213193178177, -0.3035484552383423, 0.4047982394695282, 0.2984367907047272, -0.11722828447818756, 0.02604193426668644, -0.010335640981793404, -0.4180436432361603, 0.19472433626651764, 0.25865018367767334, 0.15111564099788666, -0.006608509924262762, 0.078560009598732, -0.6448959112167358, 0.07684329897165298, 0.21074563264846802, 0.06345009803771973, 0.10344710946083069, -0.04476981237530708, -0.0013382364995777607, -0.352884441614151, 0.40066099166870117, -0.42035800218582153, 0.23446838557720184, -0.14283843338489532], [0.31003081798553467, -0.20225559175014496, 0.13921642303466797, -0.051320817321538925, -0.2626974880695343, 0.19304457306861877, -0.17497576773166656, 0.07022824883460999, 0.2463388890028, -0.12091271579265594, 0.4704136550426483, -0.38194262981414795, -0.19505012035369873, 0.24572549760341644, -0.40461310744285583, 0.28091177344322205, 0.025339042767882347, -0.10590080171823502, -0.07334354519844055, 0.4244132936000824, 0.5800747275352478, -0.11559194326400757, -0.1287994235754013, -0.1778610348701477, 0.40748685598373413, 0.4524068236351013, 0.3255259394645691, -0.27215126156806946, -0.05962482467293739, -0.1153094545006752, 0.16852830350399017, 0.0721902921795845, -0.09771893918514252, -0.007299266755580902, -0.11206511408090591, 0.11726225912570953, 0.10307791829109192, 0.20330746471881866, 0.4411713778972626, 0.09096135199069977, 0.00570489140227437, -0.08215894550085068, 0.3455541729927063, -0.3153046667575836, -0.12641926109790802, 0.11585000157356262, 0.025201408192515373, -0.10634325444698334], [0.24843423068523407, 0.3548935353755951, 0.04945055767893791, -0.4771786630153656, 0.14766477048397064, 0.4532082974910736, 0.2779988646507263, -0.5004062056541443, 0.1533498764038086, 0.03503928706049919, 0.1303170621395111, 0.013937282375991344, -0.4658755958080292, -0.21590080857276917, 0.35742196440696716, 0.06831180304288864, 0.5593672394752502, -0.21931597590446472, -0.0406479649245739, 0.08950654417276382, 0.020288866013288498, -0.15495026111602783, 0.03862572833895683, -0.16799308359622955, 0.06935320049524307, -0.2567129135131836, 0.3655001223087311, 0.46951621770858765, -0.3290930986404419, -0.07365383952856064, -0.05637364834547043, -0.21549470722675323, -0.21337927877902985, 0.2141878604888916, -0.15813548862934113, -0.09225054830312729, 0.06788954883813858, -0.12338977307081223, -0.059600237756967545, 0.20304352045059204, 0.17339400947093964, -0.13816657662391663, -0.31049245595932007, 0.5330352187156677, -0.3945217430591583, 0.011067744344472885, -0.1513754278421402, 0.004723922349512577], [-0.1025107279419899, -0.4352510869503021, 0.508419394493103, -0.048328474164009094, 0.22238072752952576, -0.2639565169811249, 0.18741896748542786, -0.17299817502498627, -0.016066912561655045, 0.5965331792831421, -0.07640115916728973, 0.1679632067680359, 0.08007334172725677, -0.19058017432689667, 0.3237629234790802, -0.2540782392024994, 0.07522738724946976, -0.14878010749816895, 0.380825012922287, -0.5972354412078857, -0.26878735423088074, -0.33601850271224976, 0.21854136884212494, 0.03579630330204964, 0.016482871025800705, -0.12376008927822113, -0.16277152299880981, 0.030280204489827156, -0.4760802388191223, -0.10469949245452881, 0.303708553314209, 0.34669339656829834, -0.13605011999607086, -0.05539599433541298, 0.03334931284189224, 0.2379106730222702, -0.15180136263370514, 0.4042983055114746, 0.019717222079634666, -0.5938393473625183, 0.24861548840999603, 0.2223670929670334, -0.3109610676765442, -0.056710295379161835, -0.10536370426416397, 0.31772246956825256, -0.16697874665260315, 0.040315378457307816], [0.16521839797496796, 0.05149669572710991, 0.029126469045877457, -0.15980201959609985, -0.04234131798148155, -0.1909288763999939, 0.07425559312105179, 0.19553373754024506, -0.5163177251815796, -0.21809467673301697, 0.03813257813453674, 0.05524090304970741, -0.11833115667104721, -0.03976350650191307, -0.09150076657533646, 0.0328659862279892, 0.19658340513706207, -0.09216942638158798, 0.09336254000663757, 0.21337135136127472, -0.03181011602282524, -0.26968908309936523, -0.42561468482017517, 0.013812756165862083, 0.06142646074295044, 0.3137965798377991, -0.042215220630168915, 0.5059536099433899, -0.08433477580547333, -0.3580312430858612, 0.37015658617019653, -0.21147851645946503, -0.33928975462913513, -0.2816767692565918, -0.20987007021903992, 0.11257332563400269, -0.15153329074382782, -0.058995429426431656, -0.027732303366065025, 0.00823257677257061, -0.2779705822467804, -0.13925990462303162, -0.2123090773820877, 0.1250576674938202, -0.2210342437028885, -0.22046160697937012, 0.3137064576148987, 0.42760059237480164], [0.2447085976600647, 0.26391202211380005, -0.17765973508358002, -0.09183087944984436, -0.5232008099555969, 0.5151374936103821, -0.048846859484910965, -0.05603211000561714, 0.26561301946640015, -0.33770450949668884, 0.402215838432312, 0.37980225682258606, -0.49874550104141235, 0.3600398600101471, -0.13437683880329132, 0.06642328947782516, 0.3415886461734772, -0.1712665557861328, -0.13446955382823944, 0.5670814514160156, 0.19380123913288116, 0.1736423373222351, -0.10530843585729599, -0.15811249613761902, 0.24677041172981262, 0.08207935839891434, 0.17917035520076752, 0.20875512063503265, 0.1264972984790802, -0.08299688994884491, 0.16585466265678406, -0.42753133177757263, -0.3222862482070923, -0.23372770845890045, 0.09431543946266174, -0.4444430470466614, 0.19125376641750336, -0.4080979526042938, -0.3522558808326721, 0.061652567237615585, -0.2727903723716736, 0.040986038744449615, 0.17108485102653503, 0.4235919117927551, -0.0813407376408577, -0.09689623862504959, 0.1053462028503418, 0.3127572536468506], [-0.16470201313495636, -0.2800637483596802, 0.07640792429447174, -0.19148007035255432, 0.43018707633018494, 0.18340270221233368, 0.0884343758225441, -0.4618447422981262, 0.17809148132801056, 0.12727674841880798, -0.16212280094623566, 0.1259404420852661, -0.05261260271072388, -0.20905308425426483, 0.4458475410938263, 0.1626097857952118, 0.12924610078334808, -0.2649265229701996, 0.0739704817533493, -0.04504229873418808, -0.15515252947807312, 0.10330620408058167, -0.12642057240009308, 0.10433970391750336, 0.07244294136762619, 0.019403783604502678, 0.15598860383033752, 0.30959925055503845, -0.5947060585021973, -0.3163551092147827, 0.0021490720100700855, 0.22925806045532227, 0.0906684622168541, 0.1612682044506073, 0.3586386740207672, -0.14268431067466736, -0.05884108319878578, 0.2338842898607254, -0.17279618978500366, -0.35249021649360657, 0.15387846529483795, -0.3119271993637085, -0.3596206307411194, 0.07002976536750793, -0.2882457375526428, 0.14627113938331604, 0.13885392248630524, 0.23497061431407928], [-0.1550736129283905, 0.019651323556900024, 0.11479853093624115, 0.011443164199590683, -0.22137334942817688, -0.39683520793914795, 0.3014615476131439, -0.005126999691128731, -0.07755705714225769, 0.08925305306911469, -0.09723369032144547, 0.07634736597537994, 0.1841389536857605, 0.0179072804749012, 0.42956650257110596, 0.17635726928710938, -0.08472470939159393, 0.3724432587623596, 0.1479966789484024, -0.23267026245594025, -0.059023819863796234, 0.3658357858657837, -0.015906423330307007, 0.2435014545917511, 0.2663339078426361, -0.09018003940582275, -0.44943979382514954, 0.17677752673625946, -0.6111087203025818, 0.2290765345096588, 0.19211925566196442, 0.5531625747680664, -0.22645269334316254, 0.016802439466118813, -0.13556958734989166, 0.1246301606297493, -0.0747038722038269, 0.1235499382019043, 0.26705408096313477, -0.4153461158275604, -0.24964025616645813, 0.1307089924812317, -0.0850009024143219, -0.27837419509887695, 0.21501493453979492, -0.2625679671764374, -0.07602294534444809, 0.1710662990808487], [0.552640438079834, 0.2483844757080078, 0.05097874253988266, 0.1305011510848999, -0.5086219310760498, -0.13516870141029358, 0.5225601196289062, -0.1580207794904709, 0.3805452585220337, -0.0944414883852005, 0.38666772842407227, 0.15344414114952087, -0.08902034908533096, 0.10852742195129395, 0.06084192544221878, -0.016402969136834145, 0.2688845098018646, -0.252327561378479, -0.02984800562262535, 0.0066575524397194386, 0.10773468762636185, 0.17545272409915924, -0.006063205190002918, -0.16677314043045044, 0.03878600150346756, -0.1717032492160797, -0.04736972227692604, 0.11504192650318146, -0.22682063281536102, 0.16631940007209778, 0.41316092014312744, -0.10663776099681854, -0.40964627265930176, -0.25841039419174194, -0.0202750526368618, -0.18035383522510529, 0.10585982352495193, -0.16044284403324127, 0.13645608723163605, -0.06408961862325668, -0.43763789534568787, 0.20799580216407776, 0.2736867666244507, -0.0011470163008198142, 0.2971540093421936, -0.15668871998786926, -0.07374148815870285, 0.030007218942046165], [0.4174209535121918, 0.48699361085891724, -0.4470409154891968, 0.07546062022447586, 0.016809582710266113, -0.2822568714618683, 0.5277107357978821, -0.13597024977207184, 0.04985671117901802, -0.003717118874192238, -0.23015528917312622, 0.2257736325263977, -0.17649254202842712, -0.06282918155193329, -0.12893187999725342, 0.0011090071639046073, 0.6230623722076416, -0.15136821568012238, 0.35383301973342896, 0.30706843733787537, 0.10331602394580841, 0.01117047667503357, 0.20466111600399017, -0.18367193639278412, -0.1472625881433487, -0.40943092107772827, -0.3011368215084076, 0.08905927836894989, 0.01538135763257742, -0.4737479090690613, 0.1536741554737091, -0.25860029458999634, -0.5198686122894287, 0.04297018423676491, 0.2380988895893097, -0.3893819749355316, -0.11847136914730072, 0.2780238091945648, -0.06219262257218361, -0.059084005653858185, -0.4347425401210785, 0.4109750986099243, -0.15889577567577362, 0.3733202815055847, 0.23419055342674255, -0.1705501824617386, -0.021011000499129295, 0.35935744643211365], [0.04219258576631546, 0.09101957827806473, -0.4070161283016205, 0.047568392008543015, 0.20408415794372559, -0.095796599984169, 0.21711872518062592, -0.1589432954788208, 0.5688962936401367, 0.19221340119838715, -0.08518259972333908, 0.09928147494792938, 0.223856121301651, 0.3954380750656128, -0.343996524810791, 0.4724850356578827, 0.08470582962036133, 0.19287411868572235, 0.26574045419692993, 0.11955558508634567, -0.13016319274902344, 0.40905770659446716, 0.237851083278656, -0.5609304904937744, 0.41345450282096863, 0.09312103688716888, -0.03775789961218834, -0.13575848937034607, 0.04169931262731552, 0.04843761771917343, 0.15596987307071686, -0.02968738041818142, 0.46372073888778687, 0.31792163848876953, 0.2096906155347824, -0.5118338465690613, 0.14824402332305908, 0.03623000532388687, 0.1804710030555725, 0.08113422244787216, -0.2738456130027771, 0.12655672430992126, -0.08249188214540482, -0.218039408326149, 0.26075875759124756, -0.027470987290143967, -0.027964022010564804, 0.02341763861477375], [0.024469150230288506, -0.1729494035243988, 0.5767368078231812, -0.02998592145740986, -0.12395413964986801, 0.38995081186294556, -0.1235971674323082, -0.052391886711120605, 0.22346492111682892, 0.11860374361276627, 0.3588878810405731, -0.04353097081184387, -0.17192433774471283, 0.023373981937766075, 0.17848096787929535, -0.29397475719451904, -0.019354965537786484, -0.45997703075408936, -0.1863834261894226, -0.17996008694171906, 0.16803008317947388, -0.2259870171546936, 0.16555088758468628, 0.2728673219680786, 0.3934454321861267, 0.11981374025344849, 0.15865391492843628, -0.1418967843055725, -0.3488342761993408, 0.10053660720586777, 0.2705276608467102, -0.007344953715801239, -0.19181358814239502, 0.0668690875172615, 0.21158471703529358, 0.30984652042388916, 0.30898764729499817, 0.22051739692687988, 0.24930059909820557, -0.4843200445175171, 0.3583654463291168, -0.07803912460803986, 0.05340508744120598, -0.1580757200717926, -0.2680452764034271, 0.48197948932647705, -0.21391341090202332, 0.06509925425052643], [-0.44803738594055176, -0.43147391080856323, 0.25968608260154724, 0.18988704681396484, -0.007801470346748829, 0.1300433725118637, 0.06686177104711533, 0.20510250329971313, 0.2138306200504303, 0.10821849852800369, -0.025290707126259804, 0.24188414216041565, 0.2711948752403259, 0.2891816794872284, 0.07444602251052856, -0.1343555897474289, -0.4605388641357422, 0.05521322041749954, -0.288607656955719, -0.3576202094554901, -0.17338630557060242, 0.023623447865247726, -0.14761748909950256, 0.3411104083061218, 0.06339889019727707, 0.08569380640983582, -0.28888070583343506, -0.1533336341381073, -0.22426296770572662, 0.1457454413175583, -0.01605655811727047, 0.11129491776227951, 0.46415019035339355, -0.31935006380081177, 0.2589748501777649, 0.07361850142478943, 0.3174709975719452, -0.10847783088684082, -0.024941058829426765, -0.5122538208961487, 0.05727912113070488, 0.0191939864307642, 0.2229883372783661, -0.340293288230896, -0.1664232611656189, -0.14800065755844116, 0.2107311189174652, -0.04119676351547241], [-0.09129881858825684, -0.23782245814800262, -0.00754333334043622, -0.6622610092163086, 0.2877090275287628, 0.45130375027656555, 0.17760033905506134, 0.028155885636806488, 0.12361394613981247, 0.10662296414375305, 0.00669080438092351, 0.16314470767974854, -0.1087656021118164, -0.02146713063120842, 0.5888224840164185, 0.35493534803390503, 0.35241496562957764, -0.0015399871626868844, -0.022175341844558716, -0.12050873786211014, -0.34587061405181885, 0.28247344493865967, -0.09878040850162506, -0.0006809057085774839, -0.0679699033498764, -0.5131524801254272, -0.17331376671791077, 0.5210489630699158, -0.49347129464149475, -0.22155232727527618, -0.21472816169261932, 0.25800034403800964, -0.065357506275177, 0.040730301290750504, 0.2422822266817093, 0.06160702556371689, 0.31674325466156006, -0.15427502989768982, -0.16375288367271423, -0.0012132559204474092, -0.16482508182525635, -0.13655884563922882, -0.2948458790779114, 0.30132007598876953, -0.4137653410434723, -0.14898993074893951, -0.13076259195804596, -0.17527075111865997], [-0.1332298070192337, 0.05144255980849266, 0.37032195925712585, 0.10988505929708481, 0.19110240042209625, 0.1248384416103363, -0.0886945053935051, 0.25589290261268616, -0.40556827187538147, 0.08047199249267578, 0.12257739901542664, -0.2927562892436981, 0.5757125020027161, 0.007656940724700689, 0.06098503619432449, -0.5629003047943115, -0.23646605014801025, -0.3911757469177246, -0.2742088735103607, -0.3773162364959717, -0.21359890699386597, -0.3036494851112366, 0.14013907313346863, 0.16862869262695312, -0.06892024725675583, -0.08306898176670074, 0.28375282883644104, 0.18082599341869354, 0.29551103711128235, -0.0984078049659729, 0.0722886398434639, -0.42766332626342773, 0.1551467329263687, 0.14910189807415009, 0.2027241736650467, 0.17284999787807465, 0.03531874343752861, -0.16428051888942719, -0.049693237990140915, -0.09162221848964691, 0.29908254742622375, -0.1886325180530548, -0.04253177344799042, 0.035721439868211746, 0.28248387575149536, 0.21439439058303833, -0.4801196753978729, 0.07721160352230072], [0.6416121125221252, -0.2763523757457733, 0.21979133784770966, -0.3487341105937958, -0.2460433542728424, 0.14818419516086578, -0.04592946544289589, -0.2474767416715622, 0.3191942870616913, -0.2827100157737732, 0.27210119366645813, -0.002343876753002405, -0.23432952165603638, -0.3832758367061615, -0.19459755718708038, 0.25050708651542664, -0.046000801026821136, -0.3346147835254669, 0.13360856473445892, 0.10931193083524704, 0.10420023649930954, 0.033886611461639404, -0.16578687727451324, -0.44704753160476685, -0.019814293831586838, 0.36556974053382874, 0.4728228449821472, -0.0178692489862442, -0.20048682391643524, 0.17249900102615356, -0.02389783412218094, 0.11367718130350113, -0.07694105058908463, 0.1053912490606308, 0.016335971653461456, 0.3555661141872406, -0.1328098177909851, -0.04015582799911499, 0.22846107184886932, 0.25467541813850403, 0.07394041866064072, -0.3236459493637085, -0.14432983100414276, -0.1546642929315567, -0.18689243495464325, 0.23187781870365143, 0.24420097470283508, -0.4468683898448944], [0.15230950713157654, 0.12466450780630112, 0.14449737966060638, -0.2067001760005951, -0.05560470372438431, 0.370807021856308, -0.16223463416099548, 0.022720620036125183, -0.20030538737773895, -0.4082944989204407, 0.3993299901485443, -0.29682493209838867, 0.1186162680387497, 0.05437099561095238, -0.07445184141397476, 0.21081086993217468, -0.09220472723245621, -0.1803601235151291, -0.4223301410675049, 0.22254298627376556, 0.09177185595035553, 0.0006283166585490108, -0.43556609749794006, -0.07801991701126099, 0.18008022010326385, 0.24882665276527405, 0.5267094969749451, 0.4282628893852234, 0.12281916290521622, 0.02010507322847843, 0.10811129957437515, -0.446908175945282, 0.23473240435123444, 0.001291348715312779, -0.16561071574687958, 0.05714213475584984, 0.10966098308563232, -0.5752308964729309, -0.08370058983564377, 0.36492910981178284, 0.010717746801674366, -0.636436402797699, 0.058660611510276794, 0.12226485460996628, 0.01677725464105606, -0.033619921654462814, 0.0030199014581739902, -0.04381473362445831], [0.31762099266052246, 0.17559601366519928, -0.3589079678058624, -0.10355030000209808, -0.4225046932697296, 0.297049880027771, 0.04480011388659477, -0.17424190044403076, 0.3718671500682831, -0.3281218409538269, 0.14551809430122375, 0.5496482253074646, -0.40498843789100647, 0.17929713428020477, -0.30785638093948364, 0.13436610996723175, 0.1868676394224167, -0.03360920399427414, 0.06694688647985458, 0.43777766823768616, -0.06409148871898651, 0.2297331839799881, -0.11302289366722107, -0.48133325576782227, -0.046861689537763596, 0.1651618778705597, 0.20774684846401215, 0.1481507420539856, 0.26776936650276184, 0.07222644984722137, -0.046984922140836716, -0.3659190237522125, -0.024899305775761604, -0.21467074751853943, 0.048692841082811356, -0.4833228886127472, -0.021743126213550568, -0.4977390766143799, -0.41638174653053284, 0.3051491677761078, -0.28718301653862, 0.0748729556798935, 0.01515375915914774, 0.3648618757724762, -0.05769497901201248, -0.217550128698349, 0.3537634611129761, 0.0550585612654686], [0.4452972710132599, 0.15272633731365204, 0.31375041604042053, -0.026140937581658363, 0.0025013345293700695, 0.27041953802108765, 0.23059260845184326, -0.39552736282348633, 0.2744808495044708, -0.2006579488515854, -0.08146083354949951, -0.1441652774810791, -0.14362379908561707, -0.5593659281730652, -0.39318814873695374, -0.3070899546146393, -0.21265311539173126, -0.39814281463623047, -0.17468194663524628, -0.08218244463205338, 0.37521234154701233, -0.5470752716064453, 0.10121218115091324, -0.09624943137168884, -0.2232624739408493, 0.11803038418292999, 0.48300474882125854, -0.42238253355026245, 0.004970112349838018, 0.18480505049228668, -0.423987478017807, -0.3696126639842987, 0.2767559885978699, 0.2947298288345337, 0.06970781087875366, 0.4495832026004791, -0.07751873135566711, 0.11338674277067184, 0.3365350067615509, 0.06865652650594711, 0.6365953087806702, -0.15993578732013702, -0.07333601266145706, -0.022914789617061615, -0.24059410393238068, 0.17820756137371063, 0.17170797288417816, -0.5343999266624451], [0.03710267320275307, 0.06076275184750557, 0.32558292150497437, -0.26742643117904663, -0.3119462728500366, 0.27419114112854004, -0.17166359722614288, 0.44470781087875366, -0.04856941103935242, -0.052851054817438126, 0.4159179627895355, 0.07877468317747116, -0.07016223669052124, 0.12554606795310974, 0.28430819511413574, -0.0631629154086113, -0.021875321865081787, 0.015210776589810848, -0.2946832478046417, -0.369027316570282, -0.29905351996421814, 0.15030476450920105, 0.050417590886354446, 0.12823139131069183, 0.1089346706867218, -0.30251753330230713, -0.2414528727531433, 0.1422383189201355, 0.07760342955589294, 0.5943461060523987, 0.1911572962999344, -0.08550657331943512, -0.15306416153907776, -0.17549724876880646, -0.19911059737205505, 0.4287171959877014, 0.5463722348213196, -0.5234950184822083, 0.05543903633952141, 0.15847191214561462, -0.12196330726146698, -0.05297795310616493, 0.12083373218774796, 0.031205369159579277, -0.09802502393722534, 0.1493363082408905, -0.3077983856201172, -0.34096696972846985], [-0.15628954768180847, -0.2760485112667084, -0.04830746352672577, -0.4676035940647125, 0.3311200737953186, 0.325154185295105, -0.16682499647140503, -0.2358824759721756, 0.11244561523199081, 0.3219849467277527, 0.06657569855451584, -0.09365279972553253, -0.2874104082584381, 0.04566212743520737, 0.21953770518302917, 0.5895816087722778, 0.40615808963775635, 0.3031700551509857, 0.18412373960018158, 0.16166773438453674, 0.10992052406072617, 0.05588790029287338, -0.004115537740290165, -0.20477262139320374, 0.09891383349895477, -0.3701517581939697, 0.06384322792291641, 0.09583789855241776, -0.19139863550662994, -0.29479989409446716, -0.1750032752752304, 0.28700605034828186, 0.0962725356221199, 0.09745864570140839, -0.1743975430727005, -0.12845833599567413, 0.0324285551905632, 0.020554574206471443, -0.3029106855392456, 0.1302865892648697, 0.11770853400230408, -0.07717140018939972, -0.12839865684509277, 0.41182833909988403, -0.21519047021865845, 0.1385578215122223, -0.19891612231731415, -0.26290521025657654], [-0.16745395958423615, 0.22482308745384216, 0.027298536151647568, 0.4049227833747864, -0.026298770681023598, -0.16913576424121857, -0.08434611558914185, 0.1328473538160324, -0.4921712577342987, -0.44786083698272705, -0.07381226122379303, -0.18432451784610748, 0.2522987127304077, -0.020103074610233307, -0.09134600311517715, -0.19768570363521576, -0.3319266736507416, -0.1646859347820282, -0.39060643315315247, 0.20292547345161438, 0.1944790929555893, -0.10986384004354477, -0.44142064452171326, 0.5265580415725708, 0.07739819586277008, 0.41359758377075195, 0.04429525509476662, 0.15737567842006683, 0.06311265379190445, -0.1428641825914383, 0.206842303276062, -0.3445490896701813, 0.05613672360777855, -0.14450474083423615, 0.04129881039261818, 0.021802691742777824, -0.08055133372545242, -0.06552643328905106, -0.0046836575493216515, -0.19295348227024078, -0.0391111820936203, -0.3963303864002228, 0.1320936232805252, -0.17496003210544586, 0.12969627976417542, -0.1568002849817276, 0.3132741153240204, 0.5306383371353149], [0.013337996788322926, 0.20516438782215118, -0.13489316403865814, 0.06873822212219238, 0.021841544657945633, 0.014109503477811813, 0.38207191228866577, 0.137963205575943, 0.3581075668334961, 0.19253164529800415, 0.004702034872025251, 0.22529079020023346, 0.2525593042373657, 0.26461926102638245, 0.19052579998970032, -0.3634733557701111, 0.049651484936475754, -0.1942114382982254, -0.06352277845144272, -0.40146005153656006, -0.5454627871513367, 0.22420570254325867, 0.35841289162635803, -0.14780041575431824, -0.06285601854324341, -0.2132580429315567, -0.14814165234565735, 0.14467550814151764, 0.04761746898293495, 0.2696221172809601, 0.07218114286661148, -0.11645995080471039, 0.027119865640997887, 0.05515074357390404, 0.29556286334991455, -0.24175702035427094, 0.32074636220932007, -0.12906166911125183, 0.177806094288826, 0.03628325089812279, -0.2431529462337494, 0.33180737495422363, 0.005012825131416321, -0.14431829750537872, 0.0674145370721817, -0.16324305534362793, -0.25343284010887146, 0.05920909345149994], [-0.11390894651412964, -0.2147696614265442, 0.22227846086025238, -0.4685574173927307, -0.4371476173400879, 0.4464057981967926, -0.24427494406700134, 0.14958393573760986, 0.002211622893810272, -0.04484022781252861, 0.1020459234714508, 0.5362967252731323, -0.2363932877779007, 0.15647803246974945, -0.19726191461086273, 0.2781558334827423, -0.09577316790819168, 0.45053020119667053, 0.20825156569480896, 0.0646907165646553, -0.034898530691862106, 0.0885738655924797, 0.02783367410302162, -0.24786390364170074, 0.31003227829933167, -0.03827587515115738, -0.12896530330181122, 0.006944670341908932, 0.021061360836029053, 0.19384877383708954, -0.16823552548885345, -0.024186188355088234, 0.1391688585281372, -0.050349753350019455, -0.09426624327898026, 0.19038084149360657, 0.04788780212402344, -0.48481082916259766, -0.38662421703338623, -0.1814616620540619, 0.12370830774307251, 0.06274335086345673, -0.2540166676044464, 0.37115493416786194, -0.10679073631763458, 0.021293211728334427, 0.02841191552579403, -0.2732602059841156], [0.3373911380767822, 0.12476181238889694, -0.10308171808719635, 0.3109109699726105, -0.224966898560524, -0.4902816712856293, 0.4307470917701721, 0.08800063282251358, -0.07491376996040344, -0.11732130497694016, 0.07730959355831146, -0.059259697794914246, 0.32196617126464844, 0.1427910178899765, -0.527839720249176, -0.026637347415089607, -0.137192964553833, 0.0859910175204277, 0.003596331225708127, 0.012911120429635048, 0.060124270617961884, -0.21354341506958008, -0.2762121558189392, -0.34601011872291565, -0.2181393951177597, 0.21831707656383514, 0.05181195214390755, 0.025773299857974052, 0.2973398268222809, -0.0032551728654652834, 0.2095896303653717, -0.32555097341537476, 0.20181246101856232, -0.38026490807533264, -0.3245072066783905, -0.15251418948173523, -0.23037217557430267, -0.2747887074947357, 0.08787507563829422, 0.25016775727272034, -0.33642175793647766, 0.13923974335193634, 0.27872857451438904, -0.13841338455677032, 0.3673337399959564, -0.4728977680206299, 0.24924710392951965, -0.06887230277061462], [0.26107385754585266, -0.10808277130126953, 0.3099227845668793, -0.19927053153514862, -0.14849045872688293, -0.22667387127876282, 0.5213423371315002, 0.14696624875068665, -0.2183481603860855, 0.062256667762994766, -0.16624528169631958, 0.2366565465927124, 0.14271098375320435, -0.25646358728408813, 0.040222883224487305, -0.19570396840572357, 0.09189174324274063, -0.12563246488571167, 0.31314581632614136, -0.24915550649166107, -0.03710649535059929, -0.2374727725982666, 0.04917176440358162, 0.046640750020742416, -0.10470626503229141, -0.22663873434066772, -0.35484209656715393, 0.1633966714143753, -0.4401482939720154, -0.3069588243961334, 0.06058274209499359, 0.09754429757595062, -0.39886167645454407, -0.04140251502394676, 0.229140967130661, 0.4104892313480377, -0.17018559575080872, 0.24704888463020325, 0.17370817065238953, -0.5329423546791077, -0.09738223999738693, 0.2654987573623657, -0.3370184898376465, 0.04546922445297241, 0.023706112056970596, -0.17689408361911774, -0.021856773644685745, 0.039018966257572174], [0.3538678288459778, 0.4794783592224121, -0.4749045968055725, -0.07383279502391815, 0.26508215069770813, -0.1491302251815796, 0.16303642094135284, -0.12217558175325394, -0.07488519698381424, -0.3695956766605377, -0.31880852580070496, -0.012952450662851334, -0.15650466084480286, -0.19718894362449646, -0.4925927519798279, 0.15354344248771667, 0.18135705590248108, -0.1491207480430603, 0.14578941464424133, 0.44177013635635376, 0.013814888894557953, -0.11437314748764038, -0.13911330699920654, -0.32390424609184265, 0.06651456654071808, 0.45494839549064636, 0.18004153668880463, 0.16096076369285583, 0.18649180233478546, -0.18771755695343018, 0.029318928718566895, -0.4524178206920624, 0.05740514397621155, 0.29086101055145264, 0.02580113708972931, -0.12416961044073105, -0.12170031666755676, 0.11824577301740646, 0.26813995838165283, 0.4104681611061096, -0.158644899725914, -0.18603651225566864, -0.37375152111053467, -0.011903807520866394, -0.2300960272550583, -0.15409374237060547, 0.4678812325000763, 0.2906084358692169], [-0.2099023461341858, 0.22618089616298676, 0.08481086790561676, -0.2802106738090515, 0.5046536326408386, -0.08621691912412643, 0.17674170434474945, -0.3799196481704712, -0.2042538970708847, 0.4502131938934326, -0.6198060512542725, 0.08059138059616089, -0.028842950239777565, -0.49869903922080994, 0.11682384461164474, -0.12378377467393875, 0.1389789581298828, 0.2945506274700165, 0.4606318175792694, -0.36383503675460815, -0.14464911818504333, -0.35666024684906006, 0.5149452686309814, -0.09979517757892609, -0.057548414915800095, -0.39611855149269104, -0.08544190227985382, -0.07414291799068451, -0.12734834849834442, -0.012383166700601578, -0.39873796701431274, 0.06593066453933716, 0.16539366543293, 0.6031981706619263, -0.1120496615767479, 0.22189068794250488, -0.31532716751098633, 0.331063449382782, -0.01394183561205864, -0.15819507837295532, 0.6008647084236145, 0.19983793795108795, -0.7221641540527344, 0.37163636088371277, -0.09936840087175369, 0.19240780174732208, -0.2596147954463959, -0.16763979196548462], [-0.3792750835418701, 0.6109057068824768, -0.31717756390571594, 0.5513342022895813, 0.08165163546800613, -0.17836518585681915, -0.10156331956386566, -0.28105998039245605, -0.1341715008020401, 0.03729680925607681, -0.49623018503189087, 0.18267922103405, -0.08387754112482071, -0.04038410261273384, -0.1324378401041031, -0.28213611245155334, -0.01059170626103878, 0.22670750319957733, 0.07762379199266434, 0.18318800628185272, 0.22748401761054993, -0.03036627173423767, 0.38596194982528687, 0.3368384540081024, 0.0905139148235321, -0.16603659093379974, -0.22701816260814667, -0.40146154165267944, 0.3368752598762512, 0.02400881238281727, -0.09431058913469315, -0.30562087893486023, 0.05791044980287552, 0.26257169246673584, 0.07895739376544952, -0.41070157289505005, -0.1825351119041443, 0.21810102462768555, -0.3093688488006592, -0.30942395329475403, 0.2583780884742737, 0.2429618090391159, -0.06352021545171738, 0.2809605896472931, 0.3638657033443451, 0.15715114772319794, -0.036652710288763046, 0.46988797187805176], [0.04937050864100456, 0.4215688407421112, -0.3705260753631592, 0.22577136754989624, -0.04887654632329941, 0.017720986157655716, 0.48577266931533813, -0.45961299538612366, 0.2149057239294052, -0.2511690557003021, -0.3253835439682007, 0.23885603249073029, -0.16980530321598053, -0.07044295221567154, -0.23007461428642273, 0.010535841807723045, 0.03613680973649025, 0.12338269501924515, -0.12115567177534103, 0.32571274042129517, 0.2751500606536865, -0.08027862012386322, -0.18231306970119476, -0.013398532755672932, -0.21334494650363922, -0.007465849164873362, 0.0630907416343689, -0.028465982526540756, -0.01086784154176712, -0.10812097787857056, -0.32611745595932007, -0.34334176778793335, 0.2589899003505707, -0.08752468228340149, -0.014651386998593807, -0.44383925199508667, -0.18788550794124603, -0.12976744771003723, -0.19785642623901367, 0.012362517416477203, -0.007983297109603882, 0.07927413284778595, 0.06718125939369202, 0.2664584517478943, -0.015108403749763966, -0.5251521468162537, 0.4598236083984375, 0.1270970106124878], [0.3196473717689514, 0.10602427273988724, -0.09924241900444031, -0.10745277255773544, -0.26037293672561646, -0.030640369281172752, 0.2635847330093384, 0.23822452127933502, 0.22153273224830627, 0.16759707033634186, 0.24100439250469208, 0.038348983973264694, -0.2604006230831146, 0.26679831743240356, -0.141675665974617, 0.2442452311515808, 0.47787702083587646, 0.179545596241951, 0.14032721519470215, 0.1203698068857193, 0.2539931535720825, 0.01338863093405962, 0.18401488661766052, -0.16761654615402222, 0.040557559579610825, -0.535993754863739, -0.45203325152397156, -0.21176305413246155, 0.03273213654756546, -0.08227968215942383, 0.1587817370891571, 0.020073333755135536, -0.38779670000076294, -0.22158795595169067, -0.13223865628242493, -0.011683671735227108, 0.2990903854370117, 0.04628197103738785, 0.11995483189821243, 0.0470193475484848, -0.3929641544818878, 0.4887879192829132, 0.28515195846557617, 0.17854627966880798, 0.09150489419698715, -0.08858990669250488, -0.24657484889030457, -0.24383005499839783]], "dtype": "float32"}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(cosine_similarity_matrix, json.loads(_TESTS), 'cuda', *[1e-05, 1e-06])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")